# 零售場景的生成式 AI：從 Prompt 到可上線的評測流程

**Google Cloud & Generative AI Applications**

---

這份 notebook 要證明一件事：

> **生成式 AI 的難處不在叫模型，而在你怎麼知道它有沒有變好。**

我們會用一個真實的零售任務（商品文案生成）走完整條路：

| 節 | 內容 | 重點 |
|---|---|---|
| §1 | 商品資料與法規禁詞 | 規則要跟著商品屬性走 |
| §2 | Prompt v0 → v3，每版只加一件事 | Prompt 是規格書，不是咒語 |
| §3 | 三層評測，產出版本 × 指標對比表 | 怎麼證明它變好了 |
| §4 | 場景 B：評論洞察 | 同一套流程，反方向用 |
| §5 | 成本：實際花了多少，外推到 10 萬 SKU | 教公式，不背數字 |

**§3 的對比表是整份 notebook 的重點。** 其他都是為了讓那張表可信。

---

### 執行前須知

- 需要一個啟用了 Vertex AI API 與計費的 GCP 專案。
- 全部跑完的成本很低（見 §5 實際印出的數字），但**不是零**。
- 若沒有 GCP 專案，可在 §0 的 CONFIG 把 `USE_VERTEX` 改成 `False`
  並填入 AI Studio API key。

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **Demo 全長 12 分鐘（30:00–42:00）。開場前先 Run all 一次，讓輸出都在畫面上。**
>
> 先確認 CONFIG 的 `OFFLINE_MODE = True` —— 這是最容易忘的一項。
>
> 開場第一句（不要道歉，一句帶過）：
> 「我先說明，這些輸出是我昨天跑好存下來的，現在是重播 —— 因為會場網路我不敢賭。
> 程式碼跟評測都是當場執行的，只有呼叫模型那一步是查表。」

</div>

## §0 環境設定

### 相依套件

這一格**只在缺少套件時才安裝**，不是無條件 `pip install`。

| 環境 | 行為 |
|---|---|
| Colab | 預裝了 pandas / jinja2 / matplotlib，會補裝 `google-genai` |
| 本機 uv 環境 | 全部已在 `.venv`，直接跳過 |

會做這個判斷是因為：在 uv 管理的環境裡無條件 `pip install`，套件會裝進
`.venv` 卻不在 `uv.lock` 裡，環境就跟鎖定檔對不上了。與其寫註解叫人自己
判斷該不該跑，不如讓它自己偵測。

In [ ]:
import sys

# 渲染用，任何模式都需要。§3 的對比表用 df.style.background_gradient() 上色：
# .style 需要 jinja2、background_gradient 需要 matplotlib，缺一就會 AttributeError。
CORE = {"pandas": "pandas", "jinja2": "jinja2", "matplotlib": "matplotlib"}

# 只有要「真的呼叫 API」才需要。離線重播完全用不到它 ——
# 這是刻意設計的：要裝它就得有網路，一旦需要網路，零網路重播的前提就破功了。
LIVE = {"google.genai": "google-genai"}


def _importable(mod: str) -> bool:
    try:
        __import__(mod)
        return True
    except ImportError:
        return False


_missing = [pkg for mod, pkg in {**CORE, **LIVE}.items() if not _importable(mod)]

if _missing:
    print("缺少套件，嘗試安裝：", " ".join(_missing))
    import subprocess
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *_missing], check=True
        )
        print("✓ 安裝完成（若 import 仍失敗，重啟 runtime 再跑一次）")
    except Exception as e:
        # 安裝失敗不一定是問題：離線重播根本不需要 google-genai。
        print(f"⚠ 安裝失敗：{e}")
        if not _importable("google.genai"):
            print("  google-genai 未安裝 —— 離線重播（OFFLINE_MODE=True）不需要它，可以繼續。")
            print("  但若要真的呼叫 API，就必須在有網路的環境先裝好。")
        _still = [p for m, p in CORE.items() if not _importable(m)]
        if _still:
            print(f"  ✗ 仍缺渲染必要套件 {_still} —— §3 的對比表會無法上色。")
else:
    print("✓ 相依套件齊全，跳過安裝")

# BigQuery 只有 §4 要用；[pandas] extra 才會帶入 to_dataframe() 需要的 pyarrow
if not _importable("google.cloud.bigquery"):
    print('  （§4 若要實際寫入 BigQuery：pip install "google-cloud-bigquery[pandas]"）')

### CONFIG — 只改這一格

價格常數**使用前必須到官方頁面現查更新**：
<https://cloud.google.com/vertex-ai/generative-ai/pricing>

模型與價格變動頻繁，任何寫死的數字都會很快過期。
要看實際花費，請以 §5 執行後印出來的金額為準。

In [ ]:
"""集中設定：模型、價格、通路限制。

所有環境相關設定集中在這裡。換專案、換模型、換價格都只改這個檔案。
價格必須到官方頁面現查後更新，不要沿用寫死的數字。
"""

# --- GCP ---
PROJECT_ID = "cacafly-poc"

# ⚠️ 必須是 "global"。實測 asia-east1 上沒有任何 Gemini publisher model，
#    所有呼叫都會 404。這是很容易踩的坑：compute 資源設在 asia-east1，
#    不代表 Gemini 模型在那裡可用。
LOCATION = "global"

# 使用 Vertex AI（走 GCP 專案計費、資料落地可控）
# 若現場 GCP 權限失效，把 USE_VERTEX 改成 False 並填入 AI Studio key 作為備援
USE_VERTEX = True
FALLBACK_API_KEY = ""  # 僅備援用，勿 commit 真實金鑰

# --- 離線重播（會場網路不穩時的主要策略）---
# 在網路不穩的場合（會議室、教室、展場）即時打 API 是不能接受的風險。
#
#   RECORD_FIXTURES = True   有網路時先跑一次，把真實輸出錄下來
#   OFFLINE_MODE    = True   之後零網路重播錄好的輸出
#
# 重播時所有 cell 一樣會執行、表格一樣是當場算出來的，只有 API 呼叫被換成
# 查表。畫面上看不出差別，但完全不依賴網路。
#
# ⚠ 兩個不能同時為 True。
OFFLINE_MODE = False
RECORD_FIXTURES = False
FIXTURES_FILE = "demo_outputs.json"

# --- 模型 ---
# 以下型號皆已在 cacafly-poc / global 實測可用（2026-08-13）。
# 換專案或換 region 前請重跑 poc/check_env.py 確認。
GEN_MODEL = "gemini-flash-latest"   # 生成用：吞吐量大、要便宜
JUDGE_MODEL = "gemini-2.5-pro"      # 評審用：刻意與生成模型不同，降低 self-preference bias
CHEAP_MODEL = "gemini-2.5-flash-lite"  # 成本段示範「降級」用

GEN_TEMPERATURE = 0.4  # 用低溫，降低失敗機率
JUDGE_TEMPERATURE = 0.0

# 並行度。評審模型單次要十幾到二十幾秒，瓶頸完全在等待回應，不在 CPU，
# 所以拉高執行緒數幾乎是線性加速。
#
# 實測吞吐（gemini-2.5-pro，cacafly-poc，2026-08-13，零失敗）：
#     8 → 10.3 次/分     16 → 26.1 次/分
#    32 → 46.5 次/分     64 → 80.7 次/分
#
# 到 64 都還沒撞到配額。設 32 是留一點餘裕的預設值；
# 要更快就調到 64（評測腳本可用 --workers 覆寫，不必改這裡）。
#
# 遇到 429 時不用緊張：生成與評審兩條路徑都有指數退避重試，
# 會自動降速而不是整批失敗。真的一直撞才把這個數字調小。
MAX_WORKERS = 32

# --- 價格（USD / 每百萬 token）---
# ⚠️ 這些是佔位值。使用前請到官方頁面現查更新：
#    https://cloud.google.com/vertex-ai/generative-ai/pricing
# 不要引用這裡的數字，要看 §5 實際跑出來的花費。
PRICING = {
    "gemini-flash-latest": {"input": 0.30, "output": 2.50},
    "gemini-2.5-pro": {"input": 1.25, "output": 10.00},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
}
PRICE_LAST_CHECKED = "尚未查證 — 上場前必須更新"

# --- BigQuery（場景 B）---
# 預設關閉：重播模式不涵蓋 BigQuery 呼叫。
# 錄製時可開啟一次，把「AI 當 ETL」這件事真的做完。
USE_BIGQUERY = False
BQ_DATASET = "retail_genai_demo"
BQ_TABLE = "review_insights"
BQ_LOCATION = "asia-east1"  # BigQuery 的 region 與 Gemini 無關，這裡可以用亞洲

USD_TO_TWD = 32.0

# --- 通路限制（來自 products.json 的 _meta.channel_limits）---
SHOPEE_TITLE_MAX = 60
SEO_DESC_MAX = 120
BULLET_MAX = 30
BULLET_COUNT = 4

### 呼叫層與離線重播

底下這段程式碼做兩件事。

**第一，切換 Vertex AI 與 AI Studio 只差 Client 的建構參數**，
其他程式碼一行都不用改。選哪一個是**部署決策**
（資料落地、計費歸屬、VPC-SC、合規），不是**程式碼決策**。
先用 AI Studio 做原型，要簽 DPA 時再切到 Vertex。

**第二，離線重播。** 在網路不穩的場合即時打 API 是不能接受的風險，
所以有兩個模式：

| 設定 | 何時用 | 行為 |
|---|---|---|
| `RECORD_FIXTURES=True` | 有網路時，先跑一次 | 正常呼叫，同時把輸出錄下來 |
| `OFFLINE_MODE=True` | 之後任何時候 | 零網路，重播錄好的輸出 |

重播時**所有 cell 一樣會執行、表格一樣是當場算出來的**，
只有 API 呼叫被換成查表。`ReplayClient` 找不到對應輸出時會明確報錯，
不會靜默拿到過期資料。

In [ ]:
"""呼叫 Gemini 產生文案，並記錄 token 用量。

用 google-genai 統一 SDK。同一份程式碼可切換 Vertex AI 與 Gemini API ——
只差 Client 的建構參數。重點是：
選 Vertex 還是 AI Studio 是**部署決策**，不是**程式碼決策**。
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from types import SimpleNamespace



def gen_config(**kwargs):
    """建立 generate_content 的 config 物件。

    為什麼不直接用 `types.GenerateContentConfig`：

    離線重播的使用情境是「把單一個 .ipynb 丟到一台沒有網路的機器」——
    這種時候 google-genai 很可能根本沒安裝，而安裝它就需要網路，
    整個「零網路」的前提就破功了。

    但重播其實不需要真的 SDK 型別：`ReplayClient` 只會讀 `response_schema`
    這一個屬性來判斷是不是結構化輸出。所以 SDK 不在時退回一個等效的
    簡單物件即可。

    只有 OFFLINE_MODE 才容許退回。連線模式下缺 SDK 是真的有問題，
    要讓它直接炸出來，不能靜默降級。
    """
    try:
        from google.genai import types

        return types.GenerateContentConfig(**kwargs)
    except ImportError:
        if not OFFLINE_MODE:
            raise
        return SimpleNamespace(**kwargs)


@dataclass
class Usage:
    """單次呼叫的 token 用量。所有成本計算都從這裡出發。"""

    model: str
    input_tokens: int
    output_tokens: int
    latency_s: float

    @property
    def cost_usd(self) -> float:
        price = PRICING.get(self.model)
        if price is None:
            return 0.0
        return (
            self.input_tokens * price["input"] + self.output_tokens * price["output"]
        ) / 1_000_000


@dataclass
class GenResult:
    text: str
    usage: Usage
    error: str | None = None


@dataclass
class UsageLedger:
    """累計所有呼叫的用量 —— demo §5 成本計算的資料來源。

    刻意把 judge 的呼叫也記進來。評測本身要花錢，這是最常被漏算的一筆。
    """

    calls: list[tuple[str, Usage]] = field(default_factory=list)

    def record(self, tag: str, usage: Usage) -> None:
        self.calls.append((tag, usage))

    def total_cost_usd(self, tag_prefix: str | None = None) -> float:
        return sum(
            u.cost_usd
            for t, u in self.calls
            if tag_prefix is None or t.startswith(tag_prefix)
        )

    def total_tokens(self, tag_prefix: str | None = None) -> tuple[int, int]:
        rows = [u for t, u in self.calls if tag_prefix is None or t.startswith(tag_prefix)]
        return sum(u.input_tokens for u in rows), sum(u.output_tokens for u in rows)

    def summary(self) -> str:
        gen_in, gen_out = self.total_tokens("gen")
        jdg_in, jdg_out = self.total_tokens("judge")
        gen_cost = self.total_cost_usd("gen")
        jdg_cost = self.total_cost_usd("judge")
        total = gen_cost + jdg_cost
        judge_share = (jdg_cost / total * 100) if total else 0.0
        return (
            f"呼叫次數：{len(self.calls)}\n"
            f"  生成  input {gen_in:>7,} / output {gen_out:>7,} tokens → US${gen_cost:.4f}\n"
            f"  評審  input {jdg_in:>7,} / output {jdg_out:>7,} tokens → US${jdg_cost:.4f}\n"
            f"  合計                                        → US${total:.4f}"
            f"（約 NT${total * USD_TO_TWD:.2f}）\n"
            f"\n  ⚠ 評審佔總成本 {judge_share:.0f}% —— 這筆最常被漏算。"
        )


# --------------------------------------------------------------------------
# 離線重播 —— 會場網路不穩時的主要策略
# --------------------------------------------------------------------------
def fixture_key(model: str, contents: str, structured: bool) -> str:
    """以 (模型, prompt, 是否結構化) 當索引鍵。

    prompt 完全相同才會命中，所以只要 prompt 有任何改動，重播就會失敗並
    明確報錯 —— 不會靜默拿到舊資料。這是刻意的：寧可在測試時炸掉，
    也不要產出跟程式碼對不上的結果。
    """
    import hashlib

    raw = f"{model}\x00{int(structured)}\x00{contents}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]


class _ReplayModels:
    def __init__(self, fixtures: dict):
        self._f = fixtures
        self.hits = 0

    def generate_content(self, *, model, contents, config=None):
        structured = bool(config is not None and getattr(config, "response_schema", None))
        key = fixture_key(model, contents if isinstance(contents, str) else str(contents), structured)
        row = self._f.get("calls", {}).get(key)
        if row is None:
            raise KeyError(
                f"離線重播找不到對應輸出（key={key}, model={model}）。\n"
                f"代表 prompt 或模型在錄製之後被改過。\n"
                f"請重新錄製：把 RECORD_FIXTURES 設為 True、在有網路的環境跑一次。"
            )
        self.hits += 1
        return _Recorded(row["text"], row["input_tokens"], row["output_tokens"])

    def count_tokens(self, *, model, contents):
        key = "tok:" + fixture_key(model, contents, False)
        row = self._f.get("token_counts", {}).get(key)
        if row is None:
            raise KeyError(f"離線重播找不到 count_tokens 結果（key={key}）。請重新錄製。")
        self.hits += 1
        return _RecordedTokens(row)


class _Recorded:
    def __init__(self, text: str, pin: int, pout: int):
        self.text = text
        self.usage_metadata = _RecordedUsage(pin, pout)


class _RecordedUsage:
    def __init__(self, pin: int, pout: int):
        self.prompt_token_count = pin
        self.candidates_token_count = pout


class _RecordedTokens:
    def __init__(self, n: int):
        self.total_tokens = n


class ReplayClient:
    """從錄好的 fixtures 回放，完全不碰網路。"""

    def __init__(self, fixtures: dict):
        self.models = _ReplayModels(fixtures)


class _RecordingModels:
    def __init__(self, real):
        self._real = real
        self.fixtures = {"calls": {}, "token_counts": {}}

    def generate_content(self, *, model, contents, config=None):
        resp = self._real.generate_content(model=model, contents=contents, config=config)
        structured = bool(config is not None and getattr(config, "response_schema", None))
        key = fixture_key(model, contents if isinstance(contents, str) else str(contents), structured)
        meta = resp.usage_metadata
        self.fixtures["calls"][key] = {
            "text": resp.text or "",
            "input_tokens": getattr(meta, "prompt_token_count", 0) or 0,
            "output_tokens": getattr(meta, "candidates_token_count", 0) or 0,
        }
        return resp

    def count_tokens(self, *, model, contents):
        resp = self._real.count_tokens(model=model, contents=contents)
        self.fixtures["token_counts"]["tok:" + fixture_key(model, contents, False)] = (
            resp.total_tokens
        )
        return resp


class RecordingClient:
    """包住真實 client，一邊正常呼叫一邊把結果錄下來。"""

    def __init__(self, real_client):
        # ⚠️ 一定要保留 real_client 本身的參照，不能只留 .models。
        #
        # 只留 .models 的話，genai.Client 物件在 make_client() 回傳後就沒有人
        # 參照，會被 GC 回收並關閉底層連線；之後每一次呼叫都會拿到
        # 「Cannot send a request, as the client has been closed.」，
        # 而且是**每一次都失敗、token 全為 0**，看起來像 API 壞掉。
        #
        # 這個 bug 只在真實 client 上會發生（測試用的假 client 沒有連線可關），
        # 所以單元測試抓不到 —— 實際錄製 50 筆時整批炸掉才發現。
        self._real_client = real_client
        self.models = _RecordingModels(real_client.models)

    def dump(self) -> dict:
        return self.models.fixtures


def make_client(fixtures: dict | None = None):
    """建立 client。

    三種模式：
      OFFLINE_MODE    → ReplayClient，零網路，重播錄好的輸出（無網路時）
      RECORD_FIXTURES → RecordingClient，正常呼叫並錄下來（有網路時）
      預設             → 真實 client
    """
    if OFFLINE_MODE and RECORD_FIXTURES:
        raise RuntimeError("OFFLINE_MODE 與 RECORD_FIXTURES 不能同時為 True。")

    if OFFLINE_MODE:
        if not fixtures or not fixtures.get("calls"):
            raise RuntimeError(
                "OFFLINE_MODE=True 但沒有可用的 fixtures。\n"
                "請先在有網路的環境設定 RECORD_FIXTURES=True 跑一次，產生 "
                f"{FIXTURES_FILE}，再重新產生 notebook。"
            )
        return ReplayClient(fixtures)

    from google import genai

    if USE_VERTEX:
        real = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location=LOCATION,
        )
    else:
        if not FALLBACK_API_KEY:
            raise RuntimeError(
                "USE_VERTEX=False 但 FALLBACK_API_KEY 是空的。"
                "請在 CONFIG cell 填入 AI Studio key，或改回 USE_VERTEX=True。"
            )
        real = genai.Client(api_key=FALLBACK_API_KEY)

    return RecordingClient(real) if RECORD_FIXTURES else real


def generate(
    client,
    prompt: str,
    *,
    structured: bool = False,
    model: str | None = None,
    temperature: float | None = None,
    max_retries: int = 3,
) -> GenResult:
    """單次生成。

    structured=True 時啟用 API 原生的 responseSchema —— 這是 v3 與 v2 的唯一差別。
    重點：不要用 prompt 硬凹 JSON 格式，要用 API 參數約束，模型才真的被限制在 schema 內。
    """

    model = model or GEN_MODEL
    temperature = GEN_TEMPERATURE if temperature is None else temperature

    cfg_kwargs = {"temperature": temperature}
    if structured:
        cfg_kwargs["response_mime_type"] = "application/json"
        cfg_kwargs["response_schema"] = COPY_SCHEMA

    last_err = None
    for attempt in range(max_retries):
        started = time.time()
        try:
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config=gen_config(**cfg_kwargs),
            )
            meta = resp.usage_metadata
            return GenResult(
                text=resp.text or "",
                usage=Usage(
                    model=model,
                    input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                    output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                    latency_s=round(time.time() - started, 2),
                ),
            )
        except Exception as e:  # noqa: BLE001 — demo 要能撐過暫時性錯誤
            last_err = e
            if attempt < max_retries - 1:
                time.sleep(2**attempt)  # 指數退避

    return GenResult(
        text="",
        usage=Usage(model=model, input_tokens=0, output_tokens=0, latency_s=0.0),
        error=str(last_err),
    )


def run_parallel(fn, items, *, workers: int | None = None, label: str = "處理中"):
    """對每個項目呼叫 fn，並行執行，**保持原本的順序**。

    為什麼需要這個：評審模型單次要十幾秒，整份 demo 序列跑完要十幾分鐘
    （光是 rubric 逐條檢查那一段就約 9 分鐘）。這些呼叫彼此獨立，
    並行之後整份只剩兩三分鐘。錄製 fixtures 時差別很有感。

    刻意保持順序（用 index 回填而非 as_completed 的完成順序），
    這樣結果與序列版本完全一致，重播時的 key 也不會受影響。

    離線重播時每次呼叫都是查表、瞬間完成，並行與否沒差；但仍然走同一條
    程式路徑，避免「錄製走 A、重播走 B」這種不一致。
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    items = list(items)
    if not items:
        return []

    workers = workers or MAX_WORKERS
    results: list = [None] * len(items)
    step = max(1, len(items) // 5)
    done = 0

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(fn, item): i for i, item in enumerate(items)}
        for fut in as_completed(futures):
            idx = futures[fut]
            results[idx] = fut.result()
            done += 1
            if done % step == 0 or done == len(items):
                print(f"  {label} {done}/{len(items)}")
    return results


def count_tokens(client, text: str, model: str | None = None) -> int:
    """實測中文的 token 數。

    不要背「中文一個字約幾個 token」這種經驗法則 —— 用這個函式直接量。
    量出來的數字比任何通用經驗法則可靠。
    """
    model = model or GEN_MODEL
    resp = client.models.count_tokens(model=model, contents=text)
    return resp.total_tokens

In [ ]:
# 已錄製 118 筆真實輸出，直接內嵌在這一格。
# 只要有這份資料，這個 notebook 單獨一個檔案就能零網路跑完。
OFFLINE_MODE = True   # ← 有 fixtures 就預設離線；要真的打 API 改成 False
FIXTURES = {"calls": {"25b652acacbe93e21b9f": {"text": "{\n  \"title\": \"青研 CHINGYEN 積雪草舒緩精華液 30ml 敏弱肌適用 5%高濃度配方\",\n  \"bullets\": [\n    \"5%高純度積雪草萃取，溫和穩定肌膚狀態。\",\n    \"水感輕盈精華質地，快速吸收不黏膩。\",\n    \"添加泛醇與神經醯胺，提升肌膚防護力。\",\n    \"韓國製造，配方單純，敏弱肌友善。\"\n  ],\n  \"seo_description\": \"專為敏弱肌設計的青研積雪草舒緩精華液 (30ml)。採用高達 5% 積雪草萃取，搭配泛醇與神經醯胺，幫助肌膚維持健康穩定。水感質地清爽好吸收，為成分黨提供安心、透明的日常保養選擇。\",\n  \"hashtags\": [\n    \"青研CHINGYEN\",\n    \"積雪草精華\",\n    \"敏弱肌保養\",\n    \"成分黨\",\n    \"舒緩精華\"\n  ]\n}", "input_tokens": 924, "output_tokens": 256}, "a8e1f2a048f293548695": {"text": "好的，這是一份針對「青研 CHINGYEN 多重玻尿酸保濕面膜」設計的電商文案，包含了標題、痛點描述、產品特色、成分解析、使用體驗和購買呼籲。\n\n---\n\n## 💧 青研 CHINGYEN 多重玻尿酸保濕面膜 💧\n\n### 【主標題與吸睛句】\n\n**告別乾荒！青研五重玻尿酸面膜 — 敷出水潤光澤肌，讓肌膚喝飽水！**\n\n---\n\n### 【第一段：點出痛點與需求 (引起共鳴)】\n\n**你是否也有這些困擾？**\n\n* 換季時肌膚乾燥脫皮，底妝總是浮粉不服貼？\n* 待在冷氣房一整天，臉部緊繃乾澀，細紋悄悄浮現？\n* 試過各種保濕產品，卻感覺水分只停留在表面，無法深層補水？\n\n保濕是肌膚保養的基石！青研 CHINGYEN 專為缺水、乾燥肌設計，給你超越單一保濕的**「多重深度補水」**體驗！\n\n---\n\n### 【第二段：產品核心特色 (強調獨家優勢)】\n\n**✨ 為什麼選擇青研多重玻尿酸保濕面膜？**\n\n1. **【五重玻尿酸 深度鎖水】**：獨家配方結合大分子與小分子玻尿酸，從肌膚表層到深層，層層滲透補水、鎖水、儲水，達到長效保濕，告別短暫水潤。\n2. **【天絲蠶絲膜布 極致服貼】**：採用輕薄透氣的**天絲蠶絲膜布**，親膚性極佳，能完美貼合臉部輪廓，讓 25ml 精華液完整導入肌底，敷臉同時享受頂級舒適感。\n3. **【海藻醣加乘 穩定修護】**：添加天然保濕因子「海藻醣」，有效舒緩乾燥不適，強化肌膚屏障，使肌膚更加穩定、健康。\n\n---\n\n### 【第三段：成分解析與功效 (專業度提升)】\n\n| 成分名稱 | 核心功效 | 適用對象 |\n| :--- | :--- | :--- |\n| **五重玻尿酸** | 強效抓水、補水、鎖水，改善乾燥細紋。 | 乾燥、缺水、熟齡肌 |\n| **海藻醣** | 舒緩保濕、穩定肌膚、提升防禦力。 | 敏感、換季不適肌 |\n| **25ml 高濃度精華** | 足量精華，敷完後可塗抹頸部與全身。 | 追求高效保濕者 |\n\n**產地保證：** 台灣製造，品質安心可靠。\n\n---\n\n### 【第四段：使用體驗與情境 (情境帶入)】\n\n**敷完立即有感！水潤透亮看得見！**\n\n* **敷感：** 膜布輕薄如第二層肌膚，精華液清爽不黏膩，敷臉過程舒適無負擔。\n* **效果：** 敷後肌膚立即飽滿水潤，乾燥緊繃感消失，摸起來 Q 彈柔嫩。\n* **妝前急救：** 上妝前敷一片，讓底妝完美服貼，持妝一整天不卡粉！\n* **夜間修護：** 睡前加強保濕，隔天醒來肌膚依然水嫩透亮。\n\n---\n\n### 【購買資訊與行動呼籲】\n\n**青研 CHINGYEN 多重玻尿酸保濕面膜**\n\n* **分類：** 美妝保養 / 面膜\n* **規格：** 5片 / 盒 (每片 25ml 精華)\n* **適用膚質：** 尤其推薦乾燥、缺水肌\n* **售價：** **NT$350**\n* **平均一片不到 NT$70！** 給予肌膚頂級的保濕饗宴，CP 值超高！\n\n**🛒 立即下單，體驗五重玻尿酸帶來的澎潤奇蹟！**\n\n**👉 點擊加入購物車，讓青研成為你肌膚的最佳水庫管理員！**", "input_tokens": 125, "output_tokens": 975}, "98c756e43637c8ede624": {"text": "## 青研 CHINGYEN 積雪草舒緩精華液 30ml\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN | 5%高濃度積雪草舒緩修護精華液 30ml | 敏弱肌必備，成分黨安心首選**\n\n---\n\n### 賣點\n\n1.  5%高濃度積雪草，高效舒緩泛紅不適。\n2.  協同神經醯胺，穩定修護肌膚屏障。\n3.  清爽水感精華，快速吸收不黏膩。\n4.  專為敏弱肌設計，溫和配方零負擔。\n\n---\n\n### SEO 描述\n\n成分黨必收！青研CHINGYEN積雪草舒緩精華液，專為敏弱肌設計。添加足量**5%**積雪草萃取，搭配泛醇與神經醯胺，有效鎮靜泛紅、穩定膚況。清爽水感質地，容量**30ml**，給予肌膚溫和高效的修護力。韓國原裝進口，讓肌膚重現健康光采。\n\n---\n\n### 標籤\n\n積雪草精華、敏弱肌保養、泛紅舒緩、成分黨、5%積雪草", "input_tokens": 262, "output_tokens": 291}, "49a530dc757b4cc93e87": {"text": "{\n  \"title\": \"晨光研選 小分子魚膠原蛋白粉 蔓越莓風味 30入 每日補充美麗基底\",\n  \"bullets\": [\n    \"每包含足量深海魚膠原蛋白胜肽 2500mg\",\n    \"清新酸甜蔓越莓風味，順口無腥味\",\n    \"小分子胜肽好吸收，日常保養更輕鬆\",\n    \"獨立隨身包裝，睡前沖泡一杯即可\"\n  ],\n  \"seo_description\": \"專為熟齡女性設計的日常美容保養品。晨光研選小分子魚膠原蛋白粉，每包提供 2500mg 高品質膠原蛋白胜肽。搭配清新蔓越莓風味，口感佳，方便每日持續補充。睡前一杯，由內而外維持青春光采。\",\n  \"hashtags\": [\n    \"膠原蛋白\",\n    \"蔓越莓\",\n    \"美容保養\",\n    \"日常補充\",\n    \"小分子胜肽\"\n  ]\n}", "input_tokens": 929, "output_tokens": 239}, "7fd2bee00c7da7327416": {"text": "## 晨光研選 2500mg 小分子魚膠原蛋白粉 | 蔓越莓風味 30入 | 啟動青春光采\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  深海魚膠原胜肽，極致小分子好吸收。\n2.  足量添加 **2500mg 膠原蛋白**，美麗有感。\n3.  酸甜 **蔓越莓** 風味，告別腥味輕鬆飲。\n4.  台灣製造，獨立隨身包裝，方便攜帶。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n30-50歲女性的青春秘密！晨光研選嚴選高品質深海魚膠原蛋白，每包含足量**2500mg**，高效補充流失的**膠原蛋白**。搭配酸甜**蔓越莓**風味，口感極佳。小分子胜肽好吸收，讓您由內而外散發光采，維持水潤彈性。每日一包，輕鬆養成逆齡美肌。\n\n---\n\n### 標籤 (3-5 個)\n\n\\#膠原蛋白 \\#美容保養 \\#小分子胜肽 \\#蔓越莓 \\#青春美麗", "input_tokens": 285, "output_tokens": 285}, "b9f0fa13db69aa5e2721": {"text": "**商品標題：**\nVolta 1.4kg 輕量無線吸塵器 18000Pa強吸力 小家庭/寵物適用 續航35分鐘\n\n**賣點：**\n1. 18000Pa 颶風吸力，深層清潔毛髮塵蟎。\n2. 僅 1.4kg 主機輕量化，高處清潔不費力。\n3. 標準模式續航達 35分鐘，滿足全室清潔。\n4. 附床墊吸頭與軟絨滾筒，適用多種地面。\n\n**SEO 描述：**\nVolta 手持無線吸塵器 Lite 專為小坪數與寵物家庭設計。主機僅1.4kg，輕巧好收納。具備18000Pa強勁吸力，輕鬆吸除寵物毛髮。標準模式續航長達35分鐘，清潔無死角。\n\n**標籤：**\n#無線吸塵器\n#寵物吸塵器\n#18000Pa\n#小家庭\n#Volta", "input_tokens": 676, "output_tokens": 248}, "c02b06a830e22129c4e0": {"text": "## 【青研 CHINGYEN】胺基酸淨膚平衡洗面乳 | pH5.5弱酸配方 敏弱肌專用 120ml\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1.  專為敏弱肌設計，溫和不刺激。\n2.  pH5.5 弱酸平衡，維持健康膚況。\n3.  高濃度胺基酸，洗後水潤不緊繃。\n4.  綿密泡沫質地，深層清潔零負擔。\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n成分黨必收！青研 CHINGYEN 胺基酸洗面乳，採用親膚 **pH5.5** 弱酸性配方，溫和清潔毛孔髒污。內含高純度**胺基酸**界面活性劑與洋甘菊萃取，有效舒緩敏弱肌膚。**120ml** 大容量，台灣製造，給您最安心的潔顏體驗。\n\n---\n\n### 標籤 (3-5 個)\n\n胺基酸洗面乳, 敏弱肌, pH5.5, 弱酸性, 洋甘菊", "input_tokens": 260, "output_tokens": 278}, "51b1980034fa13f8420a": {"text": "{\n  \"title\": \"晨光研選 金盞花萃取 游離型葉黃素膠囊 30mg高劑量 60粒裝\",\n  \"bullets\": [\n    \"足量添加游離型葉黃素 30mg，提供日常所需營養。\",\n    \"採用黃金比例 5:1 游離型，分子小好吸收。\",\n    \"專為長時間使用 3C 螢幕的上班族設計。\",\n    \"每日一粒，一盒 60粒，輕鬆持續補充，維持舒適感。\"\n  ],\n  \"seo_description\": \"晨光研選葉黃素膠囊，專為長時間使用電腦的上班族設計。每粒含高劑量游離型葉黃素 30mg，搭配玉米黃素 6mg，比例 5:1 好吸收。植物膠囊，全素可食。一盒 60粒，每日持續補充，是您晶亮舒適的日常保養選擇。\",\n  \"hashtags\": [\n    \"葉黃素\",\n    \"游離型葉黃素\",\n    \"3C族保養\",\n    \"晶亮舒適\",\n    \"晨光研選\"\n  ]\n}", "input_tokens": 908, "output_tokens": 275}, "11920f8198e28572b8aa": {"text": "## 晨光研選 金盞花萃取葉黃素膠囊\n\n---\n\n### 商品標題 (不超過 60 字)\n\n**晨光研選 晶亮有神！游離型金盞花葉黃素 30mg 高效配方 60粒**\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1.  高濃度 30mg 葉黃素，足量守護。\n2.  游離型好吸收，黃金 5:1 晶亮比例。\n3.  專為長時間螢幕族設計，維持清晰舒適。\n4.  台灣製造，植物膠囊，安心無負擔。\n\n---\n\n### SEO 描述 (不超過 120 字)\n\n上班族必備眼部保健！晨光研選金盞花萃取游離型**葉黃素**，每粒含足量 **30mg** 葉黃素與玉米黃素，黃金比例高效吸收。一盒 **60粒**，每日一粒，輕鬆對抗長時間螢幕藍光，維持晶亮舒適，是您數位生活的最佳後盾。\n\n---\n\n### 標籤 (3-5 個)\n\n葉黃素、眼部保健、游離型、上班族、藍光", "input_tokens": 264, "output_tokens": 292}, "201a78f781c90a372273": {"text": "## 溯源焙所｜台東鹿野紅烏龍茶包 20包入｜蜜香熟果風味 三角立體茶包 台灣在地送禮首選\n\n---\n\n### 賣點\n\n1.  台東鹿野產區，重發酵工藝，風味醇厚。\n2.  獨特蜜香與熟果韻味，尾韻自然回甘。\n3.  三角立體茶包，快速釋放茶葉完整香氣。\n4.  精緻盒裝，內含 20包，送禮體面大方。\n\n---\n\n### SEO 描述\n\n【溯源焙所】嚴選台東鹿野優質**紅烏龍**，重發酵工藝，釋放天然蜜香與熟果風味。每盒內含 **20包** 三角立體茶包，輕鬆享受高品質台灣茶。送禮自用皆宜，品味回甘醇厚的好滋味。\n\n---\n\n### 標籤\n\n#紅烏龍 #鹿野 #台灣茶 #送禮 #三角茶包", "input_tokens": 271, "output_tokens": 236}, "af7563768f3867b2e5a6": {"text": "{\n  \"title\": \"青研 CHINGYEN 多重玻尿酸保濕面膜 5片入 天絲蠶絲膜布 敏弱肌水潤首選\",\n  \"bullets\": [\n    \"採用五重玻尿酸配方，層層補水鎖水。\",\n    \"輕薄服貼的天絲蠶絲膜布，親膚透氣。\",\n    \"添加海藻醣，溫和舒緩乾燥不適。\",\n    \"一盒內含 5片 精華面膜，密集修護。\"\n  ],\n  \"seo_description\": \"專為敏弱肌設計的青研多重玻尿酸保濕面膜，內含 5片。採用五重玻尿酸與海藻醣精華，提供肌膚長效保濕。搭配輕盈的天絲蠶絲膜布，服貼度高，精華液好吸收不黏膩。成分透明，溫和不刺激，是乾燥缺水肌的日常補水選擇。\",\n  \"hashtags\": [\n    \"敏弱肌保養\",\n    \"玻尿酸面膜\",\n    \"天絲面膜\",\n    \"成分黨\",\n    \"保濕補水\"\n  ]\n}", "input_tokens": 924, "output_tokens": 267}, "68c443f602248c9a2d1c": {"text": "## 青研 CHINGYEN 多重玻尿酸保濕面膜 (5片入)\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN | 五重玻尿酸保濕面膜 (5片入) | 敏弱肌適用、天絲服貼鎖水**\n\n---\n\n### 賣點\n\n1.  五重玻尿酸精華，深層注水瞬效補水。\n2.  採用頂級天絲膜布，極致親膚不刺激。\n3.  溫和無添加配方，敏弱肌也能安心敷。\n4.  添加海藻醣，強化肌膚長效保水屏障。\n\n---\n\n### SEO 描述\n\n專為敏弱肌與成分黨設計！青研 CHINGYEN 多重玻尿酸保濕面膜，採用五重**玻尿酸**精華，提供肌膚長效且深層的保濕。每盒 **5片**入，搭配頂級**天絲**蠶絲膜布，輕薄服貼，將 25ml 精華完整導入。溫和無添加，台灣製造，讓乾燥缺水肌重現水潤光澤。\n\n---\n\n### 標籤\n\n玻尿酸面膜、敏弱肌保養、天絲面膜、成分黨、深度保濕", "input_tokens": 262, "output_tokens": 287}, "1c41a7afbd847ca288a5": {"text": "{\n  \"title\": \"晨光研選 五益菌複方益生菌粉 30包 | 每包100億CFU | 調整體質 維持消化道機能\",\n  \"bullets\": [\n    \"五株專利益生菌配方，幫助維持消化道機能。\",\n    \"每包足量 100億 活菌，提供日常所需補給。\",\n    \"台灣製造，粉末隨身包設計，方便攜帶食用。\",\n    \"適合外食族及作息不規律者，輕鬆補充好菌。\"\n  ],\n  \"seo_description\": \"晨光研選五益菌複方益生菌粉，專為忙碌的上班族與外食族設計。每盒 30包，每日一包，輕鬆補充好菌。嚴選五株專利益生菌，每包提供 100億 CFU 活性益生菌。持續補充，幫助您調整體質，維持消化道健康，讓生活更順暢無負擔。\",\n  \"hashtags\": [\n    \"益生菌\",\n    \"消化保健\",\n    \"調整體質\",\n    \"晨光研選\",\n    \"上班族\"\n  ]\n}", "input_tokens": 915, "output_tokens": 275}, "37a8b4a2a4e20e4b61c2": {"text": "{\n  \"title\": \"Volta 無線降噪藍牙耳機 Pro 2：42dB深度降噪，續航32小時，藍牙5.4雙裝置切換\",\n  \"bullets\": [\n    \"最高 42dB 主動降噪，有效阻隔通勤雜音。\",\n    \"總續航力達 32小時，單次使用可達 8 小時。\",\n    \"藍牙5.4 穩定連線，支援雙裝置快速切換。\",\n    \"單耳僅 4.8g 輕量設計，IPX4 等級防水。\"\n  ],\n  \"seo_description\": \"Volta Pro 2 無線耳機專為專注設計。提供 42dB 深度降噪，無論在捷運或咖啡廳都能保持清晰。採用最新藍牙5.4技術，連線穩定且支援多工切換。總續航力長達 32小時，滿足全日工作與通勤需求。輕巧設計，提供舒適的聆聽體驗。\",\n  \"hashtags\": [\n    \"無線耳機\",\n    \"降噪耳機\",\n    \"藍牙54\",\n    \"通勤必備\",\n    \"遠距工作\"\n  ]\n}", "input_tokens": 789, "output_tokens": 281}, "e3303903c09206c4a0ad": {"text": "好的，這是一份為您的「五益菌複方益生菌粉」設計的電商文案。\n\n文案分為**標題與副標**、**痛點切入與產品介紹**、**核心賣點與成分強調**、**使用方式與購買呼籲**四個部分，適用於電商平台（如蝦皮、Momo、官網）的商品詳情頁。\n\n---\n\n## 晨光研選｜五益菌複方益生菌粉 (30入)\n\n### 🌟 標題與副標\n\n**主標題：** 【晨光研選】五益菌複方益生菌粉 (30入) | 100億活菌高規格 | 順暢有感，體內環保新選擇\n\n**副標題：** 專利五菌株，高定殖率，調整體質，告別卡關人生！\n\n**價格標示：** NT$1180\n\n---\n\n### 💡 痛點切入與產品介紹 (開頭吸引力)\n\n**你是否也有這些困擾？**\n\n* 😫 每天排便不規律，總覺得「卡卡」不順暢？\n* 🤢 飲食不正常，經常感到消化不良、腹脹不適？\n* 🤧 換季時容易敏感，想從根本調整體質？\n* 💻 久坐辦公室，缺乏運動，代謝力下降？\n\n**👉 你的腸道，需要更強大的後盾！**\n\n**【晨光研選】五益菌複方益生菌粉**，專為現代人設計的「有感」益生菌。我們不只追求數量，更重視菌株的**品質與定殖率**，用科學配方，為你打造健康、輕盈的體內環境。\n\n---\n\n### 💪 核心賣點與成分強調 (產品優勢)\n\n#### 🏆 為什麼選擇【晨光研選】五益菌？\n\n**1. 【高規格】每包足量 100 億 CFU 活菌**\n我們承諾，每一包 2g 的益生菌粉，都含有高達 **100 億 CFU** 的專利益生菌。足夠的活菌數，才能確保益生菌順利通過胃酸考驗，抵達並定殖於腸道，發揮最大效益。\n\n**2. 【高定殖】五株專利複合配方，協同作用更強大**\n嚴選五種經過科學驗證的專利益生菌株，針對亞洲人體質設計，具有高耐受性與高定殖率，能有效幫助：\n* **維持消化道機能**\n* **改變細菌叢生態**\n* **調整過敏體質**\n\n**3. 【高便利】獨立包裝，隨時隨地補充能量**\n採用獨立鋁箔隨身包裝 (30入/盒)，方便攜帶，確保益生菌活性不流失。粉末劑型，口感佳，可直接食用，隨時為健康加分。\n\n**4. 【高品質】台灣製造，安心保證**\n全程台灣 GMP 廠生產製造，品質嚴格把關，無添加人工色素、香料或防腐劑。給您最純淨、最安心的保健選擇。\n\n---\n\n### 📋 規格資訊與食用指南\n\n| 項目 | 內容 |\n| :--- | :--- |\n| **商品名稱** | 五益菌複方益生菌粉 |\n| **品牌** | 晨光研選 |\n| **規格** | 30包 / 盒 (每包 2g) |\n| **售價** | NT$1180 |\n| **分類** | 保健食品 / 消化保健 |\n| **主成分** | 五株專利益生菌、益生質 (詳細成分請見包裝) |\n| **活菌數** | 每包保證 100 億 CFU |\n| **劑型** | 細緻粉末隨身包 |\n| **產地** | 台灣 |\n\n#### 💡 建議食用方式：\n\n* **每日用量：** 每日 1 包。\n* **最佳時機：** 建議**空腹**或**睡前**食用，此時胃酸濃度較低，有助於益生菌順利通過。\n* **食用方法：** 可直接倒入口中配水食用，或加入低於 40°C 的飲品中。\n* **注意事項：** 請勿搭配熱飲或高溫食物食用，以免影響益生菌活性。\n\n---\n\n### 🛒 結語與購買呼籲\n\n**別讓「卡關」成為你的日常！**\n\n健康從腸道開始，給自己一個輕盈、順暢的機會。\n\n現在就選擇【晨光研選】五益菌複方益生菌粉，啟動你的體內環保計畫！\n\n**👉 立即加入購物車，體驗有感順暢！**", "input_tokens": 133, "output_tokens": 1121}, "b9add45dae454183c5e1": {"text": "## 溯源焙所 冷萃掛耳咖啡 淺焙水果調 10入\n\n---\n\n### 商品標題\n\n**溯源焙所 耶加雪菲 淺焙 冷萃掛耳咖啡 10包入 | 柑橘莓果花香調**\n\n---\n\n### 賣點\n\n1.  精選衣索比亞**耶加雪菲**，**淺焙**保留細緻果酸。\n2.  綻放柑橘、莓果與優雅花香的層次風味。\n3.  專為冷萃設計，快速浸泡，內含**10包**。\n4.  職人級精品咖啡，隨時享受冰鎮純粹。\n\n---\n\n### SEO 描述\n\n溯源焙所推出精品**淺焙**冷萃掛耳，嚴選衣索比亞**耶加雪菲**產區，風味乾淨明亮，帶有柑橘與莓果調性。專利掛耳設計，輕鬆製作冰鎮黑咖啡。每盒**10包**，適合追求高品質與便利性的咖啡愛好者。\n\n---\n\n### 標籤\n\n#冷萃咖啡\n#耶加雪菲\n#掛耳包\n#淺焙\n#精品咖啡", "input_tokens": 812, "output_tokens": 273}, "ec200bec8cbbaf859ebc": {"text": "好的，這是一個高品質的台灣紅烏龍茶產品，文案需要強調其獨特的風味、便利性以及產地優勢。\n\n以下為您撰寫的電商文案，包含標題、痛點/情境描述、產品介紹、核心賣點、規格與購買資訊，適用於電商平台（如蝦皮、MOMO、官網）。\n\n---\n\n## 🍵 【溯源焙所】台灣紅烏龍茶包 (三角立體茶包 20入)\n\n**台東鹿野重發酵 | 蜜香熟果韻 | 快速回甘**\n\n### 🌟 標語/主視覺文案\n\n**一口入魂的台灣味：忙碌生活中的頂級茶歇。**\n\n---\n\n### 📌 情境與痛點描述 (為什麼你需要它？)\n\n* **你是否厭倦了：** 辦公室裡只有索然無味的速溶飲品，想喝好茶卻苦於沒有時間慢慢沖泡？\n* **你是否渴望：** 在快節奏的生活中，也能隨時享受一杯層次豐富、香氣迷人的台灣好茶？\n* **【溯源焙所】聽到你的心聲！** 我們將頂級的台東鹿野紅烏龍，裝入便利的三角立體茶包中，讓你隨時隨地，90秒品味極致。\n\n---\n\n### ✨ 產品介紹：台東鹿野的陽光滋味\n\n**商品名稱：** 台灣紅烏龍茶包 (三角立體茶包 20入)\n**品牌：** 溯源焙所\n**售價：** NT$380\n\n這款紅烏龍茶，來自台灣東部最純淨的土地——**台東鹿野**。我們堅持採用**重發酵**工藝，結合烏龍的醇厚與紅茶的甘甜，創造出獨一無二的「紅烏龍」風味。\n\n**它不只是茶包，它是你專屬的行動茶席。**\n\n---\n\n### 🏆 核心賣點 (為什麼選擇溯源焙所紅烏龍？)\n\n#### 1. 【獨特風味】蜜香熟果，層次豐富\n\n紅烏龍的魅力在於其複雜而迷人的香氣：\n\n* **前調：** 撲鼻而來的天然**蜜香**，溫暖而療癒。\n* **中調：** 飽滿的**熟果**韻味，如同品嚐頂級紅酒般醇厚。\n* **尾韻：** 入口滑順，喉韻極佳，**回甘**迅速持久，久泡不澀。\n\n#### 2. 【頂級產地】台東鹿野，溯源保證\n\n* 嚴選台東鹿野優質茶菁，地理環境優越，日夜溫差大，造就茶葉內含物質豐富。\n* **重發酵工藝：** 提升茶葉的甘甜度與耐泡度，茶湯呈現琥珀紅色，溫潤暖胃。\n\n#### 3. 【便利升級】三角立體茶包，完美釋放\n\n* 採用食品級**三角立體茶包**設計，讓茶葉有足夠空間完整舒展，釋放出最完整的香氣與滋味，媲美手沖現泡。\n* 每包獨立包裝（3g），定量精準，方便攜帶，隨時享受高品質茶飲。\n\n#### 4. 【快速享受】90秒黃金沖泡法\n\n* 無需複雜茶具，只需熱水：**95°C 熱水，浸泡 90 秒**，即可達到最佳風味。\n* 適合辦公、旅行、居家，隨時隨地享受一杯頂級好茶。\n\n---\n\n### 📝 產品規格一覽\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **品牌** | 溯源焙所 |\n| **商品** | 台灣紅烏龍茶包 |\n| **規格** | 20包 / 盒 (每包 3g) |\n| **淨含量** | 60g |\n| **產地** | 台灣 台東鹿野 |\n| **發酵度** | 重發酵 |\n| **風味特色** | 蜜香、熟果香、尾韻回甘 |\n| **建議沖泡** | 95°C 熱水，浸泡 90 秒 |\n| **保存期限** | 詳見包裝 |\n| **售價** | NT$380 |\n\n---\n\n### 🛒 立即購買，讓生活多一點溫潤的甜\n\n**【溯源焙所】台灣紅烏龍茶包，是你對生活品質的堅持。**\n\n* **送禮自用兩相宜：** 精緻盒裝，體面大方。\n* **辦公室必備：** 告別咖啡因焦慮，享受溫和回甘的茶飲。\n\n**點擊加入購物車，將台東鹿野的頂級風味帶回家！**", "input_tokens": 131, "output_tokens": 1126}, "02324d580ce63f368202": {"text": "{\n  \"title\": \"Volta 無線吸塵器 Lite | 18000Pa強吸力 1.4kg輕量 35分鐘續航 小宅寵物適用\",\n  \"bullets\": [\n    \"18000Pa強勁吸力，深層清潔毛髮與微塵，適用硬地板與地毯。\",\n    \"機身僅1.4kg，單手操作輕巧，高處清潔與收納無負擔。\",\n    \"標準模式續航達35分鐘，滿足小坪數家庭全屋清潔需求。\",\n    \"0.5L集塵盒與濾網可水洗，維護簡單，保持吸塵器高效能。\"\n  ],\n  \"seo_description\": \"Volta 手持無線吸塵器 Lite 專為小家庭與寵物族群設計。具備18000Pa強勁吸力，輕鬆吸除毛髮與粉塵。機身僅1.4kg，搭配35分鐘續航，清潔無死角。可水洗集塵盒，維護成本低。\",\n  \"hashtags\": [\n    \"無線吸塵器\",\n    \"寵物家庭\",\n    \"小坪數家電\",\n    \"18000Pa\",\n    \"輕量化\"\n  ]\n}", "input_tokens": 783, "output_tokens": 296}, "ca6bd3edb03197a8cb99": {"text": "{\n  \"title\": \"溯源焙所 台東鹿野 紅烏龍茶包 蜜香熟果風味 三角立體茶包 20包入\",\n  \"bullets\": [\n    \"台東鹿野產區，重發酵工藝，風味醇厚。\",\n    \"獨特蜜香與熟果韻味，尾韻自然回甘。\",\n    \"立體三角茶包，完整釋放茶葉精華。\",\n    \"內含 20包，台灣茶愛好者日常品飲或送禮首選。\"\n  ],\n  \"seo_description\": \"來自台東鹿野的特色紅烏龍茶，經過重發酵，展現迷人的蜜香與熟果風味。三角立體茶包設計，讓茶葉完整舒展。每盒 20包 入，是台灣茶愛好者日常品飲或年節送禮的最佳選擇。\",\n  \"hashtags\": [\n    \"台灣茶\",\n    \"紅烏龍\",\n    \"鹿野茶\",\n    \"三角茶包\",\n    \"送禮推薦\"\n  ]\n}", "input_tokens": 922, "output_tokens": 247}, "8641c4eb712aa95909af": {"text": "## 青研 CHINGYEN 多重玻尿酸保濕面膜 5片入\n\n---\n\n### 商品標題 (不超過 60 個字)\n\n**青研 CHINGYEN 五重玻尿酸保濕面膜 5片入 敏弱肌友善 清爽補水**\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1. 五重**玻尿酸**，層層滲透，緩解乾燥缺水。\n2. 採用**天絲**蠶絲膜布，輕薄服貼，精華液好吸收。\n3. 無酒精香料，溫和配方，敏弱肌適用。\n4. 一盒**5片**裝，NT$350，日常密集保濕。\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n敏弱肌換季必備！青研多重**玻尿酸**保濕面膜，專為乾燥缺水肌設計。添加五重玻尿酸與海藻醣，提供肌膚長效水潤。輕盈的**天絲**蠶絲膜布，服貼舒適，精華液好吸收不黏膩。一盒**5片**入，讓您輕鬆享受溫和高效的居家保濕護理。台灣製造，成分透明，安心有感。\n\n---\n\n### 標籤 (3-5 個)\n\n#敏弱肌 #成分黨 #保濕面膜 #玻尿酸 #天絲面膜", "input_tokens": 817, "output_tokens": 331}, "d268b76ec3424ee95aaa": {"text": "## Volta 無線降噪藍牙耳機 Pro 2\n\n### 商品標題\n**Volta Pro 2 旗艦降噪耳機 42dB深度降噪 藍牙5.4 總續航32小時** (47字)\n\n---\n\n### 賣點\n1. 42dB深度主動降噪，隔絕通勤與辦公室喧囂。\n2. 總續航**32小時**，長途差旅或遠距工作電力不中斷。\n3. **藍牙5.4**穩定連線，支援雙裝置無縫切換。\n4. 單耳僅4.8g輕量設計，IPX4防水防汗。\n\n---\n\n### SEO 描述\nVolta Pro 2 專為通勤族與遠距工作者設計。搭載42dB深度**降噪**技術，有效提升專注力。採用最新**藍牙5.4**晶片，連線穩定低延遲。搭配充電盒提供**32小時**總續航，滿足一週使用需求。輕巧舒適，IPX4防水，是您高效辦公與移動的最佳音訊夥伴。 (109字)\n\n---\n\n### 標籤\n#降噪耳機\n#藍牙耳機\n#通勤必備\n#遠距辦公\n#Volta", "input_tokens": 682, "output_tokens": 298}, "b7da0279ca8b6b47b426": {"text": "{\n\"title\": \"溯源焙所 低溫烘焙 綜合堅果 350g 無調味 | 嚴選四種頂級堅果\",\n\"bullets\": [\n\"職人低溫慢焙，保留堅果原始營養與風味。\",\n\"嚴選腰果、核桃、杏仁、夏威夷豆四種頂級堅果。\",\n\"堅持完全無調味，無添加糖、鹽、油，純粹健康。\",\n\"健身族群與注重原型食物家庭的優質點心選擇。\"\n],\n\"seo_description\": \"溯源焙所的綜合堅果 350g，採用職人低溫烘焙技術，非油炸，完整保留堅果的天然油脂與香氣。堅持無調味、無添加糖鹽，是追求原型食物的家庭或健身族群補充優質脂肪與膳食纖維的最佳選擇。\",\n\"hashtags\": [\n\"溯源焙所\",\n\"綜合堅果\",\n\"原型食物\",\n\"健身飲食\",\n\"無調味堅果\"\n]\n}", "input_tokens": 920, "output_tokens": 240}, "8cc2ed63ae4a1e974e27": {"text": "## 晨光研選 小分子魚膠原蛋白胜肽 2500mg 蔓越莓風味 30入\n\n---\n\n### 賣點\n\n1.  足量 2500mg 小分子膠原蛋白胜肽。\n2.  清爽酸甜蔓越莓風味，順口無腥味。\n3.  獨立隨身包裝，方便每日持續補充。\n4.  每日補充，為美麗打下穩固基礎。\n\n---\n\n### SEO 描述\n\n晨光研選專為注重日常保養的女性設計。每包含有足量 **2500mg** 小分子魚**膠原蛋白**胜肽，分子小好吸收。搭配清新的**蔓越莓**風味，讓您輕鬆告別魚腥味。每日一包，持續補充美麗所需，由內而外散發光采。台灣製造，品質安心。\n\n---\n\n### 標籤\n\n*   膠原蛋白粉\n*   蔓越莓\n*   美容保養\n*   晨光研選\n*   小分子膠原", "input_tokens": 822, "output_tokens": 238}, "07ddd306c0c58d7cfd30": {"text": "## 台灣電商平台商品文案\n\n### 商品標題\n\n**【溯源焙所】原型食物首選！低溫烘焙綜合堅果 350g。純淨無調味，家庭健身高纖健康零食。** (57字)\n\n---\n\n### 賣點 (4 條)\n\n1. 低溫慢焙，非油炸製程，完整保留堅果原始營養。\n2. 堅持無調味、無添加糖鹽，純淨無負擔。\n3. 富含膳食纖維與優質油脂，健身族群必備補給。\n4. 嚴選腰果、核桃、杏仁、夏威夷豆，安心溯源。\n\n---\n\n### SEO 描述\n\n尋找最純淨的健康零食？溯源焙所低溫烘焙**綜合堅果**是您原型食物的首選。一罐**350g**，採用非油炸製程，完整保留堅果的天然風味與營養。堅持**無調味**、無添加，提供家庭與健身族群最安心的高纖維、好油脂補給。 (109字)\n\n---\n\n### 標籤 (Tags)\n\n原型食物, 綜合堅果, 健身零食, 低溫烘焙, 無調味", "input_tokens": 269, "output_tokens": 288}, "99a6cae296211492f3d6": {"text": "{\n\"title\": \"Volta 65W 氮化鎵三孔快充頭 雙C一A配置 輕巧體積 差旅首選\",\n\"bullets\": [\n\"65W大功率輸出，筆電手機同時快充。\",\n\"採用第三代氮化鎵，體積輕巧易攜帶。\",\n\"USB-C×2 + USB-A×1 三孔配置，滿足多裝置需求。\",\n\"僅重112g，5.2cm立方設計，不佔插座空間。\"\n],\n\"seo_description\": \"Volta 65W 氮化鎵三孔快充充電器，專為商務人士與多裝置使用者設計。具備雙USB-C及單USB-A輸出，總功率達65W，可同時為筆電、平板、手機進行快充。採用先進氮化鎵技術，體積小巧僅112g，輕鬆收納。支援過壓過流過熱保護，充電更安心。\",\n\"hashtags\": [\n\"氮化鎵\",\n\"65W快充\",\n\"差旅必備\",\n\"多孔充電\",\n\"Volta\"\n]\n}", "input_tokens": 790, "output_tokens": 266}, "2f7050ae6cae4f2cb5b5": {"text": "## Volta 65W 氮化鎵三孔快充充電器\n\n---\n\n### 商品標題\n\n**Volta 65W 氮化鎵【迷你三孔】快充頭：筆電手機同步滿電，出差族輕量化首選！** (39 字)\n\n---\n\n### 賣點 (4 條)\n\n1.  65W大功率輸出，筆電手機極速回血。 (16 字)\n2.  採用第三代氮化鎵，體積輕巧不佔空間。 (16 字)\n3.  2C1A三孔設計，同時充飽三裝置。 (15 字)\n4.  僅112g超輕量，差旅不再攜帶多顆充電器。 (17 字)\n\n---\n\n### SEO 描述\n\n專為出差族與多裝置使用者設計的Volta **65W** **氮化鎵**快充。極致輕巧的機身，提供穩定高效的65W輸出。具備USB-C×2+USB-A×1**三孔**設計，讓您一機搞定筆電、平板與手機的充電需求。體積小巧，收納方便，是您辦公與旅行的最佳電力夥伴。 (98 字)\n\n---\n\n### 標籤\n\n*   氮化鎵充電器\n*   65W快充\n*   出差必備\n*   多孔充電\n*   PD快充頭", "input_tokens": 261, "output_tokens": 330}, "cbc0fd23ac246a656f6a": {"text": "**【晨光研選】五益菌複方益生菌粉**\n\n---\n\n### 商品標題\n\n**【晨光研選】外食族/夜貓族必備！五益菌複方益生菌粉 30包，每包 100億，順暢有感。** (54 字)\n\n---\n\n### 賣點\n\n1.  每包足量 **100億** 專利益生菌，有感調整體質。\n2.  獨立隨身包裝 (**30包**)，隨時補充，空腹或睡前皆可。\n3.  緩解外食油膩負擔，幫助消化道機能更順暢。\n4.  台灣製造，五株專利菌，打造黃金體質基底。\n\n---\n\n### SEO 描述\n\n專為外食族與作息不規律上班族設計的消化道保健品。晨光研選五益菌複方**益生菌**粉，一盒內含 **30包**，每包提供高達 **100億** CFU 專利益生菌。有效調整體質，幫助消化，告別卡卡人生。粉末劑型，方便攜帶，隨時補充好菌，維持順暢。 (98 字)\n\n---\n\n### 標籤\n\n#益生菌 #消化保健 #外食族 #100億 #順暢", "input_tokens": 271, "output_tokens": 314}, "7bedbd96d70fe3e72980": {"text": "{\n  \"title\": \"溯源焙所 耶加雪菲 淺焙冷萃掛耳咖啡 柑橘莓果花香風味 10包入\",\n  \"bullets\": [\n    \"衣索比亞 耶加雪菲，經典水果調性。\",\n    \"精選淺焙工藝，釋放飽滿柑橘與莓果香。\",\n    \"專為冷萃設計，輕鬆享受職人級冰咖啡。\",\n    \"每盒 10包獨立包裝，隨時隨地享用精品風味。\"\n  ],\n  \"seo_description\": \"溯源焙所推出精品冷萃掛耳咖啡。嚴選衣索比亞 耶加雪菲 產區豆，採用精準 淺焙 工藝，完整保留柑橘、莓果與優雅花香。只需簡單浸泡，即可享受清爽回甘的冰咖啡。每盒內含 10包，是夏日或辦公室的理想選擇。\",\n  \"hashtags\": [\n    \"耶加雪菲\",\n    \"冷萃咖啡\",\n    \"淺焙咖啡\",\n    \"精品咖啡\",\n    \"掛耳包\"\n  ]\n}", "input_tokens": 919, "output_tokens": 265}, "a81e8f5e35c97068099c": {"text": "好的，這是一份為您的「晨光研選 金盞花萃取葉黃素膠囊」量身打造的電商文案。\n\n這份文案包含**標題、痛點切入、產品介紹（FAB法則）、成分亮點、食用建議**與**行動呼籲**，適用於商品詳情頁面或廣告素材。\n\n---\n\n## 🌟 晨光研選 | 金盞花萃取葉黃素膠囊 (60粒/盒)\n\n### 【主標題】\n**高濃度游離型 30mg！晶亮守護神🛡️ 告別模糊感，迎接清晰「視」界！**\n\n### 【副標題】\n**黃金比例 5:1 葉黃素：玉米黃素 | 純淨植物膠囊 | 台灣製造**\n\n---\n\n## 🔍 你的「視」界，是否正在發出警訊？\n\n**📱 低頭族、💻 上班族、📚 學生族、👵 熟齡族… 你是否也面臨以下困擾？**\n\n* **長時間緊盯螢幕，感到乾澀疲勞？**\n* **閱讀或看遠處時，開始出現模糊感？**\n* **光線變化時，適應力變差？**\n* **想為家人或自己，找到最有效的晶亮保養品？**\n\n**別讓數位生活偷走你的清晰！「晨光研選」為你帶來最高規格的晶亮守護方案！**\n\n---\n\n## ✨ 產品介紹：晨光研選 金盞花萃取葉黃素膠囊\n\n### 💎 【核心優勢】黃金配方，高效吸收\n\n| 特點 (Feature) | 優勢 (Advantage) | 效益 (Benefit) |\n| :--- | :--- | :--- |\n| **高劑量游離型葉黃素 30mg** | 吸收率高，不需經過消化分解即可直接吸收。 | 快速補充黃斑部所需營養，達到國際認可的有效保養劑量。 |\n| **葉黃素：玉米黃素 5:1 黃金比例** | 完美模擬人體黃斑部結構的比例。 | 雙重防護，協同作用，提供最全面的晶亮保護。 |\n| **純淨植物膠囊 (素食可食)** | 無動物明膠、無多餘添加物。 | 溫和不刺激，素食者也能安心食用，負擔更少。 |\n| **嚴選金盞花萃取物** | 來自天然來源，品質穩定可靠。 | 提供高品質的葉黃素與玉米黃素，來源安心有保障。 |\n\n### 🔬 【成分亮點】高規格，看得見的晶亮\n\n**1. 游離型 Lutein 30mg：**\n* **高濃度、高吸收：** 每日一粒，輕鬆達到美國眼科醫學會建議的每日保養劑量。\n* **分子小：** 比酯化型葉黃素更易於人體吸收利用。\n\n**2. 玉米黃素 Zeaxanthin 6mg：**\n* 與葉黃素共同構成防護網，有效過濾藍光，加強保護力。\n\n**3. 台灣製造，品質保證：**\n* 嚴格品管，安心可靠，給您最優質的保健選擇。\n\n---\n\n## 💡 誰最需要「晨光研選」葉黃素？\n\n* **長時間使用電腦、手機的上班族。**\n* **課業繁重，需要專注力的學生。**\n* **經常夜間開車或對光線敏感者。**\n* **想維持晶亮，延緩退化的熟齡族群。**\n\n---\n\n## 💊 食用建議與產品規格\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **建議食用** | 每日 1 粒，建議隨餐或飯後食用，搭配油脂有助於吸收。 |\n| **淨含量** | 60 粒 / 盒 (約 2 個月份量) |\n| **劑型** | 植物膠囊 (全素可食) |\n| **產地** | 台灣 |\n| **售價** | **NT$890** (高規格配方，超值價格) |\n\n---\n\n## 🛒 立即行動！給你的靈魂之窗最好的呵護！\n\n**別再等待模糊感找上門！**\n\n選擇「晨光研選 金盞花萃取葉黃素膠囊」，用最高效的配方，為你的晶亮視野打下堅實基礎。\n\n**現在就加入購物車，開啟你的清晰人生！**\n\n👉 **[點擊此處，立即購買]**\n\n**【晨光研選】— 研選優質，守護健康。**", "input_tokens": 125, "output_tokens": 1081}, "a48895f6e83460497cf9": {"text": "## 溯源焙所 台灣紅烏龍茶包 專屬文案\n\n---\n\n### 商品標題\n**溯源焙所 台東鹿野紅烏龍 三角立體茶包 蜜香熟果風味 20包**\n\n### 賣點\n1. 台東鹿野產區，職人級重發酵紅烏龍\n2. 溫潤蜜香與熟果尾韻，醇厚回甘\n3. 3g三角立體茶包，完整釋放茶韻\n4. 精緻盒裝 20包，送禮自用皆宜\n\n### SEO 描述\n溯源焙所精選台東鹿野紅烏龍，重發酵工藝造就獨特的蜜香熟果風味。三角立體茶包設計，讓茶葉完整舒展。每盒 20包，方便快速沖泡，是茶葉愛好者日常品飲或年節送禮的最佳選擇。\n\n### 標籤\n紅烏龍, 鹿野茶, 蜜香, 茶包, 送禮", "input_tokens": 815, "output_tokens": 233}, "b130b88268a1a340f6e0": {"text": "## Volta 無線降噪藍牙耳機 Pro 2 商品文案\n\n---\n\n### 商品標題 (不超過 60 個字)\n\n**Volta Pro 2 旗艦無線降噪耳機 | 42dB深度靜音、32小時長效續航、藍牙5.4**\n\n---\n\n### 賣點 (4 條，每條不超過 30 個字)\n\n1. **深度降噪**：42dB 旗艦級主動降噪，隔絕通勤喧囂，瞬間進入專注模式。\n2. **超長續航**：總電力達 **32小時**，從早到晚不斷電，告別電量焦慮。\n3. **穩定連線**：最新 **藍牙5.4** 晶片，支援雙裝置切換，會議娛樂不中斷。\n4. **極致輕盈**：單耳僅 4.8g 輕量設計，IPX4 防水，全天配戴舒適無負擔。\n\n---\n\n### SEO 描述 (不超過 120 個字)\n\n專為通勤族與遠距工作者打造。Volta Pro 2 搭載深度**降噪**技術，有效阻擋環境噪音，讓您隨時保持專注。採用最新**藍牙5.4**晶片，連線穩定。超長**32小時**續航力，輕巧舒適，是您高效工作的最佳夥伴。NT$3290 即可擁有旗艦級音質與靜謐體驗。\n\n---\n\n### 標籤 (3-5 個)\n\n#降噪耳機\n#藍牙5.4\n#遠距辦公\n#通勤必備\n#32小時續航", "input_tokens": 260, "output_tokens": 386}, "de7bba1e22f9615d2006": {"text": "## Volta 手持無線吸塵器 Lite\n\n---\n\n### 商品標題\n\n**Volta Lite 輕巧無線吸塵器 | 1.4kg超輕量、18000Pa強吸力 | 小宅/寵物家庭必備**\n\n---\n\n### 賣點\n\n1.  **18000Pa** 颶風吸力，深層清除毛髮塵蟎。\n2.  僅 **1.4kg**，單手操作不費力，收納超省空間。\n3.  標準模式續航 **35分鐘**，全室清潔不斷電。\n4.  專屬床墊吸頭，集塵盒可水洗，零耗材更環保。\n\n---\n\n### SEO 描述\n\n**Volta Lite 輕量無線吸塵器，小坪數家庭首選！具備 18000Pa 強勁吸力，搭配僅 1.4kg 輕盈機身，打掃不費力。標準續航長達 35分鐘，輕鬆應對寵物毛髮與日常灰塵。**\n\n---\n\n### 標籤\n\n#無線吸塵器\n#小坪數\n#寵物家庭\n#輕量化\n#Volta", "input_tokens": 254, "output_tokens": 275}, "57197bf029d65c67a868": {"text": "{\n  \"title\": \"青研 CHINGYEN 胺基酸溫和洗面乳 120ml 敏弱肌友善 pH5.5\",\n  \"bullets\": [\n    \"胺基酸配方溫和清潔，不帶走肌膚必要水分。\",\n    \"弱酸性 pH5.5 貼近肌膚酸鹼值，維持健康屏障。\",\n    \"添加洋甘菊萃取，幫助舒緩肌膚不適感。\",\n    \"泡沫綿密細緻，洗後膚感清爽不緊繃。\"\n  ],\n  \"seo_description\": \"專為敏弱肌設計的胺基酸洗面乳 (120ml)。採用溫和胺基酸界面活性劑，搭配弱酸性 pH5.5，洗淨同時維持肌膚平衡。綿密泡沫，洗後清爽不緊繃，成分黨安心選擇。\",\n  \"hashtags\": [\n    \"敏弱肌\",\n    \"胺基酸洗面乳\",\n    \"pH5.5\",\n    \"溫和清潔\",\n    \"成分黨\"\n  ]\n}", "input_tokens": 922, "output_tokens": 250}, "aed7e2d027dd8b64fb00": {"text": "**【溯源焙所】冷萃掛耳咖啡 淺焙水果調 10入**\n\n---\n\n### 商品標題\n\n溯源焙所｜耶加雪菲 冷萃掛耳咖啡 淺焙水果花香調 10包入\n\n### 賣點\n\n1. 嚴選衣索比亞 耶加雪菲，產地溯源。\n2. 淺焙保留柑橘莓果，風味清新不酸澀。\n3. 冰鎮4小時，輕鬆享受精品冷萃。\n4. 一盒10包獨立包裝，隨時品味極致。\n\n### SEO 描述\n\n溯源焙所推出夏日必備的冷萃掛耳咖啡。嚴選耶加雪菲，採精準淺焙，完整釋放柑橘與莓果花香。一盒10包，只需簡單浸泡，即可享受冰涼清爽的精品咖啡。是您辦公室或戶外活動的最佳選擇。\n\n### 標籤\n\n#冷萃咖啡 #耶加雪菲 #精品咖啡 #掛耳包 #淺焙", "input_tokens": 268, "output_tokens": 238}, "898774be01fd7077c0d9": {"text": "## Volta 65W 氮化鎵 三孔快充充電器 差旅首選 筆電手機同步滿速\n\n---\n\n### 賣點\n\n1. **65W 總輸出，滿足筆電快充需求。**\n2. **第三代**氮化鎵**技術，體積輕巧。**\n3. **2C1A**三孔**配置，同時供電三裝置。**\n4. **多重安全防護，穩定可靠輸出。**\n\n---\n\n### SEO 描述\n\nVolta **65W 氮化鎵三孔**快充，專為出差族設計。極致輕巧體積，提供 **65W** 總輸出，高效能同時為筆電、手機、平板供電。2C1A配置，多重保護，滿足多裝置使用者對速度與便攜的需求。\n\n---\n\n### 標籤\n\n**氮化鎵, 65W, 三孔快充, 差旅必備, PD快充**", "input_tokens": 683, "output_tokens": 224}, "65b5ae8b0206866497fe": {"text": "好的，這是一個針對美容保養市場的高級保健品。文案將著重於**高效吸收、美味方便**以及**雙重美妍功效**。\n\n以下提供三種不同風格的電商文案，您可以依照您的行銷平台和目標客群選擇使用：\n\n---\n\n## 電商文案方案\n\n### 方案一：【主打功效與科學】—— 適合官方網站、專業平台或注重成分的客群\n\n**標題：【晨光研選】逆齡秘密武器！小分子魚膠原蛋白粉 (蔓越莓風味) - 2500mg 高效吸收，啟動青春光采！**\n\n**💰 售價：NT$1,580 (30入/盒)**\n\n---\n\n#### 【產品介紹：科學高效，美妍升級】\n\n**妳的膠原蛋白，真的被吸收了嗎？**\n\n傳統膠原蛋白分子大，吸收率低，效果不明顯。**晨光研選** 專注於高效吸收，為妳帶來革命性的美容配方：**小分子魚膠原蛋白胜肽**。\n\n**✨ 為什麼選擇晨光研選？**\n\n1. **極小分子，直達美麗：**\n   嚴選深海魚膠原蛋白，透過先進水解技術，製成**小分子胜肽**，每包足量 **2500mg**。分子量極小，能迅速被身體利用，效率遠勝一般膠原蛋白。\n\n2. **雙重美妍，加乘效果：**\n   特別添加 **80mg 維生素C**，不僅是天然的抗氧化劑，更是**膠原蛋白合成的關鍵輔酶**。維生素C與膠原蛋白完美搭配，讓美妍效果加倍升級。\n\n3. **美味無腥味，輕鬆養成習慣：**\n   告別傳統魚膠原的腥味困擾！我們採用清新的**天然蔓越莓風味**，酸甜可口。粉末細緻，可直接食用或溫水沖泡，讓保養成為一種享受。\n\n---\n\n#### 【規格與食用建議】\n\n* **規格：** 30包/盒 (每包 3g)\n* **主成分：** 深海魚膠原蛋白胜肽 2500mg、維生素C 80mg\n* **產地：** 台灣製造，品質安心保證。\n* **建議食用：** 每日 1 包，建議睡前以溫水沖泡飲用，把握夜間黃金修復期。\n\n---\n\n### 方案二：【主打體驗與生活情境】—— 適合社群媒體、KOL 推薦或追求儀式感的客群\n\n**標題：睡前一杯，喚醒妳的少女肌！【晨光研選】蔓越莓膠原蛋白粉，讓妳由內而外散發光芒！**\n\n---\n\n#### 【妳的睡前儀式，決定妳的明天】\n\n忙碌了一天，是時候給自己一個寵愛時刻了！\n\n**晨光研選** 懂妳對美麗的渴望，將頂級保養濃縮在一包酸甜的蔓越莓粉末中。\n\n**🌙 睡前黃金保養術：**\n\n每天晚上，當身體進入休息模式，正是吸收營養的最佳時機。\n\n* **Step 1：** 撕開一包 **晨光研選** 小分子膠原蛋白粉。\n* **Step 2：** 倒入溫水中，享受蔓越莓的清新香氣。\n* **Step 3：** 輕鬆飲用，讓 2500mg 的高效膠原胜肽與維生素C，在妳熟睡時，為妳的肌膚進行深度修護。\n\n**一覺醒來，迎接更水潤、更Q彈的自己！**\n\n---\n\n#### 【產品特色】\n\n* **[方便攜帶]** 獨立隨身包設計，出差、旅行、辦公室，隨時隨地補充美麗能量。\n* **[口感滿分]** 零腥味！清爽蔓越莓風味，就像喝果汁一樣輕鬆無負擔。\n* **[高效吸收]** 採用小分子胜肽，吸收率高，美麗不等待。\n* **[台灣製造]** 嚴格品管，安心食用。\n\n**別讓時間帶走妳的光采！現在就加入購物車，開始妳的逆齡旅程！**\n\n---\n\n### 方案三：【短版促購文案】—— 適合限時優惠、廣告投放或 LINE 訊息\n\n**【限時搶購】NT$1,580 (原價 NT$XXXX) - 買二送一！**\n\n**🔥 告別乾燥暗沉！妳的肌膚需要高效「膠原C」！**\n\n**【晨光研選】小分子魚膠原蛋白粉 (蔓越莓風味)**\n\n**💎 為什麼熱銷？**\n\n1. **超速吸收：** 2500mg 小分子魚膠原胜肽，直達肌底。\n2. **美白加乘：** 80mg 維生素C，促進膠原合成，透亮有感！\n3. **極致美味：** 酸甜蔓越莓，無腥味，天天吃也不膩。\n4. **方便性：** 30包獨立包裝，睡前一包，美麗輕鬆達標。\n\n**👉 睡前一杯，讓妳的肌膚喝飽水！**\n\n**點擊連結，立即擁有妳的青春光采！**\n[購買連結/活動頁面]", "input_tokens": 144, "output_tokens": 1252}, "fe26e02333506ff90761": {"text": "**品牌：晨光研選**\n**目標受眾：25-45 歲長時間使用螢幕的上班族**\n\n---\n\n### 商品標題 (50 字)\n\n晨光研選 游離型葉黃素 30mg 60粒 長時間螢幕族群 日常晶亮保養\n\n---\n\n### 賣點 (4 條，每條不超過 30 字)\n\n1. 足量葉黃素 30mg，游離型態好吸收。\n2. 添加玉米黃素 6mg，符合黃金比例配方。\n3. 60粒裝，每日一粒，持續補充兩個月。\n4. 專為長時間使用螢幕的上班族設計。\n\n---\n\n### SEO 描述 (99 字)\n\n長時間面對電腦、手機，需要穩定補充。晨光研選游離型**葉黃素**，單顆足量**30mg**，搭配玉米黃素黃金比例。全素植物膠囊，一盒**60粒**，為您的日常晶亮保養打底，適合注重持續性補充的您。\n\n---\n\n### 標籤 (4 個)\n\n葉黃素、上班族、眼部保養、保健食品", "input_tokens": 801, "output_tokens": 277}, "13914b0f573fd57c034c": {"text": "## 青研 CHINGYEN 胺基酸溫和洗面乳 120ml\n\n---\n\n### 商品標題\n\n**青研 CHINGYEN 胺基酸溫和洗面乳 120ml | 弱酸性 pH5.5 | 敏弱肌與成分黨適用**\n\n---\n\n### 賣點\n\n1.  高濃度胺基酸界面活性劑，溫和潔淨。\n2.  弱酸性 pH5.5，貼近肌膚健康環境。\n3.  豐盈綿密泡沫，帶走髒污不緊繃。\n4.  添加洋甘菊萃取，洗後膚感舒適。\n\n---\n\n### SEO 描述\n\n青研CHINGYEN專為敏弱肌設計。採用溫和**胺基酸**界面活性劑，維持肌膚水潤。**pH5.5**弱酸配方，洗淨同時呵護皮脂膜健康。綿密泡沫質地，容量**120ml**，每日潔顏無負擔，成分黨安心首選。\n\n---\n\n### 標籤\n\n敏弱肌、胺基酸、弱酸性、洗面乳、成分黨", "input_tokens": 815, "output_tokens": 261}, "61b7f48be0071d568721": {"text": "**晨光研選 專業保健系列**\n\n---\n\n### 商品文案\n\n#### 商品標題 (42 字)\n**晨光研選 五益菌複方益生菌粉 30包入 每日100億CFU 幫助消化道維持平衡**\n\n#### 賣點 (4 條，每條不超過 30 字)\n1. 五株專利益生菌，每包足量 **100億** CFU。\n2. 台灣製造，粉末隨身包，方便上班族隨時補充。\n3. 每日一包，持續補充好菌，幫助維持消化道機能。\n4. 適合外食族群，為忙碌生活建立體內平衡。\n\n#### SEO 描述 (109 字)\n晨光研選五益菌複方**益生菌**粉，專為忙碌的**外食族**與**上班族**設計。每盒**30包**，每包含有高達**100億**CFU的五株專利菌，幫助維持消化道穩定。粉末劑型，方便攜帶，每日持續補充，溫和調整體質，讓您輕鬆面對生活挑戰。\n\n#### 標籤\n#益生菌 #消化道保健 #上班族必備 #外食族 #100億", "input_tokens": 808, "output_tokens": 294}, "00e813accc61a17d1b6d": {"text": "**品牌：溯源焙所**\n**分類：food / 零食**\n**售價：NT$590**\n\n---\n\n### 商品標題\n**溯源焙所 職人低溫烘焙 綜合堅果 350g 無調味 原味堅果的純粹風味**\n\n---\n\n### 賣點\n1. 低溫烘焙，保留堅果原始香氣與營養。\n2. 嚴選四種頂級堅果，完全無調味、無糖鹽。\n3. 健身族群與家庭常備，優質油脂補充。\n4. 無油炸製程，口感清脆，風味純粹。\n\n---\n\n### SEO 描述\n溯源焙所堅持職人精神，為您獻上低溫烘焙**綜合堅果**。內含腰果、核桃、杏仁、夏威夷豆，淨含量**350g**。製程無油炸、完全**無調味**、無添加糖鹽，保留堅果最原始的營養與純粹風味。是注重原型食物的家庭與健身族群，隨時補充優質油脂與膳食纖維的最佳選擇。\n\n---\n\n### 標籤\n原型食物, 綜合堅果, 無調味, 低溫烘焙, 健身零食", "input_tokens": 813, "output_tokens": 290}, "00aea5e3c4732a06a5fa": {"text": "## 青研 CHINGYEN 積雪草舒緩精華液 30ml\n\n### 商品標題\n**青研 CHINGYEN 5%積雪草舒緩精華 30ml | 泛醇神經醯胺 敏弱肌安心保養**\n\n---\n\n### 賣點\n1.  高濃度 **5% 積雪草**萃取，溫和穩定膚況。\n2.  水感輕盈精華，清爽好吸收不黏膩。\n3.  泛醇與神經醯胺，幫助維持肌膚健康。\n4.  韓國製造，成分透明，敏弱肌安心選用。\n\n---\n\n### SEO 描述\n專為敏弱肌設計的青研 CHINGYEN 積雪草舒緩精華液 **30ml**。添加高達 **5% 積雪草**萃取，搭配泛醇與神經醯胺，幫助肌膚安撫不適感，維持油水平衡。水感質地清爽不厚重，是成分黨換季穩定膚況的首選精華。\n\n---\n\n### 標籤\n積雪草, 敏弱肌, 精華液, 成分黨, 舒緩", "input_tokens": 817, "output_tokens": 266}, "8a26786006c4e77006fa": {"text": "好的，這是一份為您的「冷萃掛耳咖啡 淺焙水果調」設計的電商文案。\n\n文案將分為**標題、核心賣點、產品描述、風味鑑賞、沖泡指南**和**購買呼籲**六個部分，適用於電商平台（如蝦皮、官網、Momo）的商品詳情頁面。\n\n---\n\n## ☕ 溯源焙所 | 冷萃掛耳咖啡 淺焙水果調 (10入)\n\n**標題建議：**\n* **主標題 (吸睛)：** 【夏日必備】冷萃掛耳咖啡 | 淺焙水果調 | 耶加雪菲花香爆棚 | 10入組\n* **副標題 (關鍵字)：** 柑橘莓果風味 | 冰鎮即享 | 辦公室/戶外手沖首選 | NT$460\n\n---\n\n## 一、 核心賣點 (Banner/首圖文案)\n\n**✨ 30秒抓住你的味蕾！**\n\n1. **極致冷萃體驗：** 專為冰鎮設計，告別苦澀，釋放清爽果酸。\n2. **頂級產區風味：** 嚴選衣索比亞耶加雪菲，淺焙保留最純淨的柑橘、莓果與花香。\n3. **方便快捷：** 隨身掛耳包，無論辦公室或戶外，隨時享受精品手沖級美味。\n\n---\n\n## 二、 產品描述：讓夏天充滿果香的秘密\n\n**【溯源焙所】冷萃掛耳咖啡 淺焙水果調**，是我們為追求輕盈、潔淨風味的您，精心打造的夏日咖啡解決方案。\n\n我們深知，炎炎夏日，您需要的是一杯能瞬間提振精神，卻又毫無負擔的冰咖啡。這款掛耳包不僅能熱沖，更完美適配「冷萃」需求，讓您輕鬆在家或辦公室，製作出媲美專業咖啡館的冰鎮水果調咖啡。\n\n**✅ 產品規格一覽：**\n* **分類：** Food / 咖啡\n* **售價：** NT$460 (10入/盒)\n* **淨含量：** 10包 / 盒 (每包 12g)\n* **產區：** 衣索比亞 耶加雪菲 (精品咖啡的代名詞)\n* **烘焙度：** 淺焙 (最大化保留產區水果與花卉香氣)\n\n---\n\n## 三、 風味鑑賞：舌尖上的熱帶果園\n\n這款咖啡選用來自咖啡發源地——衣索比亞的頂級耶加雪菲豆，以精準的淺焙工藝處理，確保每一口都能品嚐到產區最純粹的風味。\n\n| 風味描述 | 柑橘 (Citrus) | 莓果 (Berry) | 花香 (Floral) |\n| :--- | :--- | :--- | :--- |\n| **口感** | 清新明亮，酸質活潑 | 甜感飽滿，層次豐富 | 尾韻優雅，乾淨持久 |\n| **特色** | 彷彿咬下一口多汁的橙子，酸甜平衡 | 帶有覆盆子與藍莓的微酸甜感 | 茉莉花、伯爵茶般的細膩香氣 |\n\n**💡 淺焙的魅力：** 淺焙能將咖啡豆中的果酸和花香完整釋放，特別適合製作冷萃，口感乾淨、甜度高，完全不需額外加糖。\n\n---\n\n## 四、 沖泡指南：冷熱皆宜，隨心所欲\n\n**1. 【極致推薦】冷萃製作 (隔夜冰鎮法)**\n\n* **步驟 1：** 準備一個密封容器（如玻璃瓶或保溫瓶）。\n* **步驟 2：** 放入 1 包掛耳咖啡包，倒入 **180ml** 的常溫飲用水。\n* **步驟 3：** 輕輕搖晃，確保咖啡粉完全浸濕。\n* **步驟 4：** 放入冰箱冷藏 **8-12 小時**。\n* **步驟 5：** 取出咖啡包，即可享受一杯清爽、無苦澀的冰鎮水果調冷萃咖啡！\n\n**2. 【經典享受】熱沖手沖法**\n\n* **步驟 1：** 撕開掛耳包，掛於杯緣。\n* **步驟 2：** 注入約 **20ml** 90°C 熱水進行悶蒸（約 30 秒）。\n* **步驟 3：** 分 2-3 次緩慢注水至總水量 **150-180ml**。\n* **步驟 4：** 移除掛耳包，享受熱咖啡帶來的奔放花香。\n\n---\n\n## 五、 購買呼籲與保存資訊\n\n**【溯源焙所】堅持給您最好的：**\n\n* **新鮮烘焙：** 採小批量新鮮烘焙，確保風味處於最佳狀態。\n* **氮氣填充：** 獨立包裝內充填氮氣，有效隔絕氧氣，鎖住咖啡香氣。\n* **產地溯源：** 嚴選單一產區精品豆，風味純淨，品質保證。\n\n**🛒 現在就將這份夏日清爽帶回家！**\n\n**保存方式：** 請存放於室溫陰涼處，避免陽光直射與高溫潮濕。為確保最佳風味，開封後請盡快飲用完畢。\n\n---\n\n### 🌟 顧客常見問題 (FAQ)\n\n**Q1：冷萃和熱沖的風味差異大嗎？**\nA：風味差異顯著。熱沖能展現更強烈的花香與明亮的酸質；冷萃則會帶來更柔和的酸度、更高的甜感以及絲滑的口感，非常清爽。\n\n**Q2：一包咖啡可以沖幾次？**\nA：建議一次沖泡完畢。若製作冷萃，請勿重複使用，以免風味變淡。\n\n**Q3：這款咖啡會很酸嗎？**\nA：由於是淺焙，會帶有精品咖啡特有的「果酸」，但這是一種活潑、愉悅的柑橘酸，而非令人不適的醋酸。冷萃後酸度會更為柔和。", "input_tokens": 131, "output_tokens": 1456}, "16017cf3b7b7839a6b03": {"text": "這是一份針對「青研 CHINGYEN 積雪草舒緩精華液」設計的電商文案，包含主標題、產品亮點、成分解析與購買資訊。\n\n---\n\n## 【青研 CHINGYEN】積雪草舒緩精華液 30ml\n\n### 🌟 主標題：敏感肌的「滅火器」！5%高濃度積雪草，一滴鎮定不安膚況\n\n---\n\n### 🌿 產品核心承諾：告別泛紅、乾燥、不穩定！\n\n當肌膚發出警訊，你需要的不只是保濕，更是高效的「修護與舒緩」。\n青研 CHINGYEN 積雪草舒緩精華液，專為亞洲敏弱肌設計，以黃金比例配方，快速穩定膚況，從根源強化肌膚屏障，讓您重拾健康光采。\n\n---\n\n### ✨ 三大修護亮點，打造穩定健康肌\n\n#### 1. 高濃度舒緩：5% 積雪草萃取 (Centella Asiatica)\n我們採用高達 **5%** 的積雪草活性成分，能迅速滲透肌底，有效舒緩肌膚泛紅、乾燥引起的刺激與不適。是換季、醫美後、或日常敏感時期的最佳穩定劑。\n\n#### 2. 深層補水修復：2% 泛醇 (Panthenol / 維他命原B5)\n泛醇是公認的肌膚修復明星。2% 的添加量能深入肌膚底層，提供長效保濕，同時強化肌膚的自癒能力，加速修復受損屏障。\n\n#### 3. 屏障守護神：神經醯胺 (Ceramide)\n添加關鍵的神經醯胺成分，如同肌膚的天然水泥，有效填補角質層空隙，防止水分流失，建立強韌的保護網，讓肌膚不再輕易受到外界刺激。\n\n---\n\n### 💧 質地與使用體驗\n\n*   **輕盈水感：** 告別黏膩厚重！精華液呈現清爽的水感質地，一抹即化，秒速吸收，不造成肌膚負擔，油性肌和痘痘肌也能安心使用。\n*   **敏弱適用：** 配方溫和，無酒精、無香精、無色素。經測試證實，適用於所有膚質，特別推薦給正在經歷敏感、乾燥、換季不適的肌膚。\n*   **韓國原裝：** 堅持韓國專業研發與製造，品質與功效值得信賴。\n\n---\n\n### 📋 產品規格一覽\n\n| 項目 | 內容 |\n| :--- | :--- |\n| **品牌** | 青研 CHINGYEN |\n| **商品名稱** | 積雪草舒緩精華液 |\n| **容量** | 30ml |\n| **主成分** | 積雪草萃取 5%、泛醇 2%、神經醯胺 |\n| **質地** | 水感精華 |\n| **適用膚質** | 所有膚質，敏弱肌適用 |\n| **產地** | 韓國 |\n| **售價** | **NT$780** |\n\n---\n\n### 🛒 立即體驗：給肌膚最溫柔的穩定力量！\n\n別讓敏感成為日常困擾。現在就將【青研 CHINGYEN 積雪草舒緩精華液】加入您的保養清單，感受 5% 高濃度積雪草帶來的即時舒緩與長效修護！\n\n**NT$780 / 30ml**\n**點擊購買，讓肌膚從此穩定、透亮！**", "input_tokens": 124, "output_tokens": 832}, "5a7719dccb158b9af90e": {"text": "這是一份為「青研 CHINGYEN 胺基酸溫和洗面乳」設計的電商文案，強調其溫和、平衡與舒緩的特性。\n\n---\n\n## 💎 青研 CHINGYEN 胺基酸溫和洗面乳 (120ml)\n\n**【告別緊繃乾澀！肌膚的溫柔守護者】**\n**NT$420**\n\n### 🌟 主標題：洗出水潤光澤！弱酸性pH5.5，敏感肌也愛上的胺基酸潔顏體驗\n\n---\n\n### 【前言：你還在用會讓臉繃緊的洗面乳嗎？】\n\n許多清潔產品為了追求「強力去油」，反而破壞了肌膚的天然屏障，導致洗後乾燥、泛紅、甚至越洗越敏感。\n\n**青研 CHINGYEN** 專為追求溫和高效清潔的您設計。我們相信，好的清潔是保養的第一步，必須在帶走髒污的同時，留下肌膚最需要的滋潤與平衡。\n\n### 【三大核心賣點，給肌膚最溫柔的呵護】\n\n#### 💧 1. 溫和高效：高純度胺基酸界面活性劑\n\n我們採用親膚性極佳的**胺基酸界面活性劑**作為主要清潔成分。它能有效深入毛孔帶走髒污與多餘油脂，但絕不帶走肌膚原有的水分。\n\n*   **效果：** 深層清潔，洗後肌膚不緊繃、不乾澀。\n*   **適用：** 任何膚質，特別推薦乾燥肌、敏感肌。\n\n#### ⚖️ 2. 維持平衡：完美弱酸性 pH5.5\n\n健康的肌膚表面呈弱酸性（pH值約5.5）。本產品完美貼合肌膚的天然酸鹼值，有助於：\n\n*   **鞏固屏障：** 維持肌膚天然保護膜（酸性保護膜）。\n*   **穩定膚況：** 降低刺激，預防因清潔過度引起的泛紅與不適。\n\n#### 🌿 3. 舒緩鎮靜：洋甘菊萃取添加\n\n添加珍貴的**洋甘菊萃取**，以其卓越的舒緩與鎮靜功效聞名。在清潔的同時，為肌膚帶來溫和的療癒：\n\n*   **功效：** 舒緩環境壓力造成的肌膚不適，提升肌膚穩定度。\n\n### 【極致使用體驗：綿密泡沫，奢華享受】\n\n輕輕加水搓揉，立即產生**豐盈、細緻、有彈性的綿密泡沫**。\n\n細緻的泡沫能減少手部與肌膚的摩擦，讓清潔過程更舒適，輕鬆帶走殘妝與空污微粒，沖洗後不殘留滑膩感，只留下清爽水潤的膚觸。\n\n---\n\n### 【誰需要青研胺基酸溫和洗面乳？】\n\n*   ✅ 正在尋找不含皂鹼、不刺激的溫和洗面乳。\n*   ✅ 敏感肌、換季容易泛紅不適者。\n*   ✅ 追求洗後水潤、不緊繃的潔顏感受。\n*   ✅ 醫美課程後需要極度溫和清潔產品者。\n\n---\n\n### 📋 商品規格一覽\n\n| 項目 | 說明 |\n| :--- | :--- |\n| **品牌** | 青研 CHINGYEN |\n| **分類** | 美妝保養 / 清潔 |\n| **容量** | 120ml |\n| **售價** | NT$420 |\n| **主要成分** | 胺基酸界面活性劑、洋甘菊萃取 |\n| **pH值** | 弱酸性 pH5.5 |\n| **質地** | 綿密泡沫 |\n| **產地** | 台灣 |\n\n**👉 立即下單，給您的肌膚一個溫柔的開始！**", "input_tokens": 119, "output_tokens": 900}, "e0959e529484f7961738": {"text": "這是一份針對「Volta 手持無線吸塵器 Lite」所撰寫的電商文案，適用於商品詳情頁面或廣告投放。\n\n---\n\n## ⚡ Volta 手持無線吸塵器 Lite：輕盈解放，效率加倍 ⚡\n\n**售價：NT$4,680**\n\n厭倦了笨重、線材纏繞的傳統吸塵器嗎？Volta 專為追求效率與輕巧的您設計了「Lite」系列。它不僅是您的日常清潔好幫手，更是您隨時隨地保持居家整潔的秘密武器。\n\n---\n\n### 【主文案：告別沉重，迎接瞬吸新時代】\n\n#### 💡 輕量化設計，清潔無死角\n\n我們知道，打掃不該是體力活。Volta Lite 將主機重量控制在驚人的 **1.4kg**，比四瓶礦泉水還輕。無論是舉高清潔天花板蜘蛛網，或是單手操作清理車內空間，都能輕鬆搞定，讓您告別手痠疲勞。\n\n#### 🌪️ 輕巧身軀，蘊含強勁吸力\n\n別被 Lite 的輕巧外觀所迷惑。它搭載高效能馬達，提供 **18000Pa 的瞬吸力**。從地板毛髮、地毯深層灰塵到零食碎屑，都能一網打盡，確保您的居家環境徹底潔淨。\n\n#### 🔋 續航力升級，一次搞定全屋\n\n標準模式下提供 **35 分鐘長效續航**，足夠您完成一般公寓或套房的深度清潔。無需頻繁充電，讓打掃流程順暢不中斷。\n\n---\n\n### 【核心特色解析：全能配件，多場景應用】\n\nVolta Lite 不只是一台地板吸塵器，透過隨附的「全能配件」，讓您輕鬆應對家中所有角落：\n\n| 配件名稱 | 適用場景 | 清潔效益 |\n| :--- | :--- | :--- |\n| **軟絨滾筒** | 木地板、磁磚、短毛地毯 | 有效吸附微塵與毛髮，同時保護地板不刮傷。 |\n| **床墊吸頭** | 床鋪、沙發、布藝家具 | 專門對付塵螨、皮屑與過敏原，提升睡眠品質。 |\n| **縫隙吸頭** | 窗框、牆角、家具縫隙 | 深入狹窄難清的死角，徹底清除累積的灰塵。 |\n\n---\n\n### 【使用者體驗：清潔衛生，輕鬆維護】\n\n✅ **0.5L 大容量集塵盒：** 減少傾倒次數，滿足單次清潔需求。\n✅ **全機可水洗設計：** 集塵盒與濾網皆可直接水洗，晾乾後重複使用，環保又衛生，省下耗材費用。\n✅ **一鍵倒塵：** 傾倒垃圾不髒手，保持清潔的優雅。\n\n---\n\n### 【商品規格一覽】\n\n| 項目 | 規格細節 |\n| :--- | :--- |\n| **商品名稱** | 手持無線吸塵器 Lite |\n| **品牌** | Volta |\n| **分類** | 家電 / 清潔 |\n| **吸力** | 18000Pa |\n| **重量** | 1.4kg (主機) |\n| **續航時間** | 標準模式 35 分鐘 |\n| **集塵盒容量** | 0.5L (可水洗) |\n| **標準配件** | 縫隙吸頭、床墊吸頭、軟絨滾筒 |\n| **建議售價** | NT$4,680 |\n\n---\n\n### 🛒 【立即行動】\n\n**NT$4,680** 即可擁有輕巧、強勁、多功能的 Volta Lite！\n\n**現在下單，享受居家清潔的輕盈與效率！**\n\n👉 [點擊此處，立即搶購]\n👉 [加入購物車，解放您的雙手]", "input_tokens": 115, "output_tokens": 920}, "747b0cddea66a94d6248": {"text": "這是一個針對「Volta 無線降噪藍牙耳機 Pro 2」設計的電商文案，分為標題、核心賣點、規格細節與行動呼籲，適用於商品頁面或詳細廣告投放。\n\n---\n\n## 【Volta Pro 2】一鍵進入寂靜宇宙：最高 42dB 深度降噪，重新定義你的專注力。\n\n### 🚀 核心標語：你的世界，由你掌控。\n\n厭倦了通勤時的嘈雜、辦公室的喧囂，或是運動時被雜音干擾？Volta Pro 2 不只是耳機，它是你隨身攜帶的「寂靜艙」。結合頂級降噪技術與無縫連接體驗，讓您隨時隨地保持最佳狀態。\n\n---\n\n### 🔇 賣點一：極致深度降噪，隔絕 99% 噪音干擾\n\nVolta Pro 2 搭載業界領先的降噪技術，讓您真正享受「只聽我想聽」的自由。\n\n*   **最高 42dB 主動降噪 (ANC)：** 深度降噪效果，能有效消除飛機引擎聲、捷運轟鳴聲及環境低頻噪音。無論身處何地，都能瞬間進入專屬的寧靜空間。\n*   **智慧通透模式：** 無需取下耳機，即可清晰捕捉環境人聲，安全與便利兼顧。\n\n### 🔋 賣點二：超長續航力，電力從早到晚不間斷\n\n專為長途旅行和高強度工作設計，Volta Pro 2 讓您徹底擺脫電量焦慮。\n\n*   **單次續航 8 小時：** 滿足一整天的工作或娛樂需求。\n*   **總續航力 32 小時 (含充電盒)：** 即使週末出遊，也無需頻繁充電。\n\n### 🔗 賣點三：藍牙 5.4 + 雙裝置切換，工作娛樂無縫接軌\n\nVolta Pro 2 採用最新的藍牙技術，提供更穩定、更低延遲的連接體驗。\n\n*   **藍牙 5.4：** 連線速度更快，訊號更穩定，音訊傳輸效率大幅提升。\n*   **智慧雙裝置連接：** 同時連接手機與筆電，自動判斷音源並切換。當您在電腦上觀看影片時，手機來電也能立即接聽，無需手動配對，效率加倍。\n\n### ✨ 賣點四：輕盈無感，全天候舒適佩戴\n\n我們深知舒適度是長時間佩戴的關鍵。\n\n*   **單耳僅重 4.8g：** 輕巧設計，幾乎無感，長時間佩戴耳朵也無負擔。\n*   **IPX4 防水等級：** 有效防汗、防潑水。無論是健身房揮汗如雨，或是在戶外遇到小雨，都能安心使用。\n\n---\n\n### 📋 產品規格一覽 (Volta Pro 2)\n\n| 項目 | 規格細節 | 核心優勢 |\n| :--- | :--- | :--- |\n| **降噪深度** | 最高 42dB 主動降噪 | 業界頂級降噪，深度隔絕環境噪音 |\n| **總續航力** | 單次 8H / 總計 32H | 超長電力，告別電量焦慮 |\n| **連線技術** | 藍牙 5.4 | 連線穩定，低延遲 |\n| **特殊功能** | 支援雙裝置切換 | 工作娛樂，無縫多工處理 |\n| **防水等級** | IPX4 | 防汗防潑水，運動通勤皆宜 |\n| **單耳重量** | 4.8g | 輕盈舒適，長時間佩戴無負擔 |\n\n---\n\n### 🛒 立即體驗寂靜的力量！\n\n**Volta 無線降噪藍牙耳機 Pro 2**\n\n**定價：NT$3290**\n\n投資 Volta Pro 2，就是投資您的專注力與生活品質。\n\n**點擊下方按鈕，即刻擁有你的個人寂靜艙！**", "input_tokens": 127, "output_tokens": 946}, "45389cadc8f28b0f2942": {"text": "這是一份針對 Volta 65W 氮化鎵三孔快充充電器的電商文案，涵蓋了標題、痛點解決、核心賣點、規格細節以及購買呼籲。\n\n---\n\n## 【Volta】65W 氮化鎵三孔快充充電器\n\n**⚡️ 告別笨重！小體積，大能量！您的筆電、手機、平板，一顆搞定！**\n\n### 💰 限時優惠價：NT$990\n\n---\n\n## 💡 產品核心賣點 (Product Highlights)\n\n| 特點 | 說明 | 效益 |\n| :--- | :--- | :--- |\n| **65W 高功率輸出** | 單孔 Type-C 可達 65W 輸出。 | 輕鬆支援 MacBook、Dell XPS 等主流筆記型電腦快充。 |\n| **第三代 GaN 氮化鎵** | 採用最新 GaN 3.0 技術。 | 體積極致縮小，發熱量更低，效率更高。 |\n| **三孔設計 (2C+1A)** | 雙 Type-C + 單 USB-A 輸出。 | 同時為三台設備充電，出門只需帶一顆。 |\n| **極致輕巧便攜** | 尺寸僅 5.2 x 5.2 x 2.9 cm，重量僅 112g。 | 比傳統 65W 充電器小 50% 以上，旅行、通勤無負擔。 |\n\n---\n\n## 🔋 厭倦了笨重又慢的充電器嗎？\n\n您是否還在為每一台設備準備一個專屬的充電頭？筆電的充電器又大又重，手機的充電器又慢又佔插座？\n\n**Volta 65W 氮化鎵三孔快充，專為追求效率與極簡生活的您而生！**\n\n### 🚀 賣點一：筆電級快充，瞬間滿血復活 (65W Power)\n\n別再讓您的筆電等待！Volta 65W 氮化鎵充電器提供強勁的 65W 輸出，足以應付市面上絕大多數輕薄型筆記型電腦的充電需求。\n\n*   **實測數據：** 搭配 PD 快充線，能在 30 分鐘內為主流旗艦手機充飽約 60% 電力。\n*   **智能分流：** 內建智能晶片，當多設備同時充電時，自動分配最佳功率，確保每一台設備都能高效充電。\n\n### ✨ 賣點二：GaN 3.0 黑科技，體積革命性縮小 (Compact Design)\n\nVolta 採用領先業界的「第三代氮化鎵（GaN）」技術。這種新材料取代了傳統矽晶片，讓高功率充電器不再是笨重的代名詞。\n\n*   **小巧如魔術方塊：** 尺寸僅 **5.2 x 5.2 x 2.9 cm**，輕鬆放入口袋或背包側邊。\n*   **輕盈無感：** 重量僅 **112g**，比一顆雞蛋重一點點，讓您的旅行裝備瞬間輕量化。\n\n### 🔌 賣點三：一抵三，告別插座爭奪戰 (3-Port Convenience)\n\n出差或旅遊時，飯店的插座總是不夠用？Volta 讓您只需一個插座，就能同時搞定所有設備。\n\n*   **雙 Type-C 孔：** 滿足筆電、平板等高功率設備需求。\n*   **單 USB-A 孔：** 兼容舊款設備，或為手錶、耳機等小電流設備充電。\n*   **完美取代：** 一顆 Volta 65W，取代您原有的筆電充電器、手機快充頭、以及耳機充電頭。\n\n### 🛡️ 賣點四：多重安全防護，充電更安心 (Safety First)\n\nVolta 不僅追求速度，更重視您的設備安全。內建多重保護機制，讓您安心享受極速充電。\n\n*   **五重智能保護：** 過壓保護、過流保護、過熱保護、短路保護、低溫充電。\n*   **穩定輸出：** 智慧溫控技術，確保長時間高功率輸出時，充電器仍保持穩定低溫。\n\n---\n\n## 📋 產品規格表 (Specifications)\n\n| 項目 | 規格細節 |\n| :--- | :--- |\n| **商品名稱** | Volta 65W 氮化鎵三孔快充充電器 |\n| **品牌** | Volta |\n| **型號** | [請自行填寫型號，若無則省略] |\n| **技術** | 第三代氮化鎵 GaN |\n| **總輸出功率** | 65W Max |\n| **輸入電壓** | AC 100-240V, 50/60Hz |\n| **輸出接口** | USB-C × 2 + USB-A × 1 |\n| **單孔最大輸出** | USB-C1/C2：65W (PD/PPS) |\n| **尺寸** | 5.2 × 5.2 × 2.9 cm |\n| **重量** | 112g |\n| **安全保護** | 過壓、過流、過熱保護等 |\n| **售價** | **NT$990** |\n\n---\n\n## 🛒 立即購買，享受極速輕巧生活！\n\n**別再忍受慢速與笨重！**\n\n現在就將 Volta 65W 氮化鎵三孔快充加入購物車，讓您的充電體驗全面升級。\n\n**【NT$990 即可擁有】**\n\n**👉 點擊購買，今天下單，明天開始享受高效充電！**", "input_tokens": 130, "output_tokens": 1325}, "798d6cac5bb47d306dea": {"text": "這是一份針對「低溫烘焙綜合堅果」的電商文案，強調其健康、純淨和頂級的製程。\n\n---\n\n# 溯源焙所 | 堅果界的頂級純粹：低溫慢烘，鎖住原始營養\n\n## 💎 產品標題：【溯源焙所】低溫烘焙綜合堅果 (350g) | 零添加、無調味、非油炸 | 腰果、核桃、杏仁、夏威夷豆\n\n---\n\n## 【前言：您正在尋找真正的「原味」嗎？】\n\n市面上的堅果，是否總帶著過多的鹽分、糖分或油耗味？\n\n**溯源焙所** 堅持從源頭把關，採用最純粹的「低溫慢烘」工藝，為您呈現堅果最原始、最飽滿的風味與營養。這不只是一罐零食，更是您追求健康生活、實踐潔淨飲食的最佳夥伴。\n\nNT$590 的價值，來自於我們對製程的堅持與頂級原料的選用。\n\n---\n\n## ✨ 核心特色：三大堅持，成就頂級純粹\n\n### 1. 【低溫慢烘】營養保留的黃金法則\n\n我們拒絕高溫油炸，採用費時耗工的「低溫烘焙」製程。\n\n*   **非油炸：** 杜絕多餘油脂負擔，口感清爽不燥熱。\n*   **鎖住營養：** 溫和的烘烤溫度，能最大程度保留堅果中珍貴的 Omega-3、維生素 E 及膳食纖維，避免營養流失。\n*   **風味提升：** 慢工出細活，讓堅果自然釋放香氣，口感更加酥脆飽滿。\n\n### 2. 【極致純淨】真正的零添加承諾\n\n專為追求健康、生酮飲食、或需嚴格控制鈉攝取量的您設計。\n\n*   **完全無調味：** 零鹽、零糖、零化學添加物。\n*   **無負擔：** 讓您品嚐到堅果本身的甘甜與香氣，不再被多餘調味干擾。\n\n### 3. 【頂級四重奏】嚴選高價值堅果組合\n\n我們精選四種人氣堅果，提供豐富的口感層次與全面的營養補給。\n\n| 堅果種類 | 特色風味 | 營養價值 |\n| :--- | :--- | :--- |\n| **夏威夷豆** | 奶油般滑順，口感細膩 | 單元不飽和脂肪酸含量高，堅果界貴族 |\n| **核桃** | 濃郁香氣，微澀回甘 | 豐富 Omega-3，有助於思緒清晰 |\n| **腰果** | 自然清甜，口感軟糯 | 礦物質豐富，提供優質能量 |\n| **杏仁 (扁桃仁)** | 酥脆爽口，香氣濃郁 | 膳食纖維高，促進腸道健康 |\n\n---\n\n## 🥜 怎麼吃？隨時隨地的健康補給\n\n*   **辦公室能量站：** 下午茶時間，取代高熱量零食，迅速補充體力。\n*   **運動前後：** 快速提供優質脂肪與蛋白質，是健身人士的最佳選擇。\n*   **早餐升級：** 搭配優格、燕麥片或沙拉，增加飽足感與口感。\n*   **烘焙料理：** 搗碎後加入麵包、蛋糕或製作堅果醬，提升風味。\n\n---\n\n## 【商品規格與安心保證】\n\n| 項目 | 內容說明 |\n| :--- | :--- |\n| **品牌** | 溯源焙所 |\n| **商品名稱** | 低溫烘焙綜合堅果 (無調味) |\n| **售價** | **NT$590** |\n| **淨含量** | 350g / 罐 (密封保鮮) |\n| **內容物** | 腰果、核桃、杏仁、夏威夷豆 |\n| **製程** | 低溫烘焙、非油炸 |\n| **調味** | 完全無調味、無添加糖鹽油 |\n\n---\n\n## ⚠️ 【重要提醒：保鮮建議】\n\n由於本產品完全無添加防腐劑，為確保您享用到最佳風味：\n\n**請務必注意：** 開封後請立即**密封冷藏**，並建議於**兩週內**食用完畢，以保持堅果的酥脆度與新鮮度。\n\n---\n\n### 🛒 立即下單，感受來自大自然的純淨滋味！\n\n**溯源焙所** 邀請您，從今天開始，用最健康的方式，享受堅果的原始美好。\n\n**👉 點擊購買，開啟您的潔淨飲食之旅！**", "input_tokens": 134, "output_tokens": 1108}, "ae77193e3c09020edf81": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及柑橘、莓果或花香等具體風味描述？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將「淺焙」與明亮、水果調性等風味特色連結？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否點出「耶加雪菲」為知名的精品咖啡產區？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最好喝」、「極致享受」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以描述風味為主，而非強調提神等功能性效果？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否直接點出商品形式為「掛耳包」，讓消費者能立即理解？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「我們精心挑選」、「為您帶來」等冗餘的開頭？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1007, "output_tokens": 421}, "64a0b3bfb3b09e1d2745": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及 30mg 葉黃素為衛福部建議的每日最高攝取量？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出產品適合長時間使用電腦或手機的族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「游離型」葉黃素的優點，連結到分子小或好吸收？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「立即見效」、「保證有效」等過度承諾的詞語？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全未使用驚嘆號「！」來製造強烈或促銷的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否點出「60粒」的包裝為兩個月份量？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否在標題之外，將「金盞花萃取葉黃素」簡稱為「葉黃素」以避免內文冗長？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 999, "output_tokens": 443}, "5b1a844513f43c428709": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「42dB 降噪」連結到「通勤或辦公時的專注情境」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「雙裝置切換」功能，並點出其在電腦與手機間無縫切換的便利性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「單耳 4.8g」的重量，並連結到「長時間配戴的舒適感」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「極致」、「完美」、「無與倫比」等主觀性強烈的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全不使用表情符號（emoji）或「～」、「！」等情緒性標點符號？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「IPX4」這個規格，直接轉譯為「防汗、防潑水」等消費者能理解的日常用語？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案中關於「降噪」的描述，是否僅限於「有效降低環境噪音」，而未誇大為「完全隔絕所有聲音」？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 998, "output_tokens": 484}, "03020a6d9faa993ba992": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「三孔」能同時為筆電、手機、耳機充電？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「65W」能為 MacBook 等筆記型電腦快充？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否強調「氮化鎵」技術帶來的小體積或輕巧易攜特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「地表最強」、「無與倫比」等主觀誇飾形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以規格（如 65W、三孔）作為溝通重點，而非情感訴求？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用連續的驚嘆號（!!）或問號（??）等強烈情緒標點？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用「的」、「了」、「呢」等非必要的語助詞或贅字？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1001, "output_tokens": 442}, "093fb461176f9c093804": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將 18000Pa 的吸力連結到能有效清除「寵物毛髮」或「灰塵碎屑」？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否將 1.4kg 的重量連結到「輕巧」或「單手操作」等易用性優點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及「床墊吸頭」能用於清潔塵蟎或寢具？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「極致」、「完美」、「奇蹟般」等主觀且誇大的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以規格或功能作為溝通重點，而非營造居家情境故事？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否完全由完整的句子組成，沒有僅由名詞或規格堆砌而成的片段？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免在不同欄位（例如標題和賣點）中，使用完全相同的文句來溝通同一個賣點？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 993, "output_tokens": 463}, "65d4c20bb44eb6f94e45": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及茶湯帶有蜜香或熟果風味？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出茶葉產自「台東鹿野」這個知名產區？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否說明「三角立體茶包」能讓茶葉充分舒展？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「極品」、「頂級」等過於誇大的宣傳詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否提及「重發酵」工藝所創造的醇厚尾韻？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案的賣點描述中，是否避免重複標題已有的「紅烏龍」或「茶包」等詞語？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否點出此茶包適合送禮，或適合犒賞自己？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1010, "output_tokens": 426}, "831a41191344a9abf690": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及「低溫烘焙」能保留堅果的完整營養與原始風味？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「無調味」適合追求原型食物或健身的族群？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提到一罐就能攝取到腰果、核桃、杏仁、夏威夷豆的多樣營養？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最好吃」、「地表最強」等過度誇大的形容詞？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以描述風味或口感的方式（如：香甜、酥脆）來展現職人感，而非僅列出成分？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用「優質脂肪酸」、「抗氧化物」等需要額外解釋的營養學術語？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否直接陳述賣點，避免使用「我們精心挑選了...」、「這是一款...」等冗餘的開頭？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1007, "output_tokens": 460}, "d62311eb2efcb476ea5b": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「積雪草」能舒緩肌膚不適？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「神經醯胺」或「泛醇」有助於修護肌膚屏障？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否描述產品質地為清爽不黏膩的水感質地？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「一秒」、「立即見效」等誇大不實的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否以溫和、不具攻擊性的語氣描述肌膚問題（例如，使用「舒緩」而非「對抗」）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否將「5% 積雪草萃取」這個規格，轉化為對消費者有感的具體好處（例如：高濃度舒緩）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否避免使用表情符號 (emoji) 或特殊符號（如：★、♥），以維持專業理性的形象？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 998, "output_tokens": 464}, "b553262bfaaa68e575bf": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否提及五株專利益生菌能幫助調整體質？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否點出「隨身包」設計方便外食族或上班族攜帶補充？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「100億」的菌數連結到「每日足量補充」的概念？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「立即見效」等誇大或過度承諾的詞語？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否傳達出「每日規律補充」是維持健康的關鍵，而非強調單次速效？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否避免使用驚嘆號，以維持專業、溫和的語氣？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否用一句話就清楚說明產品如何幫助目標受眾（例如：幫助外食族補充好菌）？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    }\n  ]\n}", "input_tokens": 1006, "output_tokens": 439}, "fb4c9c667a666b3403c2": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「五重玻尿酸」能為乾燥肌膚帶來深層補水的好處？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「天絲膜布」材質服貼、親膚的特性？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否明確指出產品適合「敏弱肌」使用？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「最強」、「終極」、「奇蹟」等誇大不實的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否完全不使用驚嘆號「！」，以維持理性溫柔的語氣？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否直接說明產品的核心功能是「保濕」，而非使用「煥活」、「調理」等模糊說法？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否以短句為主，避免出現超過 20 個字的長句？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 999, "output_tokens": 431}, "79fa90ad6f5303385933": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否點出「pH5.5 弱酸性」有助於維持肌膚健康屏障？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否提及「洋甘菊萃取」能舒緩肌膚不適？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否將「胺基酸」的清潔力連結到「洗後不緊繃乾澀」的感受？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否完全避免使用「一秒」、「立即見效」等誇大不實的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否避免使用「你必須」、「務必」等命令式或過於強烈的字眼？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否明確點出此產品適合「敏弱肌」或「敏感肌」族群使用？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否以「綿密泡沫」來描述產品質地，而非僅使用抽象形容詞？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 997, "output_tokens": 445}, "2de7c6d4fdbf3ffd26cc": {"text": "{\n  \"rubrics\": [\n    {\n      \"id\": \"R1\",\n      \"criterion\": \"文案是否將「小分子」或「胜肽」的特性，轉化為「好吸收」的消費者利益點？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R2\",\n      \"criterion\": \"文案是否強調膠原蛋白含量為 2500mg，並將其作為一個關鍵優勢？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R3\",\n      \"criterion\": \"文案是否提及添加維生素C能幫助膠原蛋白合成？\",\n      \"dimension\": \"賣點覆蓋\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R4\",\n      \"criterion\": \"文案是否避免使用「立即見效」、「奇蹟」、「保證」等過度誇大的詞彙？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": true\n    },\n    {\n      \"id\": \"R5\",\n      \"criterion\": \"文案是否使用溫和、建議的語氣（如「幫助維持」、「日常補充」），而非命令或強迫的語氣（如「你必須擁有」）？\",\n      \"dimension\": \"品牌語調\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R6\",\n      \"criterion\": \"文案是否點出蔓越莓風味能提升口感或覆蓋腥味，讓補充更容易持續？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    },\n    {\n      \"id\": \"R7\",\n      \"criterion\": \"文案是否清楚說明「每日1包」或「睡前食用」等具體的食用建議？\",\n      \"dimension\": \"消費者可讀性\",\n      \"critical\": false\n    }\n  ]\n}", "input_tokens": 1019, "output_tokens": 459}, "0f07f1e09b4df45bc3cf": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"分子小好吸收\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"足量 2500mg 小分子膠原蛋白胜肽\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及維生素C及其功效\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"為美麗打下穩固基礎\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"方便每日持續補充\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"清爽酸甜蔓越莓風味，順口無腥味\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"每日一包\"\n    }\n  ]\n}", "input_tokens": 841, "output_tokens": 271}, "c271c2def9acf5537e0e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"缺少「衛福部建議的每日最高攝取量」的說明\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間使用 3C 螢幕的上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用黃金比例 5:1 游離型，分子小好吸收。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"是您晶亮舒適的日常保養選擇。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"每日一粒，一盒 60粒，輕鬆持續補充，維持舒適感。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"缺少「兩個月份量」的說明，僅提及「每日一粒，一盒 60粒」\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"晨光研選葉黃素膠囊，專為長時間使用電腦的上班族設計。\"\n    }\n  ]\n}", "input_tokens": 862, "output_tokens": 331}, "5c886a791d13872c4776": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"幫助肌膚安撫不適感\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「幫助維持肌膚健康」，未明確說明「修護肌膚屏障」。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"水感輕盈精華，清爽好吸收不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「一秒」、「立即見效」等誇大不實的詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"幫助肌膚安撫不適感\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度 5% 積雪草萃取，溫和穩定膚況。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用表情符號 (emoji) 或特殊符號。\"\n    }\n  ]\n}", "input_tokens": 874, "output_tokens": 319}, "72954f683f535326ebd3": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"5%高濃度積雪草，高效舒緩泛紅不適。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"協同神經醯胺，穩定修護肌膚屏障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"清爽水感精華，快速吸收不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「一秒」、「立即見效」等誇大詞彙，整體語氣為溫和陳述，如「溫和高效的修護力」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"高效舒緩泛紅不適。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"5%高濃度積雪草，高效舒緩泛紅不適。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用表情符號 (emoji) 或特殊裝飾性符號（如：★、♥）。\"\n    }\n  ]\n}", "input_tokens": 899, "output_tokens": 339}, "828e6405c765b118971b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"有效舒緩肌膚泛紅、乾燥引起的刺激與不適。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"2% 泛醇 (Panthenol / 維他命原B5)...加速修復受損屏障。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"輕盈水感：告別黏膩厚重！精華液呈現清爽的水感質地，一抹即化\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「秒速吸收」，屬於誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"你需要的不只是保濕，更是高效的「修護與舒緩」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"高濃度舒緩：5% 積雪草萃取\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案在多個段落標題前使用了表情符號與特殊符號（如：🌟、🌿、✨）。\"\n    }\n  ]\n}", "input_tokens": 1440, "output_tokens": 348}, "4bc6c6165ed53be188d0": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"小分子胜肽好吸收，日常保養更輕鬆\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"每包含足量深海魚膠原蛋白胜肽 2500mg\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及添加維生素C或其相關功效\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「立即見效」、「奇蹟」、「保證」等誇大詞彙，整體語氣溫和（例如：「每日補充美麗基底」、「維持青春光采」）\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"方便每日持續補充\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"清新酸甜蔓越莓風味，順口無腥味\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"睡前沖泡一杯即可\"\n    }\n  ]\n}", "input_tokens": 842, "output_tokens": 320}, "11873d8575d510f36d3b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"積雪草舒緩精華液\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"添加泛醇與神經醯胺，提升肌膚防護力。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"水感輕盈精華質地，快速吸收不黏膩。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和穩定肌膚狀態\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"溫和穩定肌膚狀態\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"5%高純度積雪草萃取，溫和穩定肌膚狀態。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計的青研積雪草舒緩精華液 (30ml)。\"\n    }\n  ]\n}", "input_tokens": 864, "output_tokens": 303}, "8df26088195c36c6854c": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"本產品完美貼合肌膚的天然酸鹼值，有助於：鞏固屏障：維持肌膚天然保護膜（酸性保護膜）。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"添加珍貴的洋甘菊萃取，以其卓越的舒緩與鎮靜功效聞名...功效：舒緩環境壓力造成的肌膚不適，提升肌膚穩定度。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"我們採用親膚性極佳的胺基酸界面活性劑作為主要清潔成分...效果：深層清潔，洗後肌膚不緊繃、不乾澀。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"我們相信，好的清潔是保養的第一步，必須在帶走髒污的同時，留下肌膚最需要的滋潤與平衡。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"專為追求溫和高效清潔的您設計。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"主標題：洗出水潤光澤！弱酸性pH5.5，敏感肌也愛上的胺基酸潔顏體驗\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕輕加水搓揉，立即產生豐盈、細緻、有彈性的綿密泡沫。\"\n    }\n  ]\n}", "input_tokens": 1489, "output_tokens": 415}, "e157a79c0fe57e300766": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"弱酸性 pH5.5 貼近肌膚酸鹼值，維持健康屏障。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"添加洋甘菊萃取，幫助舒緩肌膚不適感。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"胺基酸配方溫和清潔，不帶走肌膚必要水分。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「一秒」、「立即見效」等誇大詞彙，例如僅用「幫助舒緩肌膚不適感」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用「你必須」、「務必」等命令式字眼，例如「專為敏弱肌設計的胺基酸洗面乳」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計的胺基酸洗面乳\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"泡沫綿密細緻\"\n    }\n  ]\n}", "input_tokens": 839, "output_tokens": 350}, "5cf0d4f4c3b993004ae7": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"42dB 旗艦級主動降噪，隔絕通勤喧囂，瞬間進入專注模式。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「雙裝置切換」，但未點出是在「電腦與手機」之間切換的便利性。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"單耳僅 4.8g 輕量設計...全天配戴舒適無負擔。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案在賣點中使用了「極致輕盈」。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全不含表情符號或「～」、「！」等情緒性標點符號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「IPX4 防水」，未轉譯為「防汗、防潑水」等更具體的日常用語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"有效阻擋環境噪音，讓您隨時保持專注。\"\n    }\n  ]\n}", "input_tokens": 1014, "output_tokens": 358}, "e7b15641b196289f8ef2": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用五重玻尿酸配方，層層補水鎖水。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"輕薄服貼的天絲蠶絲膜布，親膚透氣。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計的青研多重玻尿酸保濕面膜\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和舒緩乾燥不適。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"採用五重玻尿酸配方，層層補水鎖水。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"多重玻尿酸保濕面膜\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案包含超過 20 個字的長句，例如「專為敏弱肌設計的青研多重玻尿酸保濕面膜，內含 5片。」\"\n    }\n  ]\n}", "input_tokens": 842, "output_tokens": 330}, "8fd1aec201156cab919d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"柑橘莓果花香風味\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"精選淺焙工藝，釋放飽滿柑橘與莓果香。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"衣索比亞 耶加雪菲，經典水果調性。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"專為冷萃設計，輕鬆享受職人級冰咖啡。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"完整保留柑橘、莓果與優雅花香。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"淺焙冷萃掛耳咖啡\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"衣索比亞 耶加雪菲，經典水果調性。\"\n    }\n  ]\n}", "input_tokens": 830, "output_tokens": 299}, "8ab108a93c54e3ccf78b": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"五重玻尿酸精華，深層注水瞬效補水。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用頂級天絲膜布，極致親膚不刺激。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"溫和無添加配方，敏弱肌也能安心敷。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「最強」、「終極」、「奇蹟」等字眼，整體語氣溫和，例如「溫和無添加配方，敏弱肌也能安心敷。」\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"SEO 描述中出現驚嘆號：「專為敏弱肌與成分黨設計！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案直接使用「保濕」、「補水」等詞彙，例如標題的「保濕面膜」與賣點的「瞬效補水」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"SEO 描述中出現長句，例如「青研 CHINGYEN 多重玻尿酸保濕面膜，採用五重玻尿酸精華，提供肌膚長效且深層的保濕。」（38字）\"\n    }\n  ]\n}", "input_tokens": 862, "output_tokens": 400}, "74486b328abe73553b60": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"持續補充，幫助您調整體質\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"粉末隨身包設計，方便攜帶食用。適合外食族及作息不規律者，輕鬆補充好菌。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每包足量 100億 活菌，提供日常所需補給。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"持續補充，幫助您調整體質，維持消化道健康\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"持續補充，幫助您調整體質，維持消化道健康\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"持續補充，幫助您調整體質，維持消化道健康，讓生活更順暢無負擔。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"適合外食族及作息不規律者，輕鬆補充好菌。\"\n    }\n  ]\n}", "input_tokens": 858, "output_tokens": 333}, "42da3fcc381a730d24cb": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及 30mg 是衛福部建議的每日最高攝取量。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間螢幕族設計，維持清晰舒適。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"游離型好吸收，黃金 5:1 晶亮比例。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"維持清晰舒適\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案在標題「晶亮有神！」與 SEO 描述「上班族必備眼部保健！」中使用了驚嘆號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案未說明「60粒」是「兩個月份量」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"高濃度 30mg 葉黃素，足量守護。\"\n    }\n  ]\n}", "input_tokens": 879, "output_tokens": 323}, "3b0e21f7ed24be88bf79": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"柑橘莓果花香調\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"淺焙保留細緻果酸\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「耶加雪菲產區」與「精品咖啡」，但未明確點出「耶加雪菲」本身是知名的精品咖啡產區。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"職人級精品咖啡，隨時享受冰鎮純粹。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"綻放柑橘、莓果與優雅花香的層次風味。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"冷萃掛耳咖啡\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"精選衣索比亞耶加雪菲\"\n    }\n  ]\n}", "input_tokens": 838, "output_tokens": 308}, "31aa87a7b078b0d6178e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"最高 42dB 主動降噪，有效阻隔通勤雜音。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「雙裝置切換」，但未點出在電腦與手機間切換的便利性。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"單耳僅 4.8g 輕量設計...輕巧設計，提供舒適的聆聽體驗。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用的形容詞為「深度」、「有效」、「穩定」等，未出現「極致」、「完美」等主觀性強烈的詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案完全未使用表情符號或情緒性標點符號。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"僅提及「IPX4 等級防水」，未轉譯為「防汗、防潑水」等消費者能理解的日常用語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"最高 42dB 主動降噪，有效阻隔通勤雜音。\"\n    }\n  ]\n}", "input_tokens": 909, "output_tokens": 367}, "7a3d7bdf36167d31a163": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「足量30mg」，但未說明此為衛福部建議的每日最高攝取量。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為長時間使用螢幕的上班族設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"游離型態好吸收。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"為您的日常晶亮保養打底\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案通篇未使用驚嘆號，例如「持續補充兩個月。」一句僅以句號結尾。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"60粒裝，每日一粒，持續補充兩個月。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案從未使用「金盞花萃取葉黃素」的全名，故無從簡稱。\"\n    }\n  ]\n}", "input_tokens": 864, "output_tokens": 323}, "1cbf71ada17bde19c49f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"職人低溫慢焙，保留堅果原始營養與風味。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"健身族群與注重原型食物家庭的優質點心選擇。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案僅列出含有四種堅果，但未強調「一罐就能攝取多樣營養」這個優點。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「最佳選擇」，屬於過度誇大的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及保留「原始風味」與「天然香氣」，但未具體描述風味或口感（如：香甜、酥脆）。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「優質脂肪」與「膳食纖維」，屬於需要額外解釋的營養學術語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案直接以「職人低溫慢焙」、「嚴選」、「堅持」等動詞開頭，或直接描述產品，沒有冗餘的開頭。\"\n    }\n  ]\n}", "input_tokens": 844, "output_tokens": 377}, "0f84f850ef9066af549c": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"低溫烘焙，保留堅果原始香氣與營養。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"完全無調味...是注重原型食物的家庭與健身族群...的最佳選擇。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案僅列出四種堅果（腰果、核桃、杏仁、夏威夷豆），但未提及這樣的組合能攝取到「多樣」的營養。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體未使用「最好吃」等誇大形容詞，語調務實。例如：「口感清脆，風味純粹。」\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"口感清脆，風味純粹。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「優質油脂」，屬於需要額外解釋的營養學術語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"賣點採直接陳述，例如：「低溫烘焙，保留堅果原始香氣與營養。」\"\n    }\n  ]\n}", "input_tokens": 894, "output_tokens": 363}, "435dd0554bff01565473": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"獨特蜜香與熟果韻味，尾韻自然回甘。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野產區，重發酵工藝，風味醇厚。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「快速釋放茶葉完整香氣」，但未說明三角立體茶包能讓「茶葉充分舒展」。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「風味醇厚」、「自然回甘」等詞彙描述，未見「極品」、「頂級」等誇大宣傳詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"重發酵工藝，風味醇厚。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"賣點 3「三角立體茶包」重複了標題已有的「茶包」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"送禮自用皆宜\"\n    }\n  ]\n}", "input_tokens": 806, "output_tokens": 345}, "2b1d56fe1a53d8f911f1": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"淺焙保留柑橘莓果\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"淺焙水果花香調\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及耶加雪菲為產地，但未說明其為知名的精品咖啡產區。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「品味極致」、「最佳選擇」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"淺焙保留柑橘莓果，風味清新不酸澀。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"冷萃掛耳咖啡\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案使用「嚴選」，屬於冗餘的套話。\"\n    }\n  ]\n}", "input_tokens": 803, "output_tokens": 299}, "21b7dda6ae4bd81ec0d6": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"獨特蜜香與熟果韻味，尾韻自然回甘。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野產區，重發酵工藝，風味醇厚。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"三角立體茶包設計，讓茶葉完整舒展。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野產區，重發酵工藝，風味醇厚。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野產區，重發酵工藝，風味醇厚。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"賣點「立體三角茶包，完整釋放茶葉精華。」中重複了「茶包」一詞。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"台灣茶愛好者日常品飲或送禮首選。\"\n    }\n  ]\n}", "input_tokens": 817, "output_tokens": 330}, "8b39936ffcded28d33a1": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"低溫慢焙，非油炸製程，完整保留堅果原始營養。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"【溯源焙所】原型食物首選！低溫烘焙綜合堅果 350g。純淨無調味，家庭健身高纖健康零食。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案列出了四種堅果（腰果、核桃、杏仁、夏威夷豆），但未提及「一罐就能攝取多樣營養」的概念。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「首選」、「必備」等詞，但避免了「最好吃」、「地表最強」等無法驗證的誇大形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「天然風味」，但未描述實際的風味或口感（如：香甜、酥脆）。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「優質油脂」，屬於需要額外解釋的營養學術語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"低溫慢焙，非油炸製程，完整保留堅果原始營養。\"\n    }\n  ]\n}", "input_tokens": 892, "output_tokens": 400}, "53646c27c842557898d4": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"每日持續補充，溫和調整體質\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"粉末隨身包，方便上班族隨時補充\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每包足量 100億 CFU\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和調整體質\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"每日一包，持續補充好菌\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"幫助消化道維持平衡\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"適合外食族群，為忙碌生活建立體內平衡\"\n    }\n  ]\n}", "input_tokens": 877, "output_tokens": 278}, "f5a5d256f114f4ec03db": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"溫潤蜜香與熟果尾韻，醇厚回甘\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野產區，職人級重發酵紅烏龍\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"三角立體茶包設計，讓茶葉完整舒展。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「極品」、「頂級」等誇大詞彙，而是以「職人級」、「精選」等詞描述。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案提及了「重發酵」和「醇厚尾韻」，但未明確說明醇厚尾韻是由重發酵工藝所創造的。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"賣點 1「職人級重發酵紅烏龍」重複了標題的「紅烏龍」；賣點 3「3g三角立體茶包」重複了標題的「茶包」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"送禮自用皆宜\"\n    }\n  ]\n}", "input_tokens": 803, "output_tokens": 374}, "af0490b87181274ba128": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"42dB深度主動降噪，隔絕通勤與辦公室喧囂。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「雙裝置無縫切換」，但未點出是在「電腦與手機」之間切換的情境。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"單耳僅4.8g輕量設計、輕巧舒適\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中使用了「最佳」這個主觀性強烈的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"42dB深度主動降噪，隔絕通勤與辦公室喧囂。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"IPX4防水防汗\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"隔絕通勤與辦公室喧囂\"\n    }\n  ]\n}", "input_tokens": 926, "output_tokens": 321}, "15eafcd1b06992bde17d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"pH5.5 弱酸平衡，維持健康膚況。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"內含高純度胺基酸界面活性劑與洋甘菊萃取，有效舒緩敏弱肌膚。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"高濃度胺基酸，洗後水潤不緊繃。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"專為敏弱肌設計，溫和不刺激。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案使用「必收」一詞，語氣較為強烈。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌專用\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"綿密泡沫質地\"\n    }\n  ]\n}", "input_tokens": 867, "output_tokens": 303}, "bac5fb582e785ae5c690": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"pH5.5弱酸配方，洗淨同時呵護皮脂膜健康。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案僅提及「洗後膚感舒適」，未明確點出「洋甘菊萃取」有「舒緩」肌膚不適的功效。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用溫和胺基酸界面活性劑，維持肌膚水潤。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有使用「一秒」、「立即見效」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案中沒有使用「你必須」、「務必」等命令式字眼。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌與成分黨適用\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"豐盈綿密泡沫\"\n    }\n  ]\n}", "input_tokens": 850, "output_tokens": 328}, "d82210f59fe75af31bb0": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及可為「筆電、平板、手機」充電，但未明確點出「耳機」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"65W大功率輸出，筆電手機同時快充。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用第三代氮化鎵，體積輕巧易攜帶。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案整體用語客觀，例如「65W大功率輸出」、「僅重112g」，未出現主觀誇飾詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta 65W 氮化鎵三孔快充頭 雙C一A配置 輕巧體積 差旅首選\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"全篇文案僅使用句號、頓號等標準標點，未出現連續驚嘆號或問號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"輕巧體積 差旅首選\"\n    }\n  ]\n}", "input_tokens": 852, "output_tokens": 361}, "3de9872732ee210ae64c": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa 颶風吸力，深層清除毛髮塵蟎。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"僅 1.4kg，單手操作不費力\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「床墊吸頭」和「清除塵蟎」，但未將兩者連結，沒有說明床墊吸頭的用途。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「極致」、「完美」、「奇蹟般」等主觀誇大用詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案溝通重點均為規格與功能，如「18000Pa 颶風吸力」、「僅 1.4kg，單手操作不費力」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"商品標題包含僅由規格和名詞堆砌而成的片段，例如「1.4kg超輕量、18000Pa強吸力」。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案在不同欄位（標題、賣點、SEO描述）中，皆使用不同的文句來溝通同一個賣點，沒有完全重複的句子。\"\n    }\n  ]\n}", "input_tokens": 882, "output_tokens": 408}, "7392898aeeba33007564": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": false,\n      \"evidence\": \"文案提及可同時為「筆電、手機、平板」充電，但未提及「耳機」。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"65W 總輸出，滿足筆電快充需求。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"第三代氮化鎵技術，體積輕巧。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案以規格和功能描述為主，例如「多重安全防護，穩定可靠輸出」，未出現主觀誇飾詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"2C1A三孔配置，同時供電三裝置。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"Volta 65W 氮化鎵三孔快充，專為出差族設計。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"第三代氮化鎵技術，體積輕巧。\"\n    }\n  ]\n}", "input_tokens": 810, "output_tokens": 335}, "af21ac01f8435369d67f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"蜜香熟果韻\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"台東鹿野重發酵\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用食品級三角立體茶包設計，讓茶葉有足夠空間完整舒展，釋放出最完整的香氣與滋味\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中多次使用「頂級」、「極致」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「重發酵」工藝，但未明確說明是此工藝創造了「醇厚尾韻」。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"賣點描述中重複了標題已有的「紅烏龍」和「茶包」等詞。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"送禮自用兩相宜：精緻盒裝，體面大方。\"\n    }\n  ]\n}", "input_tokens": 1696, "output_tokens": 340}, "9098e95ad0f8962fe27d": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"五重玻尿酸，層層滲透，緩解乾燥缺水。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"採用天絲蠶絲膜布，輕薄服貼\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"敏弱肌適用\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"溫和配方，敏弱肌適用。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": false,\n      \"evidence\": \"文案於 SEO 描述中使用了驚嘆號「！」\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"五重玻尿酸保濕面膜\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"所有句子與子句皆未超過 20 字，例如：「輕盈的天絲蠶絲膜布，服貼舒適，精華液好吸收不黏膩。」\"\n    }\n  ]\n}", "input_tokens": 906, "output_tokens": 314}, "8d273e9343cda4332cbf": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"五株專利菌，打造黃金體質基底。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為外食族與作息不規律上班族設計...方便攜帶，隨時補充好菌\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"每包足量 100億 專利益生菌\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用「順暢有感」、「有效調整體質」等詞，避免了「最強」、「立即見效」等誇大或過度承諾的詞語。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"隨時補充好菌，維持順暢\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案標題中使用了驚嘆號：「外食族/夜貓族必備！」\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"緩解外食油膩負擔，幫助消化道機能更順暢。\"\n    }\n  ]\n}", "input_tokens": 897, "output_tokens": 348}, "cb507dd3093e60aaa3ce": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"淺焙保留最純淨的柑橘、莓果與花香。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"烘焙度：淺焙 (最大化保留產區水果與花卉香氣)\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"產區：衣索比亞 耶加雪菲 (精品咖啡的代名詞)\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「極致冷萃體驗」、「堅持給您最好的」等誇大詞彙。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"這款咖啡選用來自咖啡發源地——衣索比亞的頂級耶加雪菲豆，以精準的淺焙工藝處理，確保每一口都能品嚐到產區最純粹的風味。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"冷萃掛耳咖啡\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「是我們為追求輕盈、潔淨風味的您，精心打造的夏日咖啡解決方案」等冗餘用語。\"\n    }\n  ]\n}", "input_tokens": 2021, "output_tokens": 372}, "4124f0f156391fa4fcba": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"「2C1A三孔設計，同時充飽三裝置」、「讓您一機搞定筆電、平板與手機的充電需求」\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"65W大功率輸出，筆電手機極速回血。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"採用第三代氮化鎵，體積輕巧不佔空間。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「地表最強」、「無與倫比」等誇飾詞，而是以「出差族輕量化首選」等基於產品特色的推薦詞溝通。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以「65W大功率輸出」、「2C1A三孔設計」、「僅112g超輕量」等規格作為溝通重點。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案僅在標題使用單一驚嘆號，內文皆使用句號，無連續或濫用之情形。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案簡潔，無「了」、「呢」等非必要語助詞。例如：「65W大功率輸出，筆電手機極速回血。」\"\n    }\n  ]\n}", "input_tokens": 916, "output_tokens": 409}, "69e728fb50801e65bdbd": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"它搭載高效能馬達，提供 18000Pa 的瞬吸力。從地板毛髮、地毯深層灰塵到零食碎屑，都能一網打盡，確保您的居家環境徹底潔淨。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"Volta Lite 將主機重量控制在驚人的 1.4kg，比四瓶礦泉水還輕。無論是舉高清潔天花板蜘蛛網，或是單手操作清理車內空間，都能輕鬆搞定，讓您告別手痠疲勞。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【配件名稱】床墊吸頭【適用場景】床鋪、沙發、布藝家具【清潔效益】專門對付塵螨、皮屑與過敏原，提升睡眠品質。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案未使用「極致」、「完美」等誇大詞彙，而是以「驚人的 1.4kg，比四瓶礦泉水還輕」等較具體的方式描述產品優點。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以「輕量化設計」、「強勁吸力」、「續航力升級」等功能為溝通重點，並附有「商品規格一覽」表格，而非情境故事。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案主體由完整句子組成，例如：「我們知道，打掃不該是體力活。Volta Lite 將主機重量控制在驚人的 1.4kg，比四瓶礦泉水還輕。」\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"文案在不同欄位溝通同一賣點時會變換說法，例如標題的「輕盈解放」與內文的「輕量化設計」、「主機重量控制在驚人的 1.4kg」，並未重複使用相同文句。\"\n    }\n  ]\n}", "input_tokens": 1527, "output_tokens": 559}, "aa3b5987dbba79ac2a57": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa強勁吸力，深層清潔毛髮與微塵，適用硬地板與地毯。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"機身僅1.4kg，單手操作輕巧，高處清潔與收納無負擔。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案未提及「床墊吸頭」、「塵蟎」或「寢具」等相關資訊。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"0.5L集塵盒與濾網可水洗，維護簡單，保持吸塵器高效能。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"標準模式續航達35分鐘，滿足小坪數家庭全屋清潔需求。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"標題「18000Pa強吸力 1.4kg輕量 35分鐘續航 小宅寵物適用」是由規格堆砌而成，並非完整句子。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"具備18000Pa強勁吸力，輕鬆吸除毛髮與粉塵。\"\n    }\n  ]\n}", "input_tokens": 903, "output_tokens": 389}, "908cac573eda7395c400": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"具備18000Pa強勁吸力，輕鬆吸除寵物毛髮。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"僅 1.4kg 主機輕量化，高處清潔不費力。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「床墊吸頭」，但未說明其可用於清潔塵蟎或寢具。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"主機僅1.4kg，輕巧好收納。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"Volta 1.4kg 輕量無線吸塵器 18000Pa強吸力 小家庭/寵物適用 續航35分鐘\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"商品標題是由名詞與規格堆砌而成，並非完整句子。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"18000Pa 颶風吸力，深層清潔毛髮塵蟎。\"\n    }\n  ]\n}", "input_tokens": 855, "output_tokens": 358}, "7c2b95d5c33173bcc7a3": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"採用最純粹的「低溫慢烘」工藝，為您呈現堅果最原始、最飽滿的風味與營養。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"專為追求健康、生酮飲食、或需嚴格控制鈉攝取量的您設計。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"我們精選四種人氣堅果，提供豐富的口感層次與全面的營養補給。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": false,\n      \"evidence\": \"文案使用「頂級純粹」、「堅果界的頂級純粹」等詞語，屬於較為誇大的形容詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"夏威夷豆：奶油般滑順，口感細膩；核桃：濃郁香氣，微澀回甘；腰果：自然清甜，口感軟糯；杏仁：酥脆爽口，香氣濃郁。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「Omega-3」、「單元不飽和脂肪酸」、「優質脂肪」等需要額外解釋的營養學術語。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": false,\n      \"evidence\": \"文案中出現「我們拒絕...」、「我們精選...」、「這不只是一罐...」等冗餘的開頭語句。\"\n    }\n  ]\n}", "input_tokens": 1712, "output_tokens": 424}, "db9130f8668f04fae62e": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"一顆 Volta 65W，取代您原有的筆電充電器、手機快充頭、以及耳機充電頭。\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": true,\n      \"evidence\": \"65W 高功率輸出 ... 輕鬆支援 MacBook、Dell XPS 等主流筆記型電腦快充。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"Volta 採用領先業界的「第三代氮化鎵（GaN）」技術。這種新材料取代了傳統矽晶片，讓高功率充電器不再是笨重的代名詞。\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"文案使用的詞彙如「強勁的 65W 輸出」、「體積極致縮小」，以具體效益取代「地表最強」等主觀誇飾詞。\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"文案以「65W 高功率輸出」、「第三代 GaN 氮化鎵」、「三孔設計 (2C+1A)」等規格作為核心賣點進行溝通。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": true,\n      \"evidence\": \"文案中的標點符號使用節制，例如「告別笨重！小體積，大能量！」僅使用單一驚嘆號。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"單孔 Type-C 可達 65W 輸出。\"\n    }\n  ]\n}", "input_tokens": 1911, "output_tokens": 428}, "1ef4c294e338dd3c1b6f": {"text": "{\n  \"results\": [\n    {\n      \"id\": \"R1\",\n      \"passed\": true,\n      \"evidence\": \"專利五菌株，高定殖率，調整體質，告別卡關人生！\"\n    },\n    {\n      \"id\": \"R2\",\n      \"passed\": false,\n      \"evidence\": \"文案提及「獨立包裝，方便攜帶」，但沒有明確點出這是為了方便「外食族」或「上班族」。\"\n    },\n    {\n      \"id\": \"R3\",\n      \"passed\": true,\n      \"evidence\": \"【高規格】每包足量 100 億 CFU 活菌\"\n    },\n    {\n      \"id\": \"R4\",\n      \"passed\": true,\n      \"evidence\": \"順暢有感，體內環保新選擇\"\n    },\n    {\n      \"id\": \"R5\",\n      \"passed\": true,\n      \"evidence\": \"每日用量：每日 1 包。\"\n    },\n    {\n      \"id\": \"R6\",\n      \"passed\": false,\n      \"evidence\": \"文案中多處使用驚嘆號，例如「告別卡關人生！」、「你的腸道，需要更強大的後盾！」等。\"\n    },\n    {\n      \"id\": \"R7\",\n      \"passed\": true,\n      \"evidence\": \"專利五菌株，高定殖率，調整體質，告別卡關人生！\"\n    }\n  ]\n}", "input_tokens": 1704, "output_tokens": 355}, "38662ec662a619ffe7c1": {"text": "{\n  \"sentiment\": \"neutral\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 342, "output_tokens": 62}, "d958833b4ddf8ce0f392": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 385, "output_tokens": 62}, "ba6dff6da87712e1b0e0": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 379, "output_tokens": 62}, "6c928cfb27796fb69180": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 374, "output_tokens": 56}, "5e9fbd86665495342d71": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"\",\n  \"urgency\": \"low\"\n}", "input_tokens": 367, "output_tokens": 56}, "58440a6086fab12a80e8": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\",\n    \"產品品質\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"考慮推出更大容量的產品規格\",\n  \"urgency\": \"low\"\n}", "input_tokens": 365, "output_tokens": 70}, "2b8e7a0d3338a5a1d16a": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"增加集塵盒容量，並重新設計集塵盒傾倒機制，避免灰塵外噴\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 366, "output_tokens": 84}, "fa917a5e62fb8cc2f406": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"使用體驗\",\n    \"包裝\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"檢查滴管長度與瓶口設計，避免滴管尖端容易沾染瓶口外部。\",\n  \"urgency\": \"low\"\n}", "input_tokens": 388, "output_tokens": 84}, "bc8fafaa509ead9bb45d": {"text": "{\n\"sentiment\": \"negative\",\n\"aspects\": [\n\"包裝\"\n],\n\"is_about_product\": false,\n\"actionable_suggestion\": \"加強出貨時的外部保護包裝，避免商品外盒在運送過程中受損\",\n\"urgency\": \"medium\"\n}", "input_tokens": 377, "output_tokens": 71}, "bac8771b1a183a650142": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"重新檢視面膜布的剪裁設計，特別是眼下區域，以改善服貼度；或調整精華液配方，增加黏稠度以減少滴落。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 373, "output_tokens": 101}, "2277e8e527fd1ad971f4": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"物流配送\",\n    \"客服\"\n  ],\n  \"is_about_product\": false,\n  \"actionable_suggestion\": \"優化出貨流程以縮短配送時間，並要求客服提供明確的延遲原因與預計送達時間，而非僅回覆處理中。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 377, "output_tokens": 95}, "31e3c2063800a86f260a": {"text": "{\n  \"sentiment\": \"positive\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"更新掛耳咖啡沖泡說明，提供明確的水溫建議（例如：88°C）以避免澀感。\",\n  \"urgency\": \"low\"\n}", "input_tokens": 383, "output_tokens": 88}, "5b1af89879928e397480": {"text": "{\n  \"sentiment\": \"negative\",\n  \"aspects\": [\n    \"使用體驗\",\n    \"產品品質\",\n    \"客服\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"在商品頁面或說明書中增加適用膚質的詳細說明或警語，並追蹤是否有其他客戶回報致痘問題。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 367, "output_tokens": 98}, "f83f9cbfb867fa1b36b9": {"text": "{\n\"sentiment\": \"neutral\",\n\"aspects\": [\"產品品質\", \"使用體驗\", \"價格\"],\n\"is_about_product\": true,\n\"actionable_suggestion\": \"評估產品定價策略，或考慮推出大包裝/多盒組合優惠以降低單價感\",\n\"urgency\": \"low\"\n}", "input_tokens": 380, "output_tokens": 76}, "80671142b38a72919fff": {"text": "{\n  \"sentiment\": \"neutral\",\n  \"aspects\": [\n    \"產品品質\",\n    \"使用體驗\"\n  ],\n  \"is_about_product\": true,\n  \"actionable_suggestion\": \"1. 增加耳塞尺寸選項（如XS），改善小耳道用戶的配戴舒適度。2. 檢查通話麥克風的降噪功能，排除通話時對方聽到的雜音問題。\",\n  \"urgency\": \"medium\"\n}", "input_tokens": 372, "output_tokens": 110}}, "token_counts": {"tok:9d37906d3581365ecfaf": 14, "tok:6caa929e1c491a647ff5": 13, "tok:199e5367866f34a4f260": 387, "tok:febcc7fe24883e573941": 24}}
print(f'已載入 {len(FIXTURES["calls"]):,} 筆錄製輸出' + ('　模式：離線重播' if OFFLINE_MODE else '　模式：連線'))

In [ ]:
import json, time, re
from dataclasses import dataclass, field, asdict
import pandas as pd

client = make_client(FIXTURES)

if OFFLINE_MODE:
    print("✓ 離線重播模式 — 不會連網")
elif RECORD_FIXTURES:
    print("✓ 錄製模式 — 正常呼叫並記錄輸出")
    print(f"  {'Vertex AI' if USE_VERTEX else 'Gemini API'}")
else:
    print(f"✓ {'Vertex AI — project=' + PROJECT_ID if USE_VERTEX else 'Gemini API (AI Studio key)'}")

print(f"  生成模型：{GEN_MODEL}")
print(f"  評審模型：{JUDGE_MODEL}  ← 刻意與生成模型不同，降低 self-preference bias")

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **30:00 — 捲動帶過，不要逐行讀。**
>
> 要說的一句：「注意這幾行 —— 切 Vertex 或 AI Studio 只差 client 的參數，
> 底下所有程式碼一行都不用改。這是部署決策，不是技術決策。」

</div>

## §1 商品資料

12 筆模擬台灣電商 PIM 匯出的商品資料。刻意涵蓋三種法規類別：

- `food` — 保健食品／食品，受《食品安全衛生管理法》第 28 條規範
- `cosmetic` — 美妝保養，受《化粧品衛生安全管理法》第 10 條規範
- `general` — 3C／家電，無特殊廣告限制

**這個欄位是後面所有事情的關鍵。** 「護眼」用在葉黃素上是違規，
用在螢幕護目鏡上不是。規則必須跟著商品屬性走。

In [ ]:
PRODUCTS_RAW = {
 "_meta": {
  "description": "示範用假商品資料，模擬台灣電商 PIM 匯出格式。所有品牌、SKU、價格均為虛構，請勿對應真實商品。",
  "note": "regulated_category 決定套用哪一組法規禁詞：food=食安法, cosmetic=化粧品衛管法, general=無特殊限制",
  "channel_limits": {
   "shopee_title_max": 60,
   "momo_title_max": 50,
   "seo_desc_max": 120,
   "bullet_max": 30
  }
 },
 "products": [
  {
   "sku": "HB-1001",
   "name": "金盞花萃取葉黃素膠囊 60粒",
   "brand": "晨光研選",
   "category": "保健食品 / 眼部保養",
   "regulated_category": "food",
   "price": 890,
   "specs": {
    "淨含量": "60粒 / 盒",
    "主成分": "游離型葉黃素 30mg、玉米黃素 6mg",
    "劑型": "植物膠囊",
    "產地": "台灣",
    "建議食用": "每日1粒，隨餐食用"
   },
   "must_include_keywords": [
    "葉黃素",
    "60粒",
    "30mg"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "25-45 歲長時間使用螢幕的上班族"
  },
  {
   "sku": "HB-1002",
   "name": "五益菌複方益生菌粉 30入",
   "brand": "晨光研選",
   "category": "保健食品 / 消化保健",
   "regulated_category": "food",
   "price": 1180,
   "specs": {
    "淨含量": "30包 / 盒，每包 2g",
    "主成分": "五株專利益生菌，每包 100 億 CFU",
    "劑型": "粉末隨身包",
    "產地": "台灣",
    "建議食用": "每日1包，空腹或睡前"
   },
   "must_include_keywords": [
    "益生菌",
    "30包",
    "100億"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "外食族、作息不規律的上班族"
  },
  {
   "sku": "HB-1003",
   "name": "小分子魚膠原蛋白粉 蔓越莓風味 30入",
   "brand": "晨光研選",
   "category": "保健食品 / 美容保養",
   "regulated_category": "food",
   "price": 1580,
   "specs": {
    "淨含量": "30包 / 盒，每包 3g",
    "主成分": "深海魚膠原蛋白胜肽 2500mg、維生素C 80mg",
    "劑型": "粉末隨身包",
    "產地": "台灣",
    "建議食用": "每日1包，睡前溫水沖泡"
   },
   "must_include_keywords": [
    "膠原蛋白",
    "2500mg",
    "蔓越莓"
   ],
   "brand_tone": "溫和、專業、不誇大，訴求日常持續補充",
   "target_audience": "30-50 歲注重外在保養的女性"
  },
  {
   "sku": "CS-2001",
   "name": "積雪草舒緩精華液 30ml",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 精華液",
   "regulated_category": "cosmetic",
   "price": 780,
   "specs": {
    "容量": "30ml",
    "主成分": "積雪草萃取 5%、泛醇 2%、神經醯胺",
    "質地": "水感精華",
    "適用膚質": "所有膚質，敏弱肌適用",
    "產地": "韓國"
   },
   "must_include_keywords": [
    "積雪草",
    "30ml",
    "5%"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "CS-2002",
   "name": "胺基酸氨基酸溫和洗面乳 120ml",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 清潔",
   "regulated_category": "cosmetic",
   "price": 420,
   "specs": {
    "容量": "120ml",
    "主成分": "胺基酸界面活性劑、洋甘菊萃取",
    "質地": "綿密泡沫",
    "pH值": "弱酸性 pH5.5",
    "產地": "台灣"
   },
   "must_include_keywords": [
    "胺基酸",
    "120ml",
    "pH5.5"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "CS-2003",
   "name": "多重玻尿酸保濕面膜 5片入",
   "brand": "青研 CHINGYEN",
   "category": "美妝保養 / 面膜",
   "regulated_category": "cosmetic",
   "price": 350,
   "specs": {
    "容量": "5片 / 盒，每片 25ml 精華",
    "主成分": "五重玻尿酸、海藻醣",
    "膜布": "天絲蠶絲膜布",
    "適用膚質": "乾燥、缺水肌",
    "產地": "台灣"
   },
   "must_include_keywords": [
    "玻尿酸",
    "5片",
    "天絲"
   ],
   "brand_tone": "清爽、成分透明、理性溫柔，重視敏弱肌友善",
   "target_audience": "20-35 歲敏弱肌、成分黨消費者"
  },
  {
   "sku": "EL-3001",
   "name": "無線降噪藍牙耳機 Pro 2",
   "brand": "Volta",
   "category": "3C / 音訊",
   "regulated_category": "general",
   "price": 3290,
   "specs": {
    "降噪深度": "最高 42dB 主動降噪",
    "續航": "單次 8 小時，含充電盒 32 小時",
    "連線": "藍牙 5.4，支援雙裝置切換",
    "防水": "IPX4",
    "重量": "單耳 4.8g"
   },
   "must_include_keywords": [
    "降噪",
    "32小時",
    "藍牙5.4"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "通勤族、遠距工作者"
  },
  {
   "sku": "EL-3002",
   "name": "65W 氮化鎵三孔快充充電器",
   "brand": "Volta",
   "category": "3C / 充電",
   "regulated_category": "general",
   "price": 990,
   "specs": {
    "輸出": "65W 總輸出，USB-C×2 + USB-A×1",
    "技術": "第三代氮化鎵 GaN",
    "尺寸": "5.2 × 5.2 × 2.9 cm",
    "重量": "112g",
    "保護": "過壓、過流、過熱保護"
   },
   "must_include_keywords": [
    "65W",
    "氮化鎵",
    "三孔"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "出差族、多裝置使用者"
  },
  {
   "sku": "EL-3003",
   "name": "手持無線吸塵器 Lite",
   "brand": "Volta",
   "category": "家電 / 清潔",
   "regulated_category": "general",
   "price": 4680,
   "specs": {
    "吸力": "18000Pa",
    "續航": "標準模式 35 分鐘",
    "集塵盒": "0.5L 可水洗",
    "重量": "1.4kg",
    "配件": "縫隙吸頭、床墊吸頭、軟絨滾筒"
   },
   "must_include_keywords": [
    "18000Pa",
    "35分鐘",
    "1.4kg"
   ],
   "brand_tone": "直接、規格導向、略帶科技感，不用形容詞堆砌",
   "target_audience": "小坪數家庭、養寵物族群"
  },
  {
   "sku": "FD-4001",
   "name": "冷萃掛耳咖啡 淺焙水果調 10入",
   "brand": "溯源焙所",
   "category": "food / 咖啡",
   "regulated_category": "food",
   "price": 460,
   "specs": {
    "淨含量": "10包 / 盒，每包 12g",
    "產區": "衣索比亞 耶加雪菲",
    "烘焙度": "淺焙",
    "風味": "柑橘、莓果、花香",
    "保存": "室溫陰涼處，開封後盡快飲用"
   },
   "must_include_keywords": [
    "耶加雪菲",
    "10包",
    "淺焙"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "25-45 歲精品咖啡愛好者"
  },
  {
   "sku": "FD-4002",
   "name": "低溫烘焙綜合堅果 無調味 350g",
   "brand": "溯源焙所",
   "category": "food / 零食",
   "regulated_category": "food",
   "price": 590,
   "specs": {
    "淨含量": "350g / 罐",
    "內容": "腰果、核桃、杏仁、夏威夷豆",
    "製程": "低溫烘焙，無油炸",
    "調味": "完全無調味、無添加糖鹽",
    "保存": "開封後冷藏，兩週內食用完畢"
   },
   "must_include_keywords": [
    "綜合堅果",
    "350g",
    "無調味"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "注重原型食物的家庭與健身族群"
  },
  {
   "sku": "FD-4003",
   "name": "台灣紅烏龍茶包 三角立體茶包 20入",
   "brand": "溯源焙所",
   "category": "food / 茶葉",
   "regulated_category": "food",
   "price": 380,
   "specs": {
    "淨含量": "20包 / 盒，每包 3g",
    "產區": "台東鹿野",
    "發酵度": "重發酵",
    "風味": "蜜香、熟果、尾韻回甘",
    "沖泡": "95°C 熱水，浸泡 90 秒"
   },
   "must_include_keywords": [
    "紅烏龍",
    "鹿野",
    "20包"
   ],
   "brand_tone": "職人感、重視產區與風味描述，不誇大功效",
   "target_audience": "30-55 歲台灣茶愛好者、送禮需求"
  }
 ]
}

PRODUCTS = PRODUCTS_RAW['products']
CHANNEL_LIMITS = PRODUCTS_RAW['_meta']['channel_limits']

import pandas as pd
pd.DataFrame([{
    'SKU': p['sku'], '商品': p['name'][:22],
    '法規類別': p['regulated_category'], '售價': p['price'],
} for p in PRODUCTS])

### 禁詞清單

依台灣廣告法規整理，分成四組。**罰則差異很大**：

| 類別 | 法源 | 罰則 |
|---|---|---|
| 醫療效能 | 食安法 §28 第2項 | NT$60萬 – 500萬 |
| 不實誇張 | 食安法 §28 第1項 | NT$4萬 – 400萬 |
| 化粧品虛偽誇大 | 化粧品衛管法 §10 | NT$4萬 – 20萬 |

> 這不是工程潔癖，是不做會收罰單。這也是為什麼規則層要放在第一層。

⚠️ 本清單為示範用的簡化版，不構成法律意見；實際商用前應由法務或法規顧問審閱。

In [ ]:
BANNED_DATA = {
 "_meta": {
  "description": "商品文案禁詞清單，依台灣廣告法規分類。用於評測規則層。",
  "disclaimer": "本清單為示範用的簡化版，依主管機關公開認定準則的分類邏輯整理，不構成法律意見。實際商用前應由法務或法規顧問審閱，並以主管機關最新公告為準。",
  "legal_basis": {
   "food": {
    "law": "食品安全衛生管理法 第28條",
    "sub_regulation": "食品及相關產品標示宣傳廣告涉及不實誇張易生誤解或醫療效能認定準則",
    "penalty_exaggeration": "違反第1項（不實、誇張、易生誤解）：新臺幣 4 萬元以上 400 萬元以下罰鍰",
    "penalty_medical": "違反第2項（醫療效能）：新臺幣 60 萬元以上 500 萬元以下罰鍰",
    "criteria_ref": "認定準則 第4條（不實誇張易生誤解）、第5條（醫療效能）"
   },
   "cosmetic": {
    "law": "化粧品衛生安全管理法 第10條",
    "sub_regulation": "化粧品標示宣傳廣告涉及虛偽誇大或醫療效能認定準則",
    "note": "化粧品不得為虛偽或誇大之標示、宣傳、廣告，亦不得有醫療效能之標示、宣傳或廣告。",
    "penalty_exaggeration": "違反第1項（虛偽、誇大）：新臺幣 4 萬元以上 20 萬元以下罰鍰",
    "penalty_escalation": "若宣稱內容已使產品構成藥品，另可能違反藥事法（未經核准擅自製造販賣藥品／誇大醫療效能廣告），罰則顯著加重。說明此類升級風險時應指出其法源不同，勿逕稱『化粧品罰 500 萬』。"
   }
  },
  "important_nuance": "領有衛福部『健康食品』許可證（小綠人標章）的產品，得於許可範圍內宣稱經核可的保健功效。本清單假設商品『未取得』該許可，因此一律禁用。這正是規則層需要按商品屬性套用不同清單的原因。"
 },
 "medical_efficacy": {
  "_note": "涉及疾病之預防、改善、減輕、診斷或治療（認定準則第5條）。這是罰則最重的一類。",
  "applies_to": [
   "food",
   "cosmetic"
  ],
  "severity": "critical",
  "terms": [
   "治療",
   "療效",
   "醫療",
   "根治",
   "治癒",
   "痊癒",
   "藥效",
   "特效",
   "預防疾病",
   "抗癌",
   "防癌",
   "抑制腫瘤",
   "降血壓",
   "降血糖",
   "降膽固醇",
   "降尿酸",
   "降血脂",
   "消炎",
   "抗發炎",
   "殺菌",
   "抗菌",
   "抗病毒",
   "改善過敏",
   "治療過敏",
   "根治過敏",
   "改善失眠",
   "治療失眠",
   "護肝",
   "解毒",
   "排毒",
   "增強免疫力",
   "提升免疫力",
   "調節免疫",
   "抗憂鬱",
   "改善憂鬱",
   "壯陽",
   "生髮",
   "治療掉髮",
   "改善關節炎",
   "治療骨質疏鬆"
  ]
 },
 "exaggeration": {
  "_note": "不實、誇張或易生誤解（認定準則第4條）。含無證據支持之絕對化描述。",
  "applies_to": [
   "food",
   "cosmetic",
   "general"
  ],
  "severity": "high",
  "terms": [
   "最有效",
   "最強",
   "第一名",
   "唯一",
   "獨家專利保證",
   "百分之百有效",
   "100%有效",
   "保證有效",
   "絕對有效",
   "神奇",
   "奇蹟",
   "奇效",
   "驚人效果",
   "完全無副作用",
   "零副作用",
   "無任何副作用",
   "永久",
   "終身有效",
   "一勞永逸",
   "立即見效",
   "馬上見效",
   "三天見效",
   "七天見效",
   "全球最",
   "世界第一",
   "業界唯一"
  ]
 },
 "body_function_food": {
  "_note": "涉及維持或改變人體器官、組織、生理或外觀功能之描述（認定準則第4條第3款）。未取得健康食品許可者不得宣稱。",
  "applies_to": [
   "food"
  ],
  "severity": "high",
  "terms": [
   "美白",
   "淡斑",
   "除皺",
   "抗老",
   "逆齡",
   "回春",
   "瘦身",
   "減肥",
   "燃脂",
   "消脂",
   "瘦小腹",
   "阻斷澱粉",
   "豐胸",
   "長高",
   "改善視力",
   "恢復視力",
   "治療近視",
   "預防近視",
   "增強記憶力",
   "提升智力",
   "變聰明",
   "改善腸道",
   "整腸",
   "促進排便",
   "改善便秘",
   "護眼",
   "顧眼睛"
  ]
 },
 "cosmetic_medical": {
  "_note": "化粧品不得宣稱之醫療或類醫療效能（化粧品衛生安全管理法第10條）。",
  "applies_to": [
   "cosmetic"
  ],
  "severity": "critical",
  "terms": [
   "醫學美容",
   "醫美級",
   "藥用",
   "處方級",
   "換膚",
   "去角質換膚",
   "煥膚療程",
   "治療痘痘",
   "根治粉刺",
   "消除痘疤",
   "除皺",
   "撫平皺紋",
   "填補皺紋",
   "溶脂",
   "拉皮",
   "微整形效果",
   "生髮",
   "防止掉髮",
   "治療禿頭",
   "殺菌",
   "消毒",
   "抗菌配方",
   "修復受損肌膚細胞",
   "重建肌膚屏障"
  ]
 },
 "safe_alternatives": {
  "_note": "示範用的合規替代寫法。用來說明『規則層不只是擋，還能給修改方向』。",
  "mapping": {
   "護眼": "陪伴長時間用眼的日常",
   "改善視力": "日常補充葉黃素",
   "增強免疫力": "均衡飲食之外的日常補充",
   "整腸": "順暢的一天",
   "改善便秘": "維持日常規律作息",
   "美白": "亮采、勻淨膚況（化粧品限用，食品不可）",
   "抗老": "維持日常保養節奏",
   "醫學美容": "溫和配方",
   "殺菌": "清潔",
   "立即見效": "持續使用數週後感受變化",
   "保證有效": "多數使用者回饋"
  }
 }
}

for k, v in BANNED_DATA.items():
    if isinstance(v, dict) and 'terms' in v:
        print(f"{k:24s} {len(v['terms']):>3} 詞  適用：{'/'.join(v['applies_to'])}")

## §2 Prompt v0 → v3

**設計原則：每一版只加一件事。**

這樣評測表上每一欄的改善都能歸因到單一改動。若一次加三件事，
表格會一次全綠，就失去「哪個改動帶來哪個改善」的教學價值。

| 版本 | 加了什麼 | 預期修好 |
|---|---|---|
| v0 | 什麼都不給 | — （基準線）|
| v1 | 角色／受眾／字數／必含規格 | 長度、規格覆蓋 |
| v2 | 法規禁詞 + 品牌語調 few-shot | 禁詞、語調 |
| v3 | `responseSchema` 結構化輸出 | 可機器讀取 |

In [ ]:
"""Prompt v0 → v3 演進。

設計原則：**每一版只加一件事**，這樣評測表上每一欄的改善都能歸因到單一改動。
這是 §2 的核心。

    v0  naive          什麼都不給              → 基準線
    v1  + 通路約束      角色/受眾/字數/必含規格   → 修好「長度」與「規格覆蓋」
    v2  + 法規與語調    禁詞清單 + few-shot     → 修好「禁詞」與「語調」
    v3  + 結構化輸出    responseSchema         → 修好「可機器讀取」

刻意不做的事：不在 v1 就把法規塞進去。若一次加三件事，評測表會一次全綠，
就失去「哪個改動帶來哪個改善」的教學價值。
"""

import json


# --------------------------------------------------------------------------
# v3 的輸出 schema。同時給 Gemini 當 responseSchema，也給規則層當驗證依據。
# --------------------------------------------------------------------------
COPY_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": f"電商商品標題，最多 {SHOPEE_TITLE_MAX} 字",
        },
        "bullets": {
            "type": "array",
            "items": {"type": "string"},
            "description": f"{BULLET_COUNT} 條賣點，每條最多 {BULLET_MAX} 字",
        },
        "seo_description": {
            "type": "string",
            "description": f"SEO 描述，最多 {SEO_DESC_MAX} 字",
        },
        "hashtags": {
            "type": "array",
            "items": {"type": "string"},
            "description": "3-5 個標籤，不含 # 符號",
        },
    },
    "required": ["title", "bullets", "seo_description", "hashtags"],
}


def _product_block(product: dict) -> str:
    """把商品資料整理成 prompt 用的文字區塊。"""
    specs = "\n".join(f"  - {k}：{v}" for k, v in product["specs"].items())
    return (
        f"商品名稱：{product['name']}\n"
        f"品牌：{product['brand']}\n"
        f"分類：{product['category']}\n"
        f"售價：NT${product['price']}\n"
        f"規格：\n{specs}"
    )


# --------------------------------------------------------------------------
# v0 — naive
# --------------------------------------------------------------------------
def build_v0(product: dict, **_) -> str:
    """大多數人第一次寫的 prompt。刻意保持這麼爛。"""
    return f"幫我寫這個商品的電商文案。\n\n{_product_block(product)}"


# --------------------------------------------------------------------------
# v1 — 加通路約束
# --------------------------------------------------------------------------
def build_v1(product: dict, **_) -> str:
    """加入角色、受眾、通路硬限制、必含規格。

    這一版修好的是「可上架」：長度符合平台規則、規格沒有漏。
    還沒處理法規，所以禁詞仍會出現 —— 這是刻意的。
    """
    must = "、".join(product["must_include_keywords"])
    return f"""你是台灣電商平台的資深商品文案編輯。

請為以下商品撰寫文案，目標受眾是{product['target_audience']}。

【格式要求】
- 商品標題：**不超過 {SHOPEE_TITLE_MAX} 個字**
- 賣點：{BULLET_COUNT} 條，每條**不超過 {BULLET_MAX} 個字**
- SEO 描述：**不超過 {SEO_DESC_MAX} 個字**
- 標籤：3-5 個

【必須包含的規格資訊】
以下關鍵字必須出現在文案中，不可省略或改寫：{must}

【商品資料】
{_product_block(product)}"""


# --------------------------------------------------------------------------
# v2 — 加法規約束與品牌語調
# --------------------------------------------------------------------------
def build_v2(product: dict, banned_terms: list[str], tone_examples: list[dict] = None, **_) -> str:
    """在 v1 之上加兩件事：法規禁令、品牌語調 few-shot。

    禁詞清單依商品的 regulated_category 動態帶入 —— 保健食品和 3C 適用的規則不同，
    這是規則層必須「按商品屬性套用」的原因。
    """
    base = build_v1(product)

    banned_display = "、".join(banned_terms[:60])
    legal = {
        "food": "《食品安全衛生管理法》第 28 條",
        "cosmetic": "《化粧品衛生安全管理法》第 10 條",
        "general": "《公平交易法》關於不實廣告之規範",
    }[product["regulated_category"]]

    block = f"""

【法規限制 — 這是硬性要求，違反會被開罰】
本商品受{legal}規範。文案**絕對不可**出現下列詞彙或其同義表達：
{banned_display}

具體而言，不可宣稱：
- 疾病的預防、改善、減輕、診斷或治療
- 維持或改變人體器官、組織、生理或外觀之功能
- 無證據支持的絕對化描述（最有效、保證、100%、立即見效等）

若某個賣點只能用上述詞彙表達，請改用「描述使用情境」或「描述成分事實」的方式改寫，
不要為了避開禁詞而寫出空洞的句子。

【品牌語調】
{product['brand_tone']}"""

    if tone_examples:
        samples = "\n\n".join(
            f"範例 {i + 1}（商品：{ex['product']}）\n標題：{ex['title']}\n賣點：{ex['bullet']}"
            for i, ex in enumerate(tone_examples)
        )
        block += f"""

以下是本品牌既有的合規文案，請模仿其語氣與用字習慣：

{samples}"""

    return base + block


# --------------------------------------------------------------------------
# v3 — 加結構化輸出
# --------------------------------------------------------------------------
def build_v3(product: dict, banned_terms: list[str], tone_examples: list[dict] = None, **_) -> str:
    """v2 + 結構化輸出。

    注意：prompt 本身只多一句話，真正的工作交給 API 的 responseSchema 參數
    （見 generation.py）。這是重點 —— 不要用 prompt 硬凹 JSON 格式，
    要用 API 原生的 structured output，模型才會被約束在 schema 內。
    """
    return build_v2(product, banned_terms, tone_examples) + """

【輸出格式】
請直接輸出符合指定 schema 的 JSON，不要加上 markdown 程式碼區塊標記，不要加任何說明文字。"""


PROMPT_VERSIONS = {
    "v0": build_v0,
    "v1": build_v1,
    "v2": build_v2,
    "v3": build_v3,
}

# v3 是唯一啟用 API 層 structured output 的版本
STRUCTURED_VERSIONS = {"v3"}


# --------------------------------------------------------------------------
# 品牌語調 few-shot 範例（人工撰寫，已確認合規）
# --------------------------------------------------------------------------
TONE_EXAMPLES = {
    "晨光研選": [
        {
            "product": "B群緩釋錠",
            "title": "晨光研選 緩釋B群錠 90錠 每日一錠 全素可食",
            "bullet": "8種B群一次補齊，緩釋設計",
        },
        {
            "product": "鎂錠",
            "title": "晨光研選 甘胺酸鎂錠 120錠 睡前補充 無鎮靜成分",
            "bullet": "選用好吸收的甘胺酸螯合形式",
        },
    ],
    "青研 CHINGYEN": [
        {
            "product": "神經醯胺乳液",
            "title": "青研 神經醯胺修護乳液 50ml 敏弱肌適用 無香料",
            "bullet": "三種神經醯胺，質地清爽不黏膩",
        },
        {
            "product": "溫和卸妝油",
            "title": "青研 純淨卸妝油 150ml 好沖洗 不致粉刺測試",
            "bullet": "乳化快速，沖水後不留油感",
        },
    ],
    "Volta": [
        {
            "product": "行動電源",
            "title": "Volta 20000mAh 行動電源 45W雙向快充 可上飛機",
            "bullet": "45W輸出，筆電也充得動",
        },
        {
            "product": "USB-C 傳輸線",
            "title": "Volta 240W USB-C 編織線 2M 支援40Gbps傳輸",
            "bullet": "240W過電，8K螢幕直出",
        },
    ],
    "溯源焙所": [
        {
            "product": "日曬西達摩掛耳",
            "title": "溯源焙所 日曬西達摩掛耳咖啡 10入 中淺焙",
            "bullet": "日曬處理，草莓與黑糖尾韻",
        },
        {
            "product": "阿里山烏龍",
            "title": "溯源焙所 阿里山高山烏龍 三角茶包 20入 海拔1400m",
            "bullet": "清香型輕發酵，冷泡熱泡皆宜",
        },
    ],
}


def get_banned_terms_for(product: dict, banned_data: dict) -> list[str]:
    """依商品的 regulated_category 取出適用的禁詞。

    保健食品要擋『護眼』，3C 不用 —— 規則必須跟著商品屬性走，
    這是規則層在真實系統裡最容易被做錯的地方。
    """
    category = product["regulated_category"]
    terms: list[str] = []
    for group in banned_data.values():
        if not isinstance(group, dict) or "terms" not in group:
            continue
        if category in group.get("applies_to", []):
            terms.extend(group["terms"])
    return sorted(set(terms))

### 看一下四個版本的實際差異

In [ ]:
p = PRODUCTS[0]   # 葉黃素 — 保健食品，法規風險最高的類別
terms = get_banned_terms_for(p, BANNED_DATA)

for name, fn in PROMPT_VERSIONS.items():
    text = fn(p, banned_terms=terms, tone_examples=TONE_EXAMPLES[p['brand']])
    print(f"{name}: {len(text):>5,} 字元")

print("\n" + "=" * 72)
print("v0 全文（這就是大多數人第一次寫的 prompt）")
print("=" * 72)
print(PROMPT_VERSIONS['v0'](p))

In [ ]:
print("=" * 72)
print("v3 全文 — 注意法規段落與 few-shot 是怎麼寫的")
print("=" * 72)
print(PROMPT_VERSIONS['v3'](p, banned_terms=terms, tone_examples=TONE_EXAMPLES[p['brand']]))

### 生成

`generate(..., structured=True)` 時啟用 API 原生的 `responseSchema` ——
**這是 v3 與 v2 的唯一差別**（呼叫層的程式碼在 §0）。

**重點：不要用 prompt 硬凹 JSON 格式，要用 API 參數約束。**
prompt 裡寫「請輸出 JSON」模型會照做九成的時間；用 `responseSchema`
模型是被解碼器約束在 schema 內，那一成的失敗才會消失。

In [ ]:
ledger = UsageLedger()

# 12 商品 × 4 版 = 48 次呼叫，彼此獨立，所以並行跑。
jobs = [(p, v) for p in PRODUCTS for v in PROMPT_VERSIONS]


def _generate_one(job):
    product, version = job
    prompt = PROMPT_VERSIONS[version](
        product,
        banned_terms=get_banned_terms_for(product, BANNED_DATA),
        tone_examples=TONE_EXAMPLES.get(product['brand']),
    )
    res = generate(client, prompt, structured=(version in STRUCTURED_VERSIONS))
    ledger.record(f"gen:{version}", res.usage)
    return (product['sku'], version), res


t0 = time.time()
outputs = {}   # (sku, version) -> 文案原文
for key, res in run_parallel(_generate_one, jobs, label="生成"):
    outputs[key] = res.text
    if res.error:
        print(f"  ⚠ {key}: {res.error}")

print(f"\n共 {len(outputs)} 筆，耗時 {time.time() - t0:.1f} 秒")

### 肉眼比對：同一個商品，v0 vs v3

In [ ]:
sku = PRODUCTS[0]['sku']
print("=" * 72); print(f"{sku}  v0"); print("=" * 72)
print(outputs[(sku, 'v0')])
print("\n" + "=" * 72); print(f"{sku}  v3"); print("=" * 72)
print(outputs[(sku, 'v3')])

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **33:30 — ▶ 重跑 v0。**「看起來不錯對吧？文筆很好。」（停頓）「但它不能用。」
>
> **35:00 — ▶ 重跑 v3。**「同一個商品。」
>
> 先不要解釋為什麼 v3 比較好 —— 讓他們自己看。解釋留到對比表。

</div>

## §3 評測 — 這一節是重點

三層評測，由便宜到貴：

1. **規則層** — 免費、毫秒級、確定性，可以進 CI
2. **LLM judge** — 有成本、有偏誤，只評規則層評不了的東西
3. **人工抽樣** — 最貴，只看前兩層有分歧的

**順序很重要。** 多數團隊一上來就做 LLM judge，又貴又不穩。
先建規則層：它擋掉大部分明確錯誤，而且結果可重現。

### 第一層：規則

注意 `parse_freeform()` —— 為了評測 v0～v2 的自由文字輸出，
我們被迫寫一個靠正則猜測的 parser。**模型換個排版它就壞掉。**

這段脆弱的程式碼本身就是「為什麼要用 structured output」最好的論據。
v3 之後這整段可以刪掉。

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **36:30 — 秀 `parse_freeform()`，這段工程師最有共鳴。**
>
> 「為了評 v0，我得寫這種靠正則猜測的 parser。模型換個排版它就壞掉。
> **v3 之後這段可以整個刪掉。**」

</div>

In [ ]:
"""評測第一層：規則層。

不呼叫任何模型，毫秒級，零成本。重點：**先建這一層**。
多數團隊一上來就做 LLM judge，又貴又不穩；規則層能擋掉大部分明確錯誤，
而且結果是確定性的 —— 同樣的輸入永遠得到同樣的判定，可以進 CI。

這個模組刻意不 import 任何 GCP 套件，因此離線可測、可單元測試。
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, field



# --------------------------------------------------------------------------
# 把模型輸出正規化成統一結構
# --------------------------------------------------------------------------
@dataclass
class ParsedCopy:
    """從模型輸出解析出的文案結構。"""

    title: str = ""
    bullets: list[str] = field(default_factory=list)
    seo_description: str = ""
    hashtags: list[str] = field(default_factory=list)
    raw: str = ""
    is_structured: bool = False  # True = 直接來自合法 JSON，不需猜測


_TITLE_PAT = re.compile(
    r"^\s*(?:\**)\s*(?:商品)?標題\s*(?:\**)\s*[:：]\s*(?:\**)\s*(.+?)\s*(?:\**)\s*$",
    re.MULTILINE,
)
_BULLET_PAT = re.compile(r"^\s*(?:[-*•・‧]|\d+[.、)])\s*(.+?)\s*$", re.MULTILINE)
_SEO_PAT = re.compile(
    r"^\s*(?:\**)\s*(?:SEO\s*)?(?:描述|說明)\s*(?:\**)\s*[:：]\s*(?:\**)\s*(.+?)\s*(?:\**)\s*$",
    re.MULTILINE | re.IGNORECASE,
)
_HASHTAG_PAT = re.compile(r"#([^\s#,，、]+)")
_FENCE_PAT = re.compile(r"^\s*```(?:json)?\s*|\s*```\s*$", re.MULTILINE)


def parse_structured(text: str) -> ParsedCopy | None:
    """嘗試把輸出當成 JSON 解析。成功才算 schema 合法。"""
    cleaned = _FENCE_PAT.sub("", text).strip()
    try:
        data = json.loads(cleaned)
    except (json.JSONDecodeError, ValueError):
        return None
    if not isinstance(data, dict):
        return None

    required = {"title", "bullets", "seo_description", "hashtags"}
    if not required.issubset(data.keys()):
        return None
    if not isinstance(data["bullets"], list) or not isinstance(data["hashtags"], list):
        return None
    if not isinstance(data["title"], str) or not isinstance(data["seo_description"], str):
        return None

    return ParsedCopy(
        title=data["title"].strip(),
        bullets=[str(b).strip() for b in data["bullets"]],
        seo_description=data["seo_description"].strip(),
        hashtags=[str(h).lstrip("#").strip() for h in data["hashtags"]],
        raw=text,
        is_structured=True,
    )


def parse_freeform(text: str) -> ParsedCopy:
    """從自由文字裡「猜」出結構。

    ⚠️ 這個函式是脆弱的，而那正是重點。
    為了評測 v0～v2 的輸出，我們被迫寫這種靠正則猜測的 parser：
    模型換個排版它就壞掉。v3 用 responseSchema 之後這段就可以整個刪掉。
    這段脆弱的程式碼本身，就是「為什麼要 structured output」最好的論據。
    """
    title_match = _TITLE_PAT.search(text)
    if title_match:
        title = title_match.group(1)
    else:
        # 退而求其次：第一行非空、非 markdown 標記的文字
        title = ""
        for line in text.splitlines():
            stripped = line.strip().lstrip("#").strip()
            if stripped and not stripped.startswith("```"):
                title = stripped
                break

    bullets = [b for b in _BULLET_PAT.findall(text) if len(b) > 2]

    seo_match = _SEO_PAT.search(text)
    seo = seo_match.group(1) if seo_match else ""

    hashtags = _HASHTAG_PAT.findall(text)

    return ParsedCopy(
        title=title.strip(),
        bullets=[b.strip() for b in bullets],
        seo_description=seo.strip(),
        hashtags=hashtags,
        raw=text,
        is_structured=False,
    )


def parse_output(text: str) -> ParsedCopy:
    """先試結構化，失敗才退回猜測。"""
    return parse_structured(text) or parse_freeform(text)


# --------------------------------------------------------------------------
# 規則檢查
# --------------------------------------------------------------------------
@dataclass
class RuleResult:
    sku: str
    version: str
    schema_valid: bool
    title_length_ok: bool
    title_length: int
    bullet_length_ok: bool
    seo_length_ok: bool
    spec_coverage: float          # 0.0 - 1.0
    spec_missing: list[str]
    banned_clean: bool
    banned_hits: list[str]

    @property
    def all_pass(self) -> bool:
        return (
            self.schema_valid
            and self.title_length_ok
            and self.bullet_length_ok
            and self.seo_length_ok
            and self.spec_coverage == 1.0
            and self.banned_clean
        )


def _searchable_text(parsed: ParsedCopy) -> str:
    """組出用來搜尋禁詞與規格關鍵字的文字範圍。

    這裡有一個容易做錯、而且錯了會出事的決定：**掃描範圍要多大？**

    - 結構化輸出（v3）：掃 parsed 後的欄位就夠，範圍精確，不會把 JSON key
      名稱或模型的客套話算進去。
    - 自由文字（v0～v2）：**必須掃 raw 全文**。因為脆弱的 parser 常常只抓到
      第一行，真正的違規詞往往躲在後面幾段。只掃 parsed 欄位會讓違規漏網 ——
      而漏檢比誤判危險得多（誤判只是多花人力複查，漏檢是收罰單）。

    這條差異本身就是「為什麼要用 structured output」的另一個論據：
    有 schema 才能精確掃描，沒有 schema 就只能全文粗掃、誤判率上升。
    """
    if parsed.is_structured:
        return " ".join(
            [parsed.title, *parsed.bullets, parsed.seo_description, *parsed.hashtags]
        ).strip()
    return parsed.raw


def check_banned(text: str, banned_terms: list[str]) -> list[str]:
    """回傳所有命中的禁詞。

    用長詞優先比對，避免「治療」和「治療過敏」重複計數同一段文字。
    """
    hits: list[str] = []
    consumed: list[tuple[int, int]] = []
    for term in sorted(banned_terms, key=len, reverse=True):
        for m in re.finditer(re.escape(term), text):
            span = (m.start(), m.end())
            if any(span[0] >= s and span[1] <= e for s, e in consumed):
                continue  # 已被更長的禁詞涵蓋
            consumed.append(span)
            hits.append(term)
            break  # 同一個詞只記一次
    return hits


def evaluate_rules(
    output_text: str,
    product: dict,
    version: str,
    banned_terms: list[str],
) -> RuleResult:
    """對單一筆模型輸出跑完整規則層。"""
    parsed = parse_output(output_text)
    text = _searchable_text(parsed)

    missing = [kw for kw in product["must_include_keywords"] if kw not in text]
    coverage = 1.0 - len(missing) / max(len(product["must_include_keywords"]), 1)

    bullets_ok = bool(parsed.bullets) and all(
        len(b) <= BULLET_MAX for b in parsed.bullets
    )
    seo_ok = 0 < len(parsed.seo_description) <= SEO_DESC_MAX

    hits = check_banned(text, banned_terms)

    return RuleResult(
        sku=product["sku"],
        version=version,
        schema_valid=parsed.is_structured,
        title_length_ok=0 < len(parsed.title) <= SHOPEE_TITLE_MAX,
        title_length=len(parsed.title),
        bullet_length_ok=bullets_ok,
        seo_length_ok=seo_ok,
        spec_coverage=round(coverage, 3),
        spec_missing=missing,
        banned_clean=not hits,
        banned_hits=hits,
    )


def suggest_fix(hits: list[str], banned_data: dict) -> dict[str, str]:
    """對命中的禁詞給出合規替代寫法。

    重點：規則層不只是「擋」，還能「給修改方向」。
    這讓它從一個惹人厭的 linter 變成一個有用的工具。
    """
    mapping = banned_data.get("safe_alternatives", {}).get("mapping", {})
    return {h: mapping[h] for h in hits if h in mapping}

In [ ]:
"""把評測結果整理成「版本 × 指標」對比表。

這張表是整份分析的結論。前面所有步驟都是為了讓它可信。

設計原則：
- 每一欄對應一個 prompt 改動所修好的問題，欄位順序＝改動順序，階梯感才會出來。
- 顯示通過率（%）而非平均分數，因為「12 個商品裡有幾個能直接上架」比
  「平均 3.8 分」更接近商業決策者要的答案。
"""

from __future__ import annotations


METRIC_LABELS = {
    "schema_valid": "可機器讀取",
    "title_length_ok": "標題長度合規",
    "bullet_length_ok": "賣點長度合規",
    "seo_length_ok": "SEO描述合規",
    "spec_full": "規格完整覆蓋",
    "banned_clean": "法規禁詞 0 命中",
}

# 「這個指標是被哪一版修好的」**必須從資料算出來，不可以硬寫。**
#
# 這裡原本是一份手寫的對照表（title→v1、banned→v2⋯⋯），寫的時候看起來
# 理所當然。後來實跑 50 筆與 200 筆，六條裡有三條被資料推翻：
#
#     賣點長度   標成 v1，實際是 v3 修好的（+62pp）
#     SEO描述    標成 v1，實際是 v3 修好的（+88pp）—— v1 完全沒動
#     法規禁詞   標成 v2，實際 v1 就做掉大部分（+20pp），v2 只是收尾
#
# 一個硬寫的因果宣稱，就擺在真實數字旁邊，而且是錯的 —— 在一份講
# 「怎麼知道它有沒有變好」的材料裡，這是最不該犯的錯。
#
# 所以改成：看哪一版帶來最大的單步改善，就歸因給那一版。
# 資料換了，標註自動跟著換。


def attribution(table: dict[str, dict[str, float]]) -> dict[str, str]:
    """依實測資料判斷每個指標是被哪一版修好的。

    作法很簡單：比較相鄰版本的差距，取單步改善最大的那一版。
    改善幅度太小（<5pp）就標成「—」，不硬掰因果。
    """
    versions = sorted(table)
    out: dict[str, str] = {}
    for metric in METRIC_LABELS:
        gains = [
            (table[versions[i]][metric] - table[versions[i - 1]][metric], versions[i])
            for i in range(1, len(versions))
        ]
        if not gains:
            out[metric] = "—"
            continue
        best_gain, best_version = max(gains)
        out[metric] = best_version if best_gain >= 5 else "—"
    return out


def _metric_values(r: RuleResult) -> dict[str, bool]:
    return {
        "schema_valid": r.schema_valid,
        "title_length_ok": r.title_length_ok,
        "bullet_length_ok": r.bullet_length_ok,
        "seo_length_ok": r.seo_length_ok,
        "spec_full": r.spec_coverage == 1.0,
        "banned_clean": r.banned_clean,
    }


def build_table(results: list[RuleResult]) -> dict[str, dict[str, float]]:
    """results 為所有 (商品 × 版本) 的規則結果，回傳 {版本: {指標: 通過率}}。"""
    by_version: dict[str, list[RuleResult]] = {}
    for r in results:
        by_version.setdefault(r.version, []).append(r)

    table: dict[str, dict[str, float]] = {}
    for version in sorted(by_version):
        rows = by_version[version]
        agg = {m: 0 for m in METRIC_LABELS}
        for r in rows:
            for m, ok in _metric_values(r).items():
                agg[m] += int(ok)
        table[version] = {m: round(100 * c / len(rows), 1) for m, c in agg.items()}
        table[version]["_全數通過"] = round(
            100 * sum(r.all_pass for r in rows) / len(rows), 1
        )
    return table


def to_dataframe(results: list[RuleResult]):
    """回傳 pandas DataFrame，notebook 裡直接顯示用。"""
    import pandas as pd

    table = build_table(results)
    attr = attribution(table)
    ordered = [
        "title_length_ok",
        "bullet_length_ok",
        "seo_length_ok",
        "spec_full",
        "banned_clean",
        "schema_valid",
    ]
    df = pd.DataFrame(
        {
            METRIC_LABELS[m] + f"\n({attr[m]}修好)": [
                table[v][m] for v in sorted(table)
            ]
            for m in ordered
        },
        index=sorted(table),
    )
    df["★ 全數通過"] = [table[v]["_全數通過"] for v in sorted(table)]
    df.index.name = "Prompt 版本"
    return df


def render_text_table(results: list[RuleResult]) -> str:
    """純文字版對比表 —— 不依賴 pandas，離線也能看。"""
    table = build_table(results)
    attr = attribution(table)
    ordered = [
        "title_length_ok",
        "bullet_length_ok",
        "seo_length_ok",
        "spec_full",
        "banned_clean",
        "schema_valid",
    ]
    versions = sorted(table)

    def pad_to(text: str, width: int) -> str:
        """CJK 字元在等寬終端佔兩格，用顯示寬度而非字元數對齊。"""
        display = sum(2 if ord(c) > 127 else 1 for c in text)
        return text + " " * max(width - display, 1)

    width = 22 + 12 + 8 * len(versions)
    lines = [
        pad_to("指標", 22) + pad_to("實測歸因", 12) + "".join(f"{v:>8}" for v in versions),
        "─" * width,
    ]

    for m in ordered:
        lines.append(
            pad_to(METRIC_LABELS[m], 22)
            + pad_to(attr[m] + ("※" if m in PARSER_DEPENDENT else ""), 12)
            + "".join(f"{table[v][m]:>7.0f}%" for v in versions)
        )

    lines.append("─" * width)
    lines.append(
        pad_to("★ 全數通過", 22)
        + pad_to("", 12)
        + "".join(f"{table[v]['_全數通過']:>7.0f}%" for v in versions)
    )
    return "\n".join(lines)


def noise_floor(n_items: int, n_checks_each: int = 1) -> float:
    """這張表上「多少百分點以內的差距應該當成雜訊」。

    用最粗的估計：一次檢查翻轉所造成的百分點變化。12 個商品 × 7 條 rubric
    = 84 次檢查，翻一條就是 1.2 個百分點；所以 2 個百分點以內的差距
    不該當成改善。

    為什麼要放這個函式：demo 的樣本數很小（12 個商品），
    很容易看到 92.9% vs 95.2% 就宣稱「v2 比較好」，但那其實只差一次檢查。
    **把雜訊當成訊號，是評測工作最容易犯、也最傷的錯。**
    """
    total = max(n_items * n_checks_each, 1)
    return round(100.0 / total, 1)


def significance_note(n_items: int, n_checks_each: int = 1) -> str:
    floor = noise_floor(n_items, n_checks_each)
    return (
        f"⚠ 樣本數 {n_items} 筆 × 每筆 {n_checks_each} 項 = {n_items * n_checks_each} 次檢查。\n"
        f"  單次檢查翻轉 ≈ {floor} 個百分點。\n"
        f"  **差距在 {floor * 2:.0f} 個百分點以內時，請當成雜訊，不要宣稱是改善。**\n"
        f"  要做出可靠的小幅比較，需要更大的評測集（實務上 200 筆以上）。"
    )


# 這兩個指標**只在結構化輸出上可靠**。
#
# 自由文字要靠 parse_freeform() 猜出哪幾句是賣點、哪一段是 SEO 描述，
# 而那個 parser 是脆弱的。實測 12 筆時它剛好都猜中，換到 200 筆就大量失效 ——
# 於是 v0～v2 的「賣點長度合規」掉到個位數。
#
# 但那個數字測到的是 **parser 的成功率**，不是模型的合規率。
# 把它當成 0% 呈現會誤導成「模型寫超長」，實際上是我們根本量不到。
# 所以非結構化的版本一律標成「不可測」（NaN），不填 0。
#
# 這件事本身就是 structured output 最有力的論據：
# 沒有 schema，連「有沒有做對」都無法誠實測量。
PARSER_DEPENDENT = ("bullet_length_ok", "seo_length_ok")


def combined_table(rule_results: list[RuleResult], rubric_summary: dict):
    """規則層 + rubric 層的合併對比表。

    每一版只負責修好一件事，這張表要能讓那件事一眼看出來。
    依賴 parser 的指標在非結構化版本上以「—」呈現，不假裝量得到。
    """
    import pandas as pd

    table = build_table(rule_results)
    versions = sorted(table)
    attr = attribution(table)  # 歸因一律由資料算出，欄名不硬寫版本

    # 某一版是否為結構化輸出：用該版 schema_valid 是否過半判定，
    # 不寫死版本名，這樣改了版本命名也不會壞。
    structured = {v: table[v]["schema_valid"] >= 50 for v in versions}

    def parser_dependent(metric: str) -> list:
        return [table[v][metric] if structured[v] else float("nan") for v in versions]

    def rubric_col(key: str) -> list:
        return [rubric_summary.get(v, {}).get(key, float("nan")) for v in versions]

    def rubric_attr(key: str) -> str:
        """rubric 欄位的歸因同樣用實測算，不預設是哪一版。"""
        vals = [rubric_summary.get(v, {}).get(key) for v in versions]
        gains = [
            (vals[i] - vals[i - 1], versions[i])
            for i in range(1, len(versions))
            if vals[i] is not None and vals[i - 1] is not None
        ]
        if not gains:
            return "—"
        best, ver = max(gains)
        return ver if best >= 5 else "—"

    rows = {
        f"標題長度合規 ({attr['title_length_ok']})": [
            table[v]["title_length_ok"] for v in versions
        ],
        f"規格完整覆蓋 ({attr['spec_full']})": [table[v]["spec_full"] for v in versions],
        f"法規禁詞 0 命中 ({attr['banned_clean']})": [
            table[v]["banned_clean"] for v in versions
        ],
        "賣點長度 ※": parser_dependent("bullet_length_ok"),
        "SEO描述長度 ※": parser_dependent("seo_length_ok"),
        f"rubric 通過率 ({rubric_attr('rubric通過率%')})": rubric_col("rubric通過率%"),
        f"可直接上架 ({rubric_attr('可直接上架%')})": rubric_col("可直接上架%"),
        f"可機器讀取 ({attr['schema_valid']})": [table[v]["schema_valid"] for v in versions],
    }
    df = pd.DataFrame(rows, index=versions)
    df.index.name = "Prompt 版本"
    return df


PARSER_CAVEAT = """※ 這兩欄只在結構化輸出上可靠。

自由文字要靠正則猜出哪幾句是賣點、哪一段是 SEO 描述，而那個 parser 很脆弱。
非結構化的版本標成「—」，代表**量不到**，不是「不合格」。

這正是 structured output 的核心價值：沒有 schema，
連「有沒有做對」都無法誠實測量 —— 而測不到的東西就管不了。"""


def violation_detail(results: list[RuleResult], version: str, limit: int = 5) -> str:
    """列出某一版的違規細節 —— demo 中用來證明『這不是假資料』。"""
    rows = [r for r in results if r.version == version and not r.banned_clean]
    if not rows:
        return f"{version}：無禁詞違規 ✓"
    out = [f"{version} 的禁詞命中（前 {limit} 筆）："]
    for r in rows[:limit]:
        out.append(f"  {r.sku}  →  {'、'.join(r.banned_hits)}")
    if len(rows) > limit:
        out.append(f"  ...另有 {len(rows) - limit} 筆")
    return "\n".join(out)

### ★ 規則層對比表

In [ ]:
rule_results = []
for product in PRODUCTS:
    terms = get_banned_terms_for(product, BANNED_DATA)
    for version in PROMPT_VERSIONS:
        rule_results.append(
            evaluate_rules(outputs[(product['sku'], version)], product, version, terms)
        )

print(render_text_table(rule_results))

In [ ]:
# 同一張表的 DataFrame 版本（投影用，數字大一點好讀）
to_dataframe(rule_results).style.format("{:.0f}%").background_gradient(cmap="RdYlGn", vmin=0, vmax=100)

### 違規細節 — 證明這不是假資料

規則層不只能擋，還能給修改方向。這讓它從一個惹人厭的 linter
變成文案人員真的願意用的工具。

In [ ]:
for version in ['v0', 'v1', 'v2', 'v3']:
    print(violation_detail(rule_results, version))
    print()

# 對命中的禁詞給出合規替代寫法
worst = max((r for r in rule_results if r.version == 'v0'),
            key=lambda r: len(r.banned_hits), default=None)
if worst and worst.banned_hits:
    print("=" * 60)
    print(f"{worst.sku} 的修改建議：")
    for bad, good in suggest_fix(worst.banned_hits, BANNED_DATA).items():
        print(f"  ✗ {bad}  →  ✓ {good}")

### 第二層：LLM-as-judge —— 用二元 rubric，不用 1–5 分

只評規則層評不了的：語調、賣點覆蓋。

#### 為什麼不用 1–5 分？

早期的 LLM-as-judge 幾乎都用 Likert 量表。實務上它有幾個難以修復的問題：

| 問題 | 後果 |
|---|---|
| 分數擠在 3–4 分 | 鑑別度低，看不出差異 |
| 同一份文案今天 4 分明天 3 分 | 不可重現，無法判斷是模型變差還是評審漂移 |
| 換評審模型整組分數平移 | 歷史數據作廢 |
| 寫得長、寫得華麗容易拿高分 | verbosity bias |
| **「3.8 分」無法行動** | 你不知道要改什麼 |

二元 rubric 把一個模糊的大問題，拆成一組**具體、可檢查的 yes/no 小問題**：

```
✗  「這份文案的品質有幾分？」          → 3.8 / 5，然後呢？
✓  「文案是否寫出游離型葉黃素 30mg？」  → 否 → 知道要補什麼
```

這也是 Vertex AI Gen AI Evaluation Service 把 adaptive rubrics
形容成**「像單元測試」**的原因。

#### 公平性：rubric 只從商品資料生成，不看被評的文案

**每個商品產生一次，v0～v3 共用同一組。**
如果讓評審看著文案即興出題，每個版本會被問不同的問題 ——
那等於每個考生考不同的考卷，對比表就沒有意義了。

In [ ]:
"""評測第二層：LLM-as-judge，採用**二元自適應 rubric**（pass / fail 檢查清單）。

只評「規則層評不了」的東西 —— 語調、賣點覆蓋這類需要語意理解的維度。
順序很重要：**先跑規則層，被擋下來的不必送 judge**，省下的是真金白銀。

---

## 為什麼不用 1–5 分？

這是這個模組最重要的設計決定。

早期的 LLM-as-judge 幾乎都用 Likert 量表（1–5 分）。實務上它有幾個難以修復的問題：

- **分數擠在中間。** 模型很少給 1 或 5，結果多數樣本都是 3–4 分，鑑別度低。
- **不可重現。** 同一份文案今天 4 分明天 3 分，你無法判斷是模型變差還是評審漂移。
- **跨模型不一致。** 換一個評審模型，整組分數就平移，歷史數據全部作廢。
- **容易被長度騙。** 寫得長、寫得華麗容易拿高分，這正是 verbosity bias。
- **「3.8 分」無法行動。** 你不知道要改什麼。

二元 rubric 把一個模糊的大問題拆成一組**具體、可檢查的 yes/no 小問題**：

    ✗  「這份文案的品質有幾分？」        → 3.8 / 5，然後呢？
    ✓  「文案是否提到 30mg 這個含量？」  → 否 → 知道要補什麼

好處是直接的：**判準明確、可重現、可累積、能直接對應到修改動作**，
而且 pass/fail 很難用堆字數來灌水。這也是為什麼 Vertex AI 的
Gen AI Evaluation Service 把 adaptive rubrics 形容成「像單元測試」。

> 註：Vertex AI 有託管版本的 rubric 評測。本模組選擇手寫，原因有二：
> 一是要看得到 rubric 長什麼樣子，那才是可帶走的概念；
> 二是託管服務的 SDK 介面仍在變動（本機安裝的 `vertexai.evaluation`
> 只有舊的 Likert 型 PointwiseMetric，新的 adaptive rubric 走另一個介面）。
> 概念完全相同，要換成託管版本時，換的是呼叫方式，不是方法論。

---

## 公平性：rubric 只從商品資料生成，不看被評的文案

這點很容易做錯，做錯了整張對比表就沒有意義。

rubric **對每個商品產生一次，v0～v3 共用同一組**。
如果讓評審看著文案即興出題，每個版本會被問不同的問題，
分數就不可比 —— 那等於每個考生考不同的考卷。
"""

from __future__ import annotations

import json
from dataclasses import dataclass, field


# --------------------------------------------------------------------------
# 已知偏誤與本專案的緩解方式
# --------------------------------------------------------------------------
JUDGE_BIASES = {
    "self-preference": {
        "說明": "模型傾向給自己產出的內容較高分。",
        "緩解": f"評審模型（{JUDGE_MODEL}）與生成模型（{GEN_MODEL}）刻意選不同的。",
    },
    "verbosity": {
        "說明": "較長、較華麗的回答容易被評為較好，即使內容沒有更好。",
        "緩解": "改用 pass/fail 判準，「有沒有提到 30mg」無法靠寫得長來灌水。",
    },
    "position": {
        "說明": "成對比較時，先出現的選項容易獲勝。",
        "緩解": "採 pointwise（單篇對照 rubric），不做 pairwise，從源頭避開。",
    },
    "score-clustering": {
        "說明": "Likert 分數容易全部擠在 3–4 分，鑑別度低且不可重現。",
        "緩解": "不用分數。二元判準沒有中間地帶可以躲。",
    },
    "criteria-drift": {
        "說明": "評審每次自己想判準，等於每個版本考不同的考卷。",
        "緩解": "rubric 只從商品資料生成一次，v0～v3 共用同一組。",
    },
}

# --------------------------------------------------------------------------
# Schemas
# --------------------------------------------------------------------------
RUBRIC_GEN_SCHEMA = {
    "type": "object",
    "properties": {
        "rubrics": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "R1, R2, ..."},
                    "criterion": {
                        "type": "string",
                        "description": "一句可用是／否回答的具體判準",
                    },
                    "dimension": {
                        "type": "string",
                        "enum": ["賣點覆蓋", "品牌語調", "消費者可讀性"],
                    },
                    "critical": {
                        "type": "boolean",
                        "description": "未通過就不該上架者為 true",
                    },
                },
                "required": ["id", "criterion", "dimension", "critical"],
            },
        }
    },
    "required": ["rubrics"],
}

RUBRIC_CHECK_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string"},
                    "passed": {"type": "boolean"},
                    "evidence": {
                        "type": "string",
                        "description": "通過就引用文案中的原文；未通過就說明缺什麼",
                    },
                },
                "required": ["id", "passed", "evidence"],
            },
        }
    },
    "required": ["results"],
}

# --------------------------------------------------------------------------
# Prompts
# --------------------------------------------------------------------------
RUBRIC_GEN_PROMPT = """你是資深電商文案主管，要為一個商品訂出「文案驗收清單」。

這份清單會用來檢查 AI 產出的文案能不能上架。請產生 7 條判準。

【⚠️ 最重要：字數預算】
被檢查的文案有嚴格的通路字數限制，總共只有約 {total_budget} 個字：

  - 標題：最多 {title_max} 字
  - 賣點：{bullet_count} 條，每條最多 {bullet_max} 字
  - SEO 描述：最多 {seo_max} 字

**你的每一條判準都必須是在這個字數預算內可以達成的。**

這是最容易出錯的地方。不要寫出需要長篇說明才能滿足的判準：

  ✗ 「文案是否說明為何隨餐食用有助吸收？」    （解釋機制，30 字的賣點塞不下）
  ✗ 「文案是否說明黃金比例能提供更全面的保護？」（同上）
  ✓ 「文案是否寫出葉黃素 30mg 的含量？」       （幾個字就能達成）
  ✓ 「文案是否點出適合長時間看螢幕的族群？」    （一句話能達成）

**如果一條判準只有把文案寫長才可能通過，它就是壞判準。**
壞判準會讓冗長雜亂的文案得高分、讓精簡合規的文案被扣分，
整張評測表的結論就會顛倒。

【判準的寫法】
- 每一條都必須能用「是」或「否」回答，不可以是程度問題。
  ✗ 「文案的語氣是否夠專業？」（程度問題，無法一致判定）
  ✓ 「文案是否避免使用驚嘆號？」（可以直接看出來）
- 判準要**具體指向這個商品**，不要寫成通用的空話。

【⚠️ 不可與規則層衝突】
以下規格關鍵字是**硬性要求**，文案一定會出現它們：

  {must_include}

**絕對不可以寫出「避免使用某某專有名詞」這類會懲罰上述關鍵字的判準。**

這是真實踩過的坑：規則層要求文案必須寫出「游離型葉黃素 30mg」，
rubric 卻出了一條「是否避免使用『游離型』這種需要解釋的專有名詞」——
兩層互相打架，文案怎麼寫都會被扣分，整張評測表就失去意義。

你可以要求「把專有名詞轉成好懂的說法」，但不可以要求「不要出現它」。

【其他不要納入的範圍】
以下已由程式規則自動檢查，**不要**寫成判準：字數是否超標、JSON 格式是否合法、
法規禁詞、必含關鍵字是否存在。你只負責語意層面。

【維度分配 — 必須照這個比例，不可偏重賣點覆蓋】
- 賣點覆蓋：3 條（把規格轉成消費者看得懂的好處，但要在字數內可達成）
- 品牌語調：2 條（是否符合下面的品牌語調設定）
- 消費者可讀性：2 條（是否精簡、無冗詞、無誇大語氣、可直接上架）

【critical 的判定】
未通過就不該上架的設為 true，請設 2 條為 critical。

【商品資料】
商品名稱：{name}
品牌：{brand}
品牌語調設定：{brand_tone}
目標受眾：{audience}
規格：
{specs}

請依 schema 輸出判準清單。"""

RUBRIC_CHECK_PROMPT = """你是電商文案審核員。請對照驗收清單逐條檢查下面這份文案。

【文案的字數預算】
這份文案受通路限制，總共只有約 {total_budget} 個字可用
（標題 {title_max} 字、{bullet_count} 條賣點各 {bullet_max} 字、SEO 描述 {seo_max} 字）。

**請在這個前提下判斷。** 文案簡潔不是缺點，那是通路要求。
不要因為文案沒有展開說明、沒有解釋原理就判定不通過 ——
只要在字數內把該講的講到了，就算通過。

【檢查原則】
- 每一條**只回答通過或不通過**，不要給分數。
- 判斷依據只能是文案中實際出現的內容，不要腦補。
- 通過時，`evidence` 請引用文案中的原文片段。
- 不通過時，`evidence` 請說明缺少什麼。
- **絕對不要因為文案較長、較華麗就傾向通過。**
  冗長、堆砌形容詞、重複同一個賣點，在「消費者可讀性」維度應判為不通過。
- 文案可能是自由文字，也可能是 JSON，兩者一視同仁看內容。

【驗收清單】
{rubric_list}

【待檢查的文案】
{copy_text}

請依 schema 逐條回覆，results 的長度必須與清單條數相同。"""


# --------------------------------------------------------------------------
# 資料結構
# --------------------------------------------------------------------------
@dataclass
class Rubric:
    id: str
    criterion: str
    dimension: str
    critical: bool


@dataclass
class RubricSet:
    sku: str
    rubrics: list[Rubric] = field(default_factory=list)
    error: str | None = None

    def as_prompt_list(self) -> str:
        return "\n".join(
            f"{r.id}. [{r.dimension}]{'（critical）' if r.critical else ''} {r.criterion}"
            for r in self.rubrics
        )


@dataclass
class RubricReport:
    sku: str
    version: str
    passed_ids: list[str] = field(default_factory=list)
    failed: list[tuple[str, str, bool]] = field(default_factory=list)  # (id, 原因, critical)
    total: int = 0
    error: str | None = None

    @property
    def pass_rate(self) -> float:
        return round(100 * len(self.passed_ids) / self.total, 1) if self.total else 0.0

    @property
    def critical_failures(self) -> int:
        return sum(1 for _, _, crit in self.failed if crit)

    @property
    def would_publish(self) -> bool:
        """critical 全過，且整體通過率達 80% 才算能上架。"""
        return self.total > 0 and self.critical_failures == 0 and self.pass_rate >= 80.0


# --------------------------------------------------------------------------
# 呼叫
# --------------------------------------------------------------------------
def _call_json(client, prompt: str, schema: dict, tag: str, ledger=None, max_retries: int = 4):
    """走 structured output 呼叫評審模型並解析 JSON。

    評審本身也用 responseSchema —— 解析失敗會讓整張評測表出現空洞，
    比生成失敗更難察覺。

    **必須重試。** 這一層在高並行下會撞到配額限制（429）。原本沒有重試，
    失敗的那一筆會被上層 except 接住、變成一個帶 error 的空結果 ——
    評測表上看不出來，只會發現受評筆數少了幾筆。那比直接報錯更危險。

    429 的退避拉得比一般錯誤久：配額是時間窗限制，馬上重試只會再撞一次。
    """
    import random
    import time


    last_err: Exception | None = None
    for attempt in range(max_retries):
        started = time.time()
        try:
            resp = client.models.generate_content(
                model=JUDGE_MODEL,
                contents=prompt,
                config=gen_config(
                    temperature=JUDGE_TEMPERATURE,
                    response_mime_type="application/json",
                    response_schema=schema,
                ),
            )
            meta = resp.usage_metadata
            if ledger is not None:
                ledger.record(
                    tag,
                    Usage(
                        model=JUDGE_MODEL,
                        input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                        output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                        latency_s=round(time.time() - started, 2),
                    ),
                )
            return json.loads(resp.text)
        except Exception as e:  # noqa: BLE001
            last_err = e
            if attempt == max_retries - 1:
                break
            text = str(e)
            throttled = "429" in text or "RESOURCE_EXHAUSTED" in text or "quota" in text.lower()
            base = 8.0 if throttled else 1.5
            # 加抖動，避免所有執行緒在同一刻一起重試又一起撞牆
            time.sleep(base * (2**attempt) + random.uniform(0, 1.5))

    raise RuntimeError(f"評審呼叫失敗（重試 {max_retries} 次）：{last_err}")


def _budget() -> dict:
    """文案的字數預算。

    rubric 生成與檢查都必須知道這個數字，否則會產生「解釋原理」這種
    在 30 字賣點裡不可能達成的判準 —— 那會讓冗長的文案得高分、
    精簡合規的文案被扣分，整張評測表的結論顛倒過來。
    """
    total = (
        SHOPEE_TITLE_MAX
        + BULLET_COUNT * BULLET_MAX
        + SEO_DESC_MAX
    )
    return {
        "title_max": SHOPEE_TITLE_MAX,
        "bullet_count": BULLET_COUNT,
        "bullet_max": BULLET_MAX,
        "seo_max": SEO_DESC_MAX,
        "total_budget": total,
    }


def generate_rubrics(client, product: dict, ledger=None) -> RubricSet:
    """為單一商品產生驗收清單。

    ⚠️ 每個商品只做一次，v0～v3 共用。不要對每個版本重新生成 —— 那會讓
    每個版本被問不同的問題，對比表就失去意義。
    """
    specs = "\n".join(f"  - {k}：{v}" for k, v in product["specs"].items())
    prompt = RUBRIC_GEN_PROMPT.format(
        name=product["name"],
        brand=product["brand"],
        brand_tone=product["brand_tone"],
        audience=product["target_audience"],
        specs=specs,
        # 讓 rubric 生成知道哪些字是規則層的硬性要求，否則會出現
        # 「避免使用這個專有名詞」這種與規則層打架的判準。
        must_include="、".join(product["must_include_keywords"]),
        **_budget(),
    )
    try:
        data = _call_json(client, prompt, RUBRIC_GEN_SCHEMA, "judge:rubric_gen", ledger)
        return RubricSet(
            sku=product["sku"],
            rubrics=[
                Rubric(
                    id=str(r["id"]),
                    criterion=str(r["criterion"]),
                    dimension=str(r["dimension"]),
                    critical=bool(r["critical"]),
                )
                for r in data["rubrics"]
            ],
        )
    except Exception as e:  # noqa: BLE001
        return RubricSet(sku=product["sku"], error=str(e))


def check_against_rubrics(
    client,
    copy_text: str,
    rubric_set: RubricSet,
    version: str,
    ledger=None,
) -> RubricReport:
    """逐條檢查文案。"""
    if rubric_set.error or not rubric_set.rubrics:
        return RubricReport(
            sku=rubric_set.sku,
            version=version,
            error=rubric_set.error or "沒有可用的 rubric",
        )

    by_id = {r.id: r for r in rubric_set.rubrics}
    prompt = RUBRIC_CHECK_PROMPT.format(
        rubric_list=rubric_set.as_prompt_list(),
        copy_text=copy_text[:4000],  # 防止異常長輸出把成本吃掉
        **_budget(),
    )
    try:
        data = _call_json(client, prompt, RUBRIC_CHECK_SCHEMA, f"judge:{version}", ledger)
        passed, failed = [], []
        for row in data["results"]:
            rid = str(row["id"])
            rub = by_id.get(rid)
            if rub is None:
                continue  # 評審捏造了不存在的 id，忽略
            if bool(row["passed"]):
                passed.append(rid)
            else:
                failed.append((rid, str(row.get("evidence", "")), rub.critical))
        return RubricReport(
            sku=rubric_set.sku,
            version=version,
            passed_ids=passed,
            failed=failed,
            total=len(by_id),
        )
    except Exception as e:  # noqa: BLE001
        return RubricReport(sku=rubric_set.sku, version=version, error=str(e))


def should_judge(rule_result) -> bool:
    """規則層已經擋下的，就不要送 judge —— 這是省錢的關鍵。

    語意檢查對一份已經違反法規的文案沒有意義，而每一次 judge 呼叫
    都是用比生成更貴的模型在燒錢。
    """
    return rule_result.banned_clean and rule_result.spec_coverage > 0.5


def summarize_rubrics(
    reports: list[RubricReport], total_items: int | None = None
) -> dict[str, dict[str, float]]:
    """彙整成 {版本: 指標}，供對比表使用。

    `total_items` 是**每個版本應有的商品總數**（例如 12）。給了它，
    「可直接上架%」就會用固定分母計算，把規則層擋下、沒送評分的那些
    一律算成不可上架。

    為什麼一定要這樣做 —— 這是實測踩到的坑：

    規則層擋下的都是最爛的那幾筆，它們不會進 rubric 評分。結果 v0 只有
    10 筆受評、v1～v3 各 12 筆，v0 的平均分被「倖存者偏差」灌水成
    74.3%，看起來比實際好。**兩個版本用不同分母，就不能直接比。**

    處理方式：
    - 「可直接上架%」用固定分母。被規則層擋下的本來就不能上架，
      算成不通過完全合理，這樣跨版本才可比，也是真正該拿來做決策的數字。
    - 「rubric通過率%」維持只在受評項目上平均（否則無法區分「沒評」與
      「評了但沒過」），但一定要一起看「受評筆數」。分母不同時不要直接比。
    """
    by_version: dict[str, list[RubricReport]] = {}
    for r in reports:
        if r.error:
            continue
        by_version.setdefault(r.version, []).append(r)

    out = {}
    for version, rows in sorted(by_version.items()):
        if not rows:
            continue
        denom = total_items or len(rows)
        out[version] = {
            "rubric通過率%": round(sum(r.pass_rate for r in rows) / len(rows), 1),
            "critical違反數": sum(r.critical_failures for r in rows),
            "可直接上架%": round(100 * sum(r.would_publish for r in rows) / denom, 1),
            "受評筆數": len(rows),
            "分母": denom,
        }
    return out


def most_common_failures(reports: list[RubricReport], rubric_sets: dict, limit: int = 5):
    """哪些判準最常沒過 —— 這才是能拿去改 prompt 的資訊。

    這正是二元 rubric 勝過分數的地方：「3.8 分」不能行動，
    「有 9 個商品沒寫出含量」可以。
    """
    counts: dict[tuple[str, str], int] = {}
    for rep in reports:
        if rep.error:
            continue
        rs = rubric_sets.get(rep.sku)
        if not rs:
            continue
        text_by_id = {r.id: r.criterion for r in rs.rubrics}
        for rid, _reason, _crit in rep.failed:
            key = (rid, text_by_id.get(rid, rid))
            counts[key] = counts.get(key, 0) + 1
    ranked = sorted(counts.items(), key=lambda kv: -kv[1])[:limit]
    return [(criterion, n) for (_rid, criterion), n in ranked]

In [ ]:
for name, info in JUDGE_BIASES.items():
    print(f"● {name}")
    print(f"    問題：{info['說明']}")
    print(f"    緩解：{info['緩解']}\n")

#### 步驟一：為每個商品產生驗收清單

這一步和被評的文案無關，只看商品資料。

In [ ]:
t0 = time.time()
rubric_sets = dict(
    run_parallel(
        lambda p: (p['sku'], generate_rubrics(client, p, ledger)),
        PRODUCTS,
        label="產生 rubric",
    )
)
print(f"  耗時 {time.time() - t0:.1f} 秒")

demo = rubric_sets[PRODUCTS[0]['sku']]
print(f"{PRODUCTS[0]['name']} 的驗收清單（{len(demo.rubrics)} 條）\n")
for r in demo.rubrics:
    mark = "★" if r.critical else " "
    print(f"  {mark} {r.id}  [{r.dimension}] {r.criterion}")
print("\n★ = critical，未通過就不該上架")

#### 步驟二：逐條檢查

只送規則層過關的文案去評審 —— **這是省錢的關鍵**。
對一份已經違反法規的文案做語意檢查沒有意義，而 judge 用的是比生成更貴的模型。

In [ ]:
by_key = {(r.sku, r.version): r for r in rule_results}

# 先篩選再送審：規則層擋下的不必評，這一步就是在省錢。
to_judge = [
    (p, v) for p in PRODUCTS for v in PROMPT_VERSIONS
    if should_judge(by_key[(p['sku'], v)])
]
skipped = len(PRODUCTS) * len(PROMPT_VERSIONS) - len(to_judge)
print(f"送審 {len(to_judge)} 筆，規則層已擋下 {skipped} 筆（省下的就是錢）")

# 這是整份 notebook 最慢的一段：評審模型單次十幾秒。
# 序列跑約 9 分鐘，並行後約 1 分鐘。
t0 = time.time()
rubric_reports = run_parallel(
    lambda job: check_against_rubrics(
        client, outputs[(job[0]['sku'], job[1])],
        rubric_sets[job[0]['sku']], job[1], ledger
    ),
    to_judge,
    label="評審",
)
print(f"  耗時 {time.time() - t0:.1f} 秒\n")
pd.DataFrame(summarize_rubrics(rubric_reports, total_items=len(PRODUCTS))).T

#### 最常沒過的判準 —— 這才是能拿去改 prompt 的東西

這正是二元 rubric 勝過分數的地方：「3.8 分」不能行動，
「有 9 個商品沒寫出含量」可以。

In [ ]:
for criterion, n in most_common_failures(rubric_reports, rubric_sets):
    print(f"  {n:>2} 次未通過 — {criterion}")

worst = [r for r in rubric_reports if r.version == 'v0' and r.failed]
if worst:
    r = worst[0]
    print(f"\n{r.sku} v0 未通過的細節（{r.pass_rate}% 通過）：")
    for rid, reason, crit in r.failed[:4]:
        print(f"  {'★' if crit else ' '} {rid}: {reason}")

### ★★ 合併對比表 —— 每一版修好一件事

這張表是整份 notebook 的結論。左邊三欄由 v1 修好、中間兩欄由 v2 修好、
最後一欄由 v3 修好 —— **每一版的貢獻都能單獨看見**。

In [ ]:
print(significance_note(len(PRODUCTS), 7))

combined = combined_table(
    rule_results, summarize_rubrics(rubric_reports, total_items=len(PRODUCTS))
)
print(PARSER_CAVEAT)
combined.style.format("{:.1f}%", na_rep="—").background_gradient(
    cmap="RdYlGn", vmin=0, vmax=100)

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **38:00 — ★ 全場高潮。表格出來後停 5 秒不要說話，讓大家看。**
>
> 然後：「每一版修好一件事。左邊三欄是 v1、中間兩欄是 v2、最後一欄是 v3。」
>
> **接著一定要講這兩句，這是誠信也是專業度：**
>
> 1.「台上這張是 12 個商品，樣本太小。**真正的結論來自另外跑的 50 筆評測集**
>    —— 就是投影片 S17c 那組數字。demo 給你看流程，結論要看評測集。」
> 2.「50 筆的結果：v0→v1、v1→v2 是真的改善；**v2→v3 只差 0.3 個百分點，
>    那是雜訊。** v3 贏的不是文案品質，是可機器讀取從 0 變成 100%。」
>
> **不要**說「你看全部都變好了」。台下有工程師，會扣分。

</div>

#### 讀這張表的時候要小心兩件事

**一、這裡的樣本太小，只能看趨勢，不能下結論。**
12 個商品、rubric 每個 7 條，共 84 次檢查，翻轉一次就是 1.2 個百分點。
把雜訊當成訊號，是評測工作最容易犯、也最傷的錯。

真正的結論來自另外跑的 **50 筆評測集**（`poc/run_eval.py`）——
350 次檢查、雜訊底線 0.29 個百分點。實測結果：

| | rubric 通過率 | 可直接上架 | 可機器讀取 |
|---|---|---|---|
| v0 | 68.4% | 14% | 0% |
| v1 | 79.1% | 42% | 0% |
| v2 | 86.3% | 72% | 0% |
| v3 | 86.6% | 68% | **100%** |

**二、v2 → v3 只差 0.3 個百分點 —— 那是雜訊，不能說 v3 讓文案變好。**

v3 真正的價值在最後一欄。**v3 不是讓文案變好，是讓文案變得可用。**
這是兩件不同的事，而你兩個都需要。

一張每格都完美遞增的表，通常代表有人在調數字。

**三、上面那一欄「實測歸因」是算出來的，不是寫死的。**
它比較相鄰版本的差距、取單步改善最大的那一版。這件事本身有教訓：
最早這裡是一份手寫對照表，後來實跑 50 筆，六條裡有三條被資料推翻
（例如「SEO描述」原本標成 v1 修好，實際上 v1 完全沒動，是 v3 修的）。
**在講評測的材料裡放一個硬寫的因果宣稱，是最不該犯的錯。**

## §4 場景 B：評論洞察

方法論完全相同，方向相反：

    場景 A   結構化資料 → 非結構化文字   （生成）
    場景 B   非結構化文字 → 結構化資料   （抽取）

**這一節的目的不是教評論分析**，而是證明你剛學的是一套流程，不是一個文案技巧。
同樣要 schema、同樣要評測、同樣要算成本。

In [ ]:
"""場景 B：評論洞察分析。

方法論與場景 A 完全相同，方向相反：
    場景 A  結構化資料 → 非結構化文字   （生成）
    場景 B  非結構化文字 → 結構化資料   （抽取）

這一節的重點不是評論分析本身，而是證明**這是一套可遷移的流程，
不是一個文案技巧**。同樣要 schema、同樣要評測、同樣要算成本。

商業價值一句話：把「客服每天讀 500 則評論」變成「BigQuery 一句 SQL」。
"""

from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from datetime import UTC


ASPECTS = ["產品品質", "價格", "物流配送", "包裝", "客服", "使用體驗", "其他"]

REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["positive", "neutral", "negative"],
        },
        "aspects": {
            "type": "array",
            "items": {"type": "string", "enum": ASPECTS},
            "description": "這則評論實際談到的面向，可多選",
        },
        "is_about_product": {
            "type": "boolean",
            "description": "評論主體是商品本身，還是物流／客服等非商品因素",
        },
        "actionable_suggestion": {
            "type": "string",
            "description": "可執行的改善建議；若無則填空字串",
        },
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": [
        "sentiment",
        "aspects",
        "is_about_product",
        "actionable_suggestion",
        "urgency",
    ],
}

EXTRACT_PROMPT = """你是電商營運分析師，要把客戶評論轉成結構化資料進資料倉儲。

【重要判準】
- `is_about_product`：評論在抱怨物流慢、客服態度、包裝破損時，**主體不是商品**，
  應為 false。這一欄的用途是把「商品不好」和「服務不好」分開 ——
  兩者要交給不同部門處理，混在一起會導致採購端誤判商品品質。
- `urgency`：只有涉及安全疑慮、法規風險、或大量客戶可能受影響時才是 high。
  單純的個人喜好不合不算。
- `actionable_suggestion`：必須是團隊真的能執行的動作（改包裝、補說明、調規格），
  不要寫「提升品質」這種無法執行的空話。無明確建議就留空字串。

【評論】
商品：{product_name}
評分：{rating} 星
內容：{text}

請依 schema 輸出。"""


@dataclass
class ReviewInsight:
    review_id: str
    sku: str
    rating: int
    sentiment: str
    aspects: list[str]
    is_about_product: bool
    actionable_suggestion: str
    urgency: str
    error: str | None = None


def extract_one(client, review: dict, product_name: str, ledger=None) -> ReviewInsight:
    """把單則評論轉成結構化資料。"""
    import time


    prompt = EXTRACT_PROMPT.format(
        product_name=product_name,
        rating=review["rating"],
        text=review["text"],
    )

    started = time.time()
    try:
        resp = client.models.generate_content(
            model=GEN_MODEL,
            contents=prompt,
            config=gen_config(
                temperature=0.0,  # 抽取任務要可重現，不要創意
                response_mime_type="application/json",
                response_schema=REVIEW_SCHEMA,
            ),
        )
        meta = resp.usage_metadata
        if ledger is not None:
            ledger.record(
                "gen:insight",
                Usage(
                    model=GEN_MODEL,
                    input_tokens=getattr(meta, "prompt_token_count", 0) or 0,
                    output_tokens=getattr(meta, "candidates_token_count", 0) or 0,
                    latency_s=round(time.time() - started, 2),
                ),
            )
        data = json.loads(resp.text)
        return ReviewInsight(
            review_id=review["review_id"],
            sku=review["sku"],
            rating=review["rating"],
            sentiment=data["sentiment"],
            aspects=list(data["aspects"]),
            is_about_product=bool(data["is_about_product"]),
            actionable_suggestion=data["actionable_suggestion"],
            urgency=data["urgency"],
        )
    except Exception as e:  # noqa: BLE001
        return ReviewInsight(
            review_id=review["review_id"],
            sku=review["sku"],
            rating=review["rating"],
            sentiment="neutral",
            aspects=[],
            is_about_product=True,
            actionable_suggestion="",
            urgency="low",
            error=str(e),
        )


def validate_insights(insights: list[ReviewInsight]) -> dict:
    """場景 B 也要評測 —— 這是本節的重點。

    抽取任務沒有「好文案」這種主觀標準，但仍有可自動檢查的規則：
    - enum 值是否合法
    - 星等與情緒是否矛盾（1 星卻標 positive，通常是抽取錯誤）
    - 高星等卻標 high urgency（同樣可疑）

    這些規則抓到的不是「模型寫得不好」，而是「模型理解錯了」—— 更嚴重。
    """
    issues = []
    for r in insights:
        if r.error:
            issues.append((r.review_id, f"抽取失敗：{r.error}"))
            continue
        if r.sentiment not in {"positive", "neutral", "negative"}:
            issues.append((r.review_id, f"sentiment 值非法：{r.sentiment}"))
        if bad := [a for a in r.aspects if a not in ASPECTS]:
            issues.append((r.review_id, f"aspects 出現未定義值：{bad}"))
        if r.rating <= 2 and r.sentiment == "positive":
            issues.append((r.review_id, f"{r.rating} 星卻判為 positive，疑似抽取錯誤"))
        if r.rating >= 4 and r.sentiment == "negative":
            issues.append((r.review_id, f"{r.rating} 星卻判為 negative，疑似抽取錯誤"))
        if r.rating == 5 and r.urgency == "high":
            issues.append((r.review_id, "5 星卻標記 high urgency，請人工確認"))

    return {
        "總筆數": len(insights),
        "問題筆數": len(issues),
        "通過率": round(100 * (len(insights) - len(issues)) / max(len(insights), 1), 1),
        "明細": issues,
    }


def to_dataframe(insights: list[ReviewInsight]):
    """轉成 DataFrame。欄位刻意設計成可直接進 BigQuery 的扁平結構。"""
    import pandas as pd

    rows = []
    for r in insights:
        d = asdict(r)
        d["aspects"] = "|".join(r.aspects)  # BigQuery 可用 SPLIT() 還原
        rows.append(d)
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# BigQuery —— 把「AI 當 ETL」這件事真的做完
# --------------------------------------------------------------------------
BQ_SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS `{project}.{dataset}.{table}` (
  review_id             STRING  NOT NULL,
  sku                   STRING  NOT NULL,
  rating                INT64,
  sentiment             STRING,          -- positive / neutral / negative
  aspects               ARRAY<STRING>,   -- 一則評論可能談到多個面向
  is_about_product      BOOL,            -- 把商品問題與服務問題分開的關鍵欄位
  actionable_suggestion STRING,
  urgency               STRING,          -- low / medium / high
  ingested_at           TIMESTAMP
)
PARTITION BY DATE(ingested_at)
CLUSTER BY sku, sentiment
"""

# 這句 SQL 就是整個場景 B 的商業價值：客服原本要讀 500 則評論，現在是一句查詢。
BQ_INSIGHT_SQL = """
SELECT
  sku,
  COUNTIF(is_about_product AND sentiment = 'negative') AS 商品負評,
  COUNTIF(NOT is_about_product AND sentiment = 'negative') AS 服務負評,
  COUNTIF(urgency = 'high')                             AS 高急迫,
  ROUND(AVG(rating), 2)                                 AS 平均星等
FROM `{project}.{dataset}.{table}`
GROUP BY sku
HAVING 商品負評 > 0 OR 服務負評 > 0
ORDER BY 商品負評 DESC, 服務負評 DESC
"""


def load_to_bigquery(insights: list[ReviewInsight], verbose: bool = True):
    """把抽取結果寫進 BigQuery。

    這一步讓「非結構化文字 → 結構化資料」這句話變成真的 —— 資料真的落地、
    真的能被 SQL 查詢、真的能接上既有的 BI 報表。

    schema 的設計重點在 `is_about_product`：把「商品不好」與「服務不好」
    分開儲存。混在一起統計，採購會以為商品品質有問題，實際上是物流慢。
    """
    from datetime import datetime

    from google.cloud import bigquery

    client = bigquery.Client(project=PROJECT_ID)
    ds_id = f"{PROJECT_ID}.{BQ_DATASET}"

    ds = bigquery.Dataset(ds_id)
    ds.location = BQ_LOCATION
    client.create_dataset(ds, exists_ok=True)

    table_ref = f"{ds_id}.{BQ_TABLE}"
    client.query(
        BQ_SCHEMA_SQL.format(
            project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE
        )
    ).result()

    now = datetime.now(UTC).isoformat()
    rows = [
        {
            "review_id": r.review_id,
            "sku": r.sku,
            "rating": r.rating,
            "sentiment": r.sentiment,
            "aspects": r.aspects,
            "is_about_product": r.is_about_product,
            "actionable_suggestion": r.actionable_suggestion,
            "urgency": r.urgency,
            "ingested_at": now,
        }
        for r in insights
        if not r.error
    ]
    if not rows:
        raise RuntimeError("沒有可寫入的資料（全部抽取失敗）")

    errors = client.insert_rows_json(table_ref, rows)
    if errors:
        raise RuntimeError(f"BigQuery 寫入失敗：{errors}")

    if verbose:
        print(f"✓ 已寫入 {len(rows)} 列 → {table_ref}")
        print("  分區：DATE(ingested_at)　叢集：sku, sentiment")
    return table_ref


def query_bigquery_insights():
    """跑那句「取代人工讀 500 則評論」的 SQL。"""
    from google.cloud import bigquery

    client = bigquery.Client(project=PROJECT_ID)
    sql = BQ_INSIGHT_SQL.format(
        project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE
    )
    return client.query(sql).to_dataframe()


def business_summary(insights: list[ReviewInsight]) -> str:
    """重點在這裡：資料變成決策。"""
    ok = [r for r in insights if not r.error]
    if not ok:
        return "無有效資料"

    product_issues = [r for r in ok if r.is_about_product and r.sentiment == "negative"]
    service_issues = [
        r for r in ok if not r.is_about_product and r.sentiment == "negative"
    ]
    urgent = [r for r in ok if r.urgency == "high"]

    aspect_counts: dict[str, int] = {}
    for r in ok:
        if r.sentiment == "negative":
            for a in r.aspects:
                aspect_counts[a] = aspect_counts.get(a, 0) + 1

    top = sorted(aspect_counts.items(), key=lambda kv: -kv[1])[:3]
    suggestions = [r.actionable_suggestion for r in ok if r.actionable_suggestion]

    lines = [
        f"有效評論 {len(ok)} 則",
        "",
        f"  商品相關負評   {len(product_issues):>3} 則  → 採購／研發要看",
        f"  服務相關負評   {len(service_issues):>3} 則  → 物流／客服要看",
        f"  高急迫案件     {len(urgent):>3} 則  → 今天就要處理",
        "",
        "負評集中的面向：",
    ]
    lines += [f"  {a}：{c} 則" for a, c in top] or ["  （無）"]
    lines += ["", f"可執行建議 {len(suggestions)} 條，例如："]
    lines += [f"  - {s}" for s in suggestions[:3]]
    lines += [
        "",
        "↑ 這就是價值所在：把「客服每天讀 500 則評論」變成一句 SQL。",
        "  注意『商品負評』與『服務負評』已經分開 —— 混在一起會讓採購誤判商品品質。",
    ]
    return "\n".join(lines)

In [ ]:
REVIEWS = {
 "_meta": {
  "description": "示範用假評論資料，模擬台灣電商評論區的真實樣態：長短不一、有錯字、有離題、有物流抱怨混在商品評價裡。",
  "note": "刻意保留雜訊（注音、錯字、只有標點的短評、抱怨物流而非商品），用來展示模型在髒資料上的表現，這才是真實情況。",
  "target_schema": {
   "sentiment": "positive | neutral | negative",
   "aspects": "list of 產品品質 | 價格 | 物流配送 | 包裝 | 客服 | 使用體驗 | 其他",
   "is_about_product": "bool — 用來把物流客訴從商品評價中分離",
   "actionable_suggestion": "string | null",
   "urgency": "low | medium | high"
  }
 },
 "reviews": [
  {
   "review_id": "R001",
   "sku": "HB-1001",
   "rating": 5,
   "text": "吃了快兩個月，本來只是想說試試看，結果現在每天都會記得吃。膠囊不大顆很好吞，也沒有什麼味道。會回購。"
  },
  {
   "review_id": "R002",
   "sku": "HB-1001",
   "rating": 2,
   "text": "東西還沒拆就先說，寄來的時候盒子整個壓扁了…雖然裡面膠囊看起來沒事但這樣送禮很尷尬耶，希望包裝可以加強"
  },
  {
   "review_id": "R003",
   "sku": "HB-1002",
   "rating": 4,
   "text": "粉末沖水很好溶解不會結塊這點加分，味道偏淡幾乎沒味道我可以接受。唯一想抱怨的是一盒才30包但價格快1200，有點貴"
  },
  {
   "review_id": "R004",
   "sku": "HB-1002",
   "rating": 1,
   "text": "下單到收到花了九天！！！客服問了三次都說在處理中，最後也沒給我明確時間。產品本身還沒吃不評論，但這個出貨速度真的不行"
  },
  {
   "review_id": "R005",
   "sku": "HB-1003",
   "rating": 5,
   "text": "蔓越莓口味比我想像中好喝很多，不會有腥味這點很重要！！睡前泡一杯變成習慣了"
  },
  {
   "review_id": "R006",
   "sku": "HB-1003",
   "rating": 3,
   "text": "還可以吧"
  },
  {
   "review_id": "R007",
   "sku": "CS-2001",
   "rating": 5,
   "text": "敏感肌用了三週沒有刺痛感，這對我來說已經是很高的評價了。質地是水感的很好推開，上妝前用也不會搓泥。唯一小缺點是滴管有時候會沾到瓶口"
  },
  {
   "review_id": "R008",
   "sku": "CS-2001",
   "rating": 2,
   "text": "我用了長痘痘...不知道是不是我不適合，成分表看起來是很溫和沒錯啦。退貨流程倒是很順利客服人很好"
  },
  {
   "review_id": "R009",
   "sku": "CS-2002",
   "rating": 4,
   "text": "泡沫很綿密洗完不緊繃，但120ml用得有點快，希望出大容量版本"
  },
  {
   "review_id": "R010",
   "sku": "CS-2003",
   "rating": 1,
   "text": "膜布跟我臉根本不合，眼睛下面那塊一直翹起來，敷到一半就掉了。精華液倒是蠻多的但都流到脖子去了= ="
  },
  {
   "review_id": "R011",
   "sku": "EL-3001",
   "rating": 5,
   "text": "降噪真的有感，捷運上開最強模式幾乎聽不到廣播聲。續航也很夠用，通勤來回一週充一次盒子就好。雙裝置切換超方便，手機筆電來回切不用重連"
  },
  {
   "review_id": "R012",
   "sku": "EL-3001",
   "rating": 3,
   "text": "音質沒話說但戴久了耳朵會痛，我耳道比較小可能不適合。另外通話時對方說我這邊有雜音，不知道是不是個案"
  },
  {
   "review_id": "R013",
   "sku": "EL-3002",
   "rating": 5,
   "text": "體積真的小！！之前那顆65W是這個的兩倍大。三個孔同時插筆電手機耳機都還能跑滿速，出差只帶這顆就夠"
  },
  {
   "review_id": "R014",
   "sku": "EL-3003",
   "rating": 2,
   "text": "吸力是不錯啦可是集塵盒也太小了吧0.5L根本掃不到一半就要倒，而且倒的時候灰塵會噴出來設計有問題"
  },
  {
   "review_id": "R015",
   "sku": "FD-4001",
   "rating": 4,
   "text": "耶加雪菲的花香有出來，掛耳能做到這樣不容易。不過我覺得建議沖法的水溫寫得有點模糊，第一次沖太燙有澀感，後來自己調到88度才對"
  }
 ]
}['reviews']

name_by_sku = {p['sku']: p['name'] for p in PRODUCTS}
insights = run_parallel(
    lambda r: extract_one(client, r, name_by_sku.get(r['sku'], '未知商品'), ledger),
    REVIEWS,
    label='抽取評論',
)

to_dataframe(insights)[['review_id','rating','sentiment','aspects',
                        'is_about_product','urgency']]

### 抽取任務也要評測

沒有「好文案」這種主觀標準，但仍有可自動檢查的規則：
星等與情緒矛盾、enum 值非法、5 星卻標高急迫。

這些抓到的不是「模型寫得不好」，而是**「模型理解錯了」—— 更嚴重**。

In [ ]:
check = validate_insights(insights)
print(f"通過率 {check['通過率']}%  （{check['總筆數'] - check['問題筆數']}/{check['總筆數']}）")
for rid, msg in check['明細']:
    print(f"  ⚠ {rid}: {msg}")

print("\n" + "=" * 60)
print(business_summary(insights))

#### 落地到 BigQuery — 讓「變成結構化資料」不只是說說

到這裡為止，結構化資料還只存在於記憶體裡。真正的價值要等它**進到資料倉儲、
能被 SQL 查詢、能接上既有的 BI 報表**才會發生。

注意 schema 的設計重點：`is_about_product` 這個欄位把「商品不好」與
「服務不好」分開儲存。混在一起統計，採購會以為商品品質有問題，
實際上是物流慢。

> 預設 `USE_BIGQUERY = False`。要真的寫入請在 CONFIG 打開，
> 並確認專案已啟用 BigQuery API 且帳號有 `bigquery.dataEditor` 權限。
> 離線重播模式不涵蓋 BigQuery 呼叫。

In [ ]:
print("建表 SQL：")
print(BQ_SCHEMA_SQL.format(project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE))
print("\n分析 SQL —— 這句就是整個場景 B 的商業價值：")
print(BQ_INSIGHT_SQL.format(project=PROJECT_ID, dataset=BQ_DATASET, table=BQ_TABLE))

In [ ]:
if USE_BIGQUERY:
    load_to_bigquery(insights)
    bq_df = query_bigquery_insights()
    display(bq_df)
else:
    print("USE_BIGQUERY = False — 只展示 schema 與 SQL，未實際寫入。")
    print("要真的跑，請在 CONFIG 把 USE_BIGQUERY 設為 True。")

## §5 成本

**教公式，不背數字。**

    單次成本 = (input_tokens × 單價_in + output_tokens × 單價_out) / 1,000,000
    月成本  = 單次 × SKU數 × 每SKU重生成次數 × (1 + judge 開銷比)

下面印出來的是**這次執行的真實花費**，比任何通用估算都準 ——
因為 prompt 長度、輸出長度、中文 token 密度全是這個專案的實際值。

In [ ]:
"""成本估算：從 demo 的實際用量外推到正式營運規模。

原則：**用公式，不要背數字。**
模型與價格每季在變（例如 Gemini 2.5 Flash-Lite 已公告 2026-10-16 退役），
任何寫死的價格很快就會過期。公式不會。

公式是穩定的，價格常數集中在 config，實際金額由執行結果給出。
"""

from __future__ import annotations

from dataclasses import dataclass



@dataclass
class ScaleEstimate:
    """把 demo 用量外推到正式規模。"""

    sku_count: int
    regen_per_sku: float          # 每個 SKU 平均重新生成幾次（改版、A/B、季節性換檔）
    judge_ratio: float            # 送 judge 的比例（規則層擋掉的不送）
    gen_cost_per_item: float      # 單次生成成本 USD
    judge_cost_per_item: float    # 單次評審成本 USD

    @property
    def total_generations(self) -> float:
        return self.sku_count * self.regen_per_sku

    @property
    def gen_cost(self) -> float:
        return self.total_generations * self.gen_cost_per_item

    @property
    def judge_cost(self) -> float:
        return self.total_generations * self.judge_ratio * self.judge_cost_per_item

    @property
    def total_usd(self) -> float:
        return self.gen_cost + self.judge_cost

    @property
    def total_twd(self) -> float:
        return self.total_usd * USD_TO_TWD

    def render(self) -> str:
        return (
            f"規模假設：{self.sku_count:,} 個 SKU × 每個重生成 {self.regen_per_sku} 次"
            f" = {self.total_generations:,.0f} 次生成\n"
            f"           其中 {self.judge_ratio:.0%} 通過規則層、需要送 LLM 評審\n\n"
            f"  生成成本   {self.total_generations:>10,.0f} 次 × US${self.gen_cost_per_item:.6f}"
            f"  = US${self.gen_cost:>10,.2f}\n"
            f"  評審成本   {self.total_generations * self.judge_ratio:>10,.0f} 次 "
            f"× US${self.judge_cost_per_item:.6f}  = US${self.judge_cost:>10,.2f}\n"
            f"  {'─' * 58}\n"
            f"  合計                                       US${self.total_usd:>10,.2f}"
            f"  （約 NT${self.total_twd:,.0f}）"
        )


def estimate_from_ledger(
    ledger,
    sku_count: int = 100_000,
    regen_per_sku: float = 1.5,
    judge_ratio: float = 0.7,
) -> ScaleEstimate:
    """用 demo 實際跑出來的平均單次成本外推。

    這比任何通用估算都準，因為 prompt 長度、輸出長度、中文 token 密度
    全部是這個專案的真實值，不是別人部落格上的數字。
    """
    gen_calls = [u for t, u in ledger.calls if t.startswith("gen")]
    judge_calls = [u for t, u in ledger.calls if t.startswith("judge")]

    gen_avg = sum(u.cost_usd for u in gen_calls) / len(gen_calls) if gen_calls else 0.0
    judge_avg = (
        sum(u.cost_usd for u in judge_calls) / len(judge_calls) if judge_calls else 0.0
    )

    return ScaleEstimate(
        sku_count=sku_count,
        regen_per_sku=regen_per_sku,
        judge_ratio=judge_ratio,
        gen_cost_per_item=gen_avg,
        judge_cost_per_item=judge_avg,
    )


# --------------------------------------------------------------------------
# 降本四招
# --------------------------------------------------------------------------
LEVERS = [
    {
        "名稱": "模型選型降級",
        "作法": "簡單任務改用更小的模型；只有需要判斷力的環節才用大模型。",
        "省下": "視價差而定，通常是最大的一筆",
        "代價": "品質下降風險 —— **必須有評測表才敢降級**，否則是在賭。",
        "適用": "文案生成這種格式固定、判準明確的任務，通常降級後評測分數不會掉。",
    },
    {
        "名稱": "Batch API",
        "作法": "非即時的工作改走批次介面。商品文案本來就不需要即時。",
        "省下": "約 5 折",
        "代價": "延遲從秒級變成小時級。",
        "適用": "初次匯入全站商品、季節性全面改版 —— 這些場景根本不在乎即時性。",
    },
    {
        "名稱": "Context caching",
        "作法": "把共用的前綴（品牌指南、法規禁詞清單、few-shot 範例）快取起來。",
        "省下": "快取命中部分的 input token 大幅折扣",
        "代價": "有最低 token 門檻與存活時間，前綴要夠長、呼叫要夠密集才划算。",
        "適用": "本專案的 v2/v3 prompt 有大量固定前綴，是典型適用情境。",
    },
    {
        "名稱": "結果快取",
        "作法": "商品規格沒變就不要重新生成。用規格的 hash 當 key。",
        "省下": "取決於商品異動率，通常這是最被低估的一招",
        "代價": "幾乎沒有，只需要一個 key-value 存放。",
        "適用": "所有場景。很多團隊每天全量重跑，其實九成商品根本沒變。",
    },
]


def render_levers() -> str:
    out = []
    for i, lever in enumerate(LEVERS, 1):
        out.append(
            f"{i}. {lever['名稱']}（省下：{lever['省下']}）\n"
            f"   作法：{lever['作法']}\n"
            f"   代價：{lever['代價']}\n"
            f"   適用：{lever['適用']}"
        )
    return "\n\n".join(out)


def measure_chinese_token_density(client, samples: list[str]) -> dict:
    """實測中文的 token 密度，不要用經驗法則。

    直接量出來。中文的 token 密度會隨內容（純中文／中英混雜／
    含數字規格）明顯變動，背一個「一字約 X token」的數字是不可靠的。
    """

    rows = []
    for s in samples:
        tokens = count_tokens(client, s)
        chars = len(s)
        rows.append(
            {
                "字元數": chars,
                "token數": tokens,
                "每字token": round(tokens / chars, 3) if chars else 0,
                "預覽": s[:24] + ("..." if len(s) > 24 else ""),
            }
        )
    densities = [r["每字token"] for r in rows if r["每字token"]]
    return {
        "明細": rows,
        "平均每字token": round(sum(densities) / len(densities), 3) if densities else 0,
        "最低": min(densities) if densities else 0,
        "最高": max(densities) if densities else 0,
    }


def price_check_reminder() -> str:
    """執行前要印出來的提醒。"""
    return (
        f"⚠ 價格常數最後查證時間：{PRICE_LAST_CHECKED}\n"
        f"  上場前請到 https://cloud.google.com/vertex-ai/generative-ai/pricing "
        f"確認並更新 PRICING，\n"
        f"  然後把 PRICE_LAST_CHECKED 改成當天日期。\n"
        f"  不要引用固定的價格數字，以本 notebook 實際印出的金額為準。"
    )

In [ ]:
print(price_check_reminder())
print("\n" + "=" * 60)
print("這次 demo 的實際用量")
print("=" * 60)
print(ledger.summary())

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **41:00 — 唸出實際的「評審佔總成本 __%」。**
>
> 這個數字每次跑都會變，**唸畫面上的，不要唸背下來的**。
> （前一天錄製時先看過一次，心裡有個底。）
>
> 「評審佔了這麼多。這是最常被漏算的一筆 —— 大家算成本只算生成。」

</div>

### 實測中文 token 密度 — 不要背經驗法則

In [ ]:
samples = [
    PRODUCTS[0]['name'],
    "游離型葉黃素30mg，吸收更直接",
    PROMPT_VERSIONS['v3'](PRODUCTS[0],
                          banned_terms=get_banned_terms_for(PRODUCTS[0], BANNED_DATA),
                          tone_examples=TONE_EXAMPLES[PRODUCTS[0]['brand']])[:500],
    "Volta 240W USB-C 編織線 2M 支援40Gbps傳輸",   # 中英數混雜
]
density = measure_chinese_token_density(client, samples)
pd.DataFrame(density['明細'])

In [ ]:
print(f"平均每字 {density['平均每字token']} tokens"
      f"（範圍 {density['最低']} – {density['最高']}）")
print("\n↑ 注意範圍有多寬。中英數混雜的文字密度和純中文差很多，")
print("  所以『中文一個字約 X 個 token』這種經驗法則不可靠，要自己量。")

### 外推到正式規模

In [ ]:
est = estimate_from_ledger(ledger, sku_count=100_000, regen_per_sku=1.5, judge_ratio=0.7)
print(est.render())

### 降本四招

In [ ]:
print(render_levers())

---

## §6 錄製 fixtures（需要離線重播時才執行）

這一格把剛剛所有真實 API 輸出存成檔案，之後就能零網路重播。

**流程：**

1. 在**有網路**的環境把 `RECORD_FIXTURES` 設為 `True`，Run all
2. 執行下面這格，下載 `demo_outputs.json`
3. 把檔案放到 `poc/data/`，執行 `python3 poc/build_notebook.py` 重新產生 notebook
4. 把 `OFFLINE_MODE` 設為 `True`，再 Run all 驗證一次 —— **這次應該完全不連網**
5. 之後在任何沒有網路的環境都能完整重跑

In [ ]:
if RECORD_FIXTURES:
    dump = client.dump()
    with open(FIXTURES_FILE, "w", encoding="utf-8") as f:
        json.dump(dump, f, ensure_ascii=False)
    print(f"✓ 已錄製 {len(dump['calls']):,} 筆呼叫、"
          f"{len(dump['token_counts']):,} 筆 token 計數 → {FIXTURES_FILE}")
    try:
        from google.colab import files
        files.download(FIXTURES_FILE)
    except ImportError:
        print("  （本機執行，檔案已寫在當前目錄）")
elif OFFLINE_MODE:
    print(f"離線重播模式：本次共命中 {client.models.hits:,} 筆錄製輸出，全程未連網。")
else:
    print("目前是一般連線模式。若要錄製以供離線重播，請把 RECORD_FIXTURES 設為 True。")

---

## 收尾

回到一開始的問題：**你怎麼知道它有沒有變好？**

現在你有一張表可以回答。而且那張表是自動產生的，
下次改 prompt、換模型、降級省錢的時候，重跑一次就知道有沒有退步。

**這才是能上線的東西。**

---

### 帶回去的三件事

1. **先建規則層。** 免費、確定性、可以進 CI。不要一開始就做 LLM judge。
2. **用 API 原生的 structured output**，不要用 prompt 硬凹 JSON。
3. **評測本身要花錢**，記得算進去 —— 這是最常被漏掉的一筆。

### 90 天導入路徑

| 階段 | 時間 | 產出 |
|---|---|---|
| Vertex AI Studio 驗證可行性 | 2 週 | 這件事到底做不做得成 |
| 建 golden set + API 串接 | 4 週 | 30–50 筆就足以開始 |
| 小流量上線 + 監控 | 6 週 | 有數字可以對董事會報告 |

<div style="border-left:6px solid #EF7622;padding-left:12px">

**🎤 講者提示**

> **41:45 — 捲回對比表收尾。**
>
> 「所以：你怎麼知道它有沒有變好？看這張表。」
>
> 然後切回投影片，講最後三件事帶回去。
>
> ---
>
> ⏱ **超時的話砍 §4（場景 B）**，不影響主論點。

</div>